# 04 — Georeferenced Project Benchmark

**Objective:** Prepare georeferenced maps and geographically disjoint project train, validation and test regions.

**Project:** GPS-Denied UAV Visual Positioning System

**Provenance rule:** Never describe local reimplementation results as official MobileGeo results.


In [13]:
from pathlib import Path
import json
import hashlib
import random
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/mobilegeo_project")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Seed:", SEED)


Project root: /content/drive/MyDrive/mobilegeo_project
Seed: 42


In [14]:
# ============================================================
# PHASE 3 — STATUS / RECOVERY CHECK
# Find exactly where Notebook 03 stopped
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

checks = [
    # Sample4Geo
    (
        6,
        "Sample4Geo embeddings",
        EMBEDDING_ROOT
        / "sample4geo_sues200_test_query_embeddings.npz",
    ),
    (
        7,
        "Sample4Geo evaluation",
        REPORT_ROOT
        / "sample4geo_location_disjoint_metrics.csv",
    ),
    (
        8,
        "Local vs Sample4Geo comparison",
        REPORT_ROOT
        / "common_benchmark_metrics.csv",
    ),

    # UltraVPR
    (
        10,
        "UltraVPR checkpoint",
        PROJECT_ROOT
        / "configs"
        / "ultravpr_checkpoint_path.txt",
    ),
    (
        12,
        "UltraVPR model smoke test",
        PHASE3_ROOT
        / "ultravpr_audit"
        / "ultravpr_model_load_smoke_test.json",
    ),
    (
        13,
        "UltraVPR embeddings",
        EMBEDDING_ROOT
        / "ultravpr_sues200_test_query_embeddings.npz",
    ),
    (
        14,
        "UltraVPR evaluation",
        REPORT_ROOT
        / "ultravpr_location_disjoint_metrics.csv",
    ),

    # Final Phase 3
    (
        15,
        "Three-method comparison",
        REPORT_ROOT
        / "final_three_method_common_metrics.csv",
    ),
    (
        16,
        "Phase 3 completion package",
        REPORT_ROOT
        / "phase3_completion_manifest.json",
    ),
]

print("=" * 76)
print("PHASE 3 RECOVERY STATUS")
print("=" * 76)

missing = []

for cell_number, description, path in checks:

    exists = path.is_file()

    print(
        f"{'✅' if exists else '❌'} "
        f"Cell {cell_number:02d} — "
        f"{description}"
    )

    print(
        f"   {path}"
    )

    if not exists:
        missing.append(
            (
                cell_number,
                description,
                path,
            )
        )

print("\n" + "=" * 76)

if not missing:

    print("✅ NOTEBOOK 03 IS COMPLETE")
    print(
        "\nYou can return to Notebook 04 "
        "and rerun Cell 1."
    )

else:

    first_missing = missing[0]

    print(
        "⚠ NOTEBOOK 03 IS NOT COMPLETE"
    )

    print(
        "\nFirst missing required stage:"
    )

    print(
        f"Cell {first_missing[0]} — "
        f"{first_missing[1]}"
    )

    print(
        "\nResume Notebook 03 from this cell."
    )

    print(
        "\nMissing stages:"
    )

    for (
        cell_number,
        description,
        path,
    ) in missing:

        print(
            f"- Cell {cell_number}: "
            f"{description}"
        )

PHASE 3 RECOVERY STATUS
✅ Cell 06 — Sample4Geo embeddings
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/sample4geo_sues200_test_query_embeddings.npz
✅ Cell 07 — Sample4Geo evaluation
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/reports/sample4geo_location_disjoint_metrics.csv
✅ Cell 08 — Local vs Sample4Geo comparison
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/reports/common_benchmark_metrics.csv
✅ Cell 10 — UltraVPR checkpoint
   /content/drive/MyDrive/mobilegeo_project/configs/ultravpr_checkpoint_path.txt
✅ Cell 12 — UltraVPR model smoke test
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/ultravpr_audit/ultravpr_model_load_smoke_test.json
✅ Cell 13 — UltraVPR embeddings
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/ultravpr_sues200_test_query_embeddings.npz
✅ Cell 14 — UltraVPR evaluation
   /content/drive/MyDrive/mobilegeo_project/results/common

In [15]:
# ============================================================
# PHASE 3 — SAMPLE4GEO RECOVERY CELL
# Restore official repo + official University-1652 checkpoint
# ============================================================

from pathlib import Path
import subprocess
import sys
import shutil
import hashlib
import zipfile

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

REPOSITORIES_ROOT = (
    PROJECT_ROOT
    / "repositories"
)

SAMPLE4GEO_REPO = (
    REPOSITORIES_ROOT
    / "Sample4Geo"
)

CONFIG_ROOT = (
    PROJECT_ROOT
    / "configs"
)

CHECKPOINT_PATH = (
    SAMPLE4GEO_REPO
    / "pretrained"
    / "university"
    / "convnext_base.fb_in22k_ft_in1k_384"
    / "weights_e1_0.9515.pth"
)

CHECKPOINT_CONFIG_PATH = (
    CONFIG_ROOT
    / "sample4geo_checkpoint_path.txt"
)

TEMP_DOWNLOAD_ROOT = Path(
    "/content/sample4geo_official_weights"
)

OFFICIAL_REPO_URL = (
    "https://github.com/Skyy93/Sample4Geo.git"
)

OFFICIAL_WEIGHTS_FOLDER = (
    "https://drive.google.com/drive/folders/"
    "1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA"
)

EXPECTED_SHA256 = (
    "04f4c8990a8fea73171806dabfe94c405"
    "b792b3b5ef87ce2db768daa529e5332"
)

REPOSITORIES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Restore official Sample4Geo repository
# ------------------------------------------------------------

if (
    SAMPLE4GEO_REPO
    / ".git"
).is_dir():

    print(
        "✅ Sample4Geo repository already exists."
    )

else:

    if SAMPLE4GEO_REPO.exists():
        shutil.rmtree(
            SAMPLE4GEO_REPO
        )

    print(
        "Cloning official Sample4Geo repository..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            OFFICIAL_REPO_URL,
            str(
                SAMPLE4GEO_REPO
            ),
        ],
        check=True,
    )

# ------------------------------------------------------------
# 3. Record exact repo commit
# ------------------------------------------------------------

commit_hash = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=SAMPLE4GEO_REPO,
    text=True,
).strip()

print("\nRepository:")
print(
    SAMPLE4GEO_REPO
)

print("\nPinned commit:")
print(
    commit_hash
)

# ------------------------------------------------------------
# 4. Install gdown only if missing
# ------------------------------------------------------------

try:
    import gdown

except ImportError:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "gdown",
        ],
        check=True,
    )

    import gdown

# ------------------------------------------------------------
# 5. Check whether checkpoint is already present
# ------------------------------------------------------------

if CHECKPOINT_PATH.is_file():

    print(
        "\n✅ University checkpoint already exists."
    )

else:

    print(
        "\nOfficial University checkpoint is missing."
    )

    TEMP_DOWNLOAD_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # 6. Download official weights folder
    # --------------------------------------------------------

    print(
        "\nDownloading official Sample4Geo weights folder..."
    )

    downloaded_files = gdown.download_folder(
        url=OFFICIAL_WEIGHTS_FOLDER,
        output=str(
            TEMP_DOWNLOAD_ROOT
        ),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )

    print(
        "\nDownloaded entries:"
    )

    if downloaded_files:

        for item in downloaded_files:
            print(
                "-",
                item
            )

    # --------------------------------------------------------
    # 7. Look for checkpoint directly first
    # --------------------------------------------------------

    direct_candidates = list(
        TEMP_DOWNLOAD_ROOT.rglob(
            "weights_e1_0.9515.pth"
        )
    )

    if direct_candidates:

        source_checkpoint = (
            direct_candidates[0]
        )

        CHECKPOINT_PATH.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            source_checkpoint,
            CHECKPOINT_PATH,
        )

        print(
            "\n✅ Copied checkpoint from downloaded folder."
        )

    else:

        # ----------------------------------------------------
        # 8. Search downloaded ZIP archive
        # ----------------------------------------------------

        zip_candidates = list(
            TEMP_DOWNLOAD_ROOT.rglob(
                "*.zip"
            )
        )

        print(
            "\nZIP archives found:"
        )

        for archive_path in zip_candidates:
            print(
                "-",
                archive_path
            )

        selected_archive = None
        selected_member = None

        for archive_path in zip_candidates:

            try:

                with zipfile.ZipFile(
                    archive_path,
                    "r",
                ) as archive:

                    members = (
                        archive.namelist()
                    )

                    matching_members = [
                        member
                        for member in members
                        if member.endswith(
                            "weights_e1_0.9515.pth"
                        )
                        and "university"
                        in member.lower()
                    ]

                    if matching_members:

                        selected_archive = (
                            archive_path
                        )

                        selected_member = (
                            matching_members[0]
                        )

                        break

            except zipfile.BadZipFile:
                continue

        if (
            selected_archive is None
            or selected_member is None
        ):

            raise FileNotFoundError(
                "Could not locate "
                "weights_e1_0.9515.pth "
                "inside the official downloaded files."
            )

        print(
            "\nSelected archive:"
        )

        print(
            selected_archive
        )

        print(
            "\nSelected archive member:"
        )

        print(
            selected_member
        )

        CHECKPOINT_PATH.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with zipfile.ZipFile(
            selected_archive,
            "r",
        ) as archive:

            with archive.open(
                selected_member,
                "r",
            ) as source_file:

                with open(
                    CHECKPOINT_PATH,
                    "wb",
                ) as destination_file:

                    shutil.copyfileobj(
                        source_file,
                        destination_file,
                    )

# ------------------------------------------------------------
# 9. SHA-256 validation
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


if not CHECKPOINT_PATH.is_file():

    raise FileNotFoundError(
        f"Checkpoint restoration failed:\n"
        f"{CHECKPOINT_PATH}"
    )

actual_sha256 = sha256_file(
    CHECKPOINT_PATH
)

print(
    "\nCheckpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "\nCheckpoint size:"
)

print(
    f"{CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB"
)

print(
    "\nSHA-256:"
)

print(
    actual_sha256
)

if (
    actual_sha256
    != EXPECTED_SHA256
):

    raise RuntimeError(
        "Checkpoint SHA-256 does not match "
        "the previously audited official file.\n"
        f"Expected: {EXPECTED_SHA256}\n"
        f"Found:    {actual_sha256}"
    )

print(
    "\n✅ Official checkpoint checksum verified."
)

# ------------------------------------------------------------
# 10. Save path configuration
# ------------------------------------------------------------

CHECKPOINT_CONFIG_PATH.write_text(
    str(
        CHECKPOINT_PATH
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 11. Import smoke test
# ------------------------------------------------------------

repo_string = str(
    SAMPLE4GEO_REPO
)

if repo_string not in sys.path:

    sys.path.insert(
        0,
        repo_string,
    )

try:

    from sample4geo.model import TimmModel

    print(
        "\n✅ Sample4Geo model import successful."
    )

except Exception as error:

    print(
        "\n❌ Repository restored, but model import failed:"
    )

    print(
        repr(
            error
        )
    )

    print(
        "\nOfficial requirements include timm>=0.8.8."
    )

    raise

# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 78
)

print(
    "✅ SAMPLE4GEO RECOVERY COMPLETE"
)

print(
    "=" * 78
)

print(
    "\nRepository:"
)

print(
    SAMPLE4GEO_REPO
)

print(
    "\nCommit:"
)

print(
    commit_hash
)

print(
    "\nCheckpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "\nCheckpoint SHA-256:"
)

print(
    actual_sha256
)

print(
    "\nConfig:"
)

print(
    CHECKPOINT_CONFIG_PATH
)

print(
    "\nNext:"
)

print(
    "Rerun PHASE 3 — CELL 6."
)

✅ Sample4Geo repository already exists.

Repository:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo

Pinned commit:
498c6511478053ddb4a43f24a9f4942fab238f09

✅ University checkpoint already exists.

Checkpoint:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth

Checkpoint size:
334.17 MB

SHA-256:
04f4c8990a8fea73171806dabfe94c405b792b3b5ef87ce2db768daa529e5332

✅ Official checkpoint checksum verified.

✅ Sample4Geo model import successful.

✅ SAMPLE4GEO RECOVERY COMPLETE

Repository:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo

Commit:
498c6511478053ddb4a43f24a9f4942fab238f09

Checkpoint:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth

Checkpoint SHA-256:
04f4c8990a8fea73171806dabfe94c405b792b3b5ef87ce2db768daa529e5332

Config:
/content/drive/MyDrive/mobilege

In [17]:
# ============================================================
# RECOVER MISSING PROJECT FILES FROM STALE BACKUP
# Current Google Drive is already mounted correctly.
# ============================================================

from pathlib import Path
import shutil
import os

CURRENT_PROJECT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

# ------------------------------------------------------------
# 1. Confirm current Drive is really mounted
# ------------------------------------------------------------

if not os.path.ismount(
    "/content/drive"
):
    raise RuntimeError(
        "Google Drive is not currently mounted."
    )

print("✅ Google Drive is mounted correctly.")

# ------------------------------------------------------------
# 2. Find stale backup automatically
# ------------------------------------------------------------

backup_candidates = sorted(
    Path("/content").glob(
        "drive_stale_backup_*"
    ),
    key=lambda p: p.name,
    reverse=True,
)

STALE_PROJECT = None

for backup_root in backup_candidates:

    candidate = (
        backup_root
        / "MyDrive"
        / "mobilegeo_project"
    )

    if candidate.is_dir():
        STALE_PROJECT = candidate
        break

if STALE_PROJECT is None:
    raise FileNotFoundError(
        "Could not find the preserved stale "
        "mobilegeo_project under /content."
    )

print("\nCurrent project:")
print(CURRENT_PROJECT)

print("\nStale backup project:")
print(STALE_PROJECT)

CURRENT_PROJECT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 3. Exact files we need to recover
# ------------------------------------------------------------

relative_files = [
    # Phase 2 manifests
    "manifests/sues200_corrected/sues200_drone_manifest.csv",
    "manifests/sues200_corrected/sues200_satellite_manifest.csv",
    "manifests/sues200_corrected/sues200_location_split.csv",
    "manifests/sues200_corrected/sues200_test_queries.csv",
    "manifests/sues200_corrected/sues200_test_gallery.csv",

    # Phase 2 evaluation
    "results/sues200_corrected/corrected_query_results.csv",
    "results/sues200_corrected/corrected_location_disjoint_metrics.csv",
    "results/sues200_corrected/corrected_location_disjoint_metrics.json",
    "results/sues200_corrected/corrected_location_disjoint_rankings.csv",
    "results/sues200_corrected/corrected_per_location_metrics.csv",

    # Phase 2 analysis
    "results/sues200_corrected/top1_failure_cases.csv",
    "results/sues200_corrected/worst_test_locations.csv",
    "results/sues200_corrected/failure_analysis.json",
    "results/sues200_corrected/descriptor_extraction_report.json",
    "results/sues200_corrected/phase2_summary.json",
    "results/sues200_corrected/phase2_corrected_sues200_report.md",

    # Phase 2 embeddings
    "results/sues200_corrected/embeddings/sues200_test_query_embeddings.npz",
    "results/sues200_corrected/embeddings/sues200_test_gallery_embeddings.npz",

    # Phase 3 Sample4Geo outputs
    "results/common_benchmark/embeddings/sample4geo_sues200_test_query_embeddings.npz",
    "results/common_benchmark/embeddings/sample4geo_sues200_test_gallery_embeddings.npz",
    "results/common_benchmark/reports/sample4geo_location_disjoint_metrics.csv",
    "results/common_benchmark/reports/sample4geo_location_disjoint_metrics.json",
    "results/common_benchmark/rankings/sample4geo_query_results.csv",

    # Phase 3 comparison
    "results/common_benchmark/reports/common_benchmark_metrics.csv",

    # configs
    "configs/sues200_resolved_paths.txt",
]

# ------------------------------------------------------------
# 4. Restore only files missing from current project
# ------------------------------------------------------------

restored = []
already_present = []
not_found = []

for relative_name in relative_files:

    source = (
        STALE_PROJECT
        / relative_name
    )

    destination = (
        CURRENT_PROJECT
        / relative_name
    )

    if destination.is_file():

        already_present.append(
            relative_name
        )

        continue

    if not source.is_file():

        not_found.append(
            relative_name
        )

        continue

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source,
        destination,
    )

    restored.append(
        relative_name
    )

# ------------------------------------------------------------
# 5. Recover Sample4Geo repository if missing
# ------------------------------------------------------------

current_sample_repo = (
    CURRENT_PROJECT
    / "repositories"
    / "Sample4Geo"
)

backup_sample_repo = (
    STALE_PROJECT
    / "repositories"
    / "Sample4Geo"
)

sample_repo_status = None

if current_sample_repo.is_dir():

    sample_repo_status = (
        "ALREADY_PRESENT"
    )

elif backup_sample_repo.is_dir():

    current_sample_repo.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        backup_sample_repo,
        current_sample_repo,
    )

    sample_repo_status = (
        "RECOVERED"
    )

else:

    sample_repo_status = (
        "NOT_FOUND_IN_BACKUP"
    )

# ------------------------------------------------------------
# 6. Critical Phase 2 status
# ------------------------------------------------------------

critical_paths = [
    CURRENT_PROJECT
    / "manifests"
    / "sues200_corrected"
    / "sues200_test_queries.csv",

    CURRENT_PROJECT
    / "manifests"
    / "sues200_corrected"
    / "sues200_test_gallery.csv",

    CURRENT_PROJECT
    / "manifests"
    / "sues200_corrected"
    / "sues200_location_split.csv",

    CURRENT_PROJECT
    / "results"
    / "sues200_corrected"
    / "corrected_query_results.csv",
]

print("\n" + "=" * 82)
print("RECOVERY SUMMARY")
print("=" * 82)

print("\nRecovered files:")
print(len(restored))

for item in restored:
    print("✅", item)

print("\nAlready present:")
print(len(already_present))

print("\nNot found in stale backup:")
print(len(not_found))

for item in not_found:
    print("❌", item)

print("\nSample4Geo repository:")
print(sample_repo_status)

print("\n" + "=" * 82)
print("CRITICAL PHASE-2 STATUS")
print("=" * 82)

for path in critical_paths:

    print(
        f"{'✅' if path.is_file() else '❌'} "
        f"{path}"
    )

all_critical_ready = all(
    path.is_file()
    for path in critical_paths
)

print("\n" + "=" * 82)

if all_critical_ready:

    print(
        "✅ PHASE-2 CRITICAL FILES RECOVERED"
    )

    print(
        "\nNext: rerun Phase 3 Cell 6 "
        "Sample4Geo descriptor extraction."
    )

else:

    print(
        "⚠ SOME PHASE-2 CRITICAL FILES "
        "ARE STILL MISSING"
    )

    print(
        "\nDo not rebuild the split yet."
    )

print("=" * 82)

✅ Google Drive is mounted correctly.

Current project:
/content/drive/MyDrive/mobilegeo_project

Stale backup project:
/content/drive_stale_backup_20260808_180033/MyDrive/mobilegeo_project

RECOVERY SUMMARY

Recovered files:
0

Already present:
25

Not found in stale backup:
0

Sample4Geo repository:
ALREADY_PRESENT

CRITICAL PHASE-2 STATUS
✅ /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_queries.csv
✅ /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_gallery.csv
✅ /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_location_split.csv
✅ /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/corrected_query_results.csv

✅ PHASE-2 CRITICAL FILES RECOVERED

Next: rerun Phase 3 Cell 6 Sample4Geo descriptor extraction.


In [18]:
# ============================================================
# PHASE 3 — UPDATED STATUS CHECK
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPORT_ROOT = PHASE3_ROOT / "reports"
RANKING_ROOT = PHASE3_ROOT / "rankings"
EMBED_ROOT = PHASE3_ROOT / "embeddings"

checks = [
    (
        6,
        "Sample4Geo query embeddings",
        EMBED_ROOT
        / "sample4geo_sues200_test_query_embeddings.npz",
    ),
    (
        6,
        "Sample4Geo gallery embeddings",
        EMBED_ROOT
        / "sample4geo_sues200_test_gallery_embeddings.npz",
    ),
    (
        7,
        "Sample4Geo evaluation",
        REPORT_ROOT
        / "sample4geo_location_disjoint_metrics.csv",
    ),
    (
        8,
        "Local vs Sample4Geo comparison",
        REPORT_ROOT
        / "common_benchmark_metrics.csv",
    ),
    (
        10,
        "UltraVPR checkpoint config",
        PROJECT_ROOT
        / "configs"
        / "ultravpr_checkpoint_path.txt",
    ),
    (
        12,
        "UltraVPR model smoke test",
        PHASE3_ROOT
        / "ultravpr_audit"
        / "ultravpr_model_load_smoke_test.json",
    ),
    (
        13,
        "UltraVPR query embeddings",
        EMBED_ROOT
        / "ultravpr_sues200_test_query_embeddings.npz",
    ),
    (
        13,
        "UltraVPR gallery embeddings",
        EMBED_ROOT
        / "ultravpr_sues200_test_gallery_embeddings.npz",
    ),
    (
        14,
        "UltraVPR evaluation",
        REPORT_ROOT
        / "ultravpr_location_disjoint_metrics.csv",
    ),
    (
        15,
        "Final three-method comparison",
        REPORT_ROOT
        / "final_three_method_common_metrics.csv",
    ),
    (
        16,
        "Phase 3 completion manifest",
        REPORT_ROOT
        / "phase3_completion_manifest.json",
    ),
]

print("=" * 82)
print("PHASE 3 — UPDATED RECOVERY STATUS")
print("=" * 82)

missing_cells = []

for cell, description, path in checks:

    exists = path.is_file()

    print(
        f"{'✅' if exists else '❌'} "
        f"Cell {cell:02d} — {description}"
    )

    print(
        f"   {path}"
    )

    if not exists:
        missing_cells.append(cell)

print("\n" + "=" * 82)

if not missing_cells:

    print("✅ NOTEBOOK 03 IS ALREADY COMPLETE")

else:

    first_missing = min(
        missing_cells
    )

    print(
        "FIRST REQUIRED MISSING CELL:"
    )

    print(
        f"Cell {first_missing}"
    )

    print(
        "\nResume Phase 3 from that cell."
    )

print("=" * 82)

PHASE 3 — UPDATED RECOVERY STATUS
✅ Cell 06 — Sample4Geo query embeddings
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/sample4geo_sues200_test_query_embeddings.npz
✅ Cell 06 — Sample4Geo gallery embeddings
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/sample4geo_sues200_test_gallery_embeddings.npz
✅ Cell 07 — Sample4Geo evaluation
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/reports/sample4geo_location_disjoint_metrics.csv
✅ Cell 08 — Local vs Sample4Geo comparison
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/reports/common_benchmark_metrics.csv
✅ Cell 10 — UltraVPR checkpoint config
   /content/drive/MyDrive/mobilegeo_project/configs/ultravpr_checkpoint_path.txt
✅ Cell 12 — UltraVPR model smoke test
   /content/drive/MyDrive/mobilegeo_project/results/common_benchmark/ultravpr_audit/ultravpr_model_load_smoke_test.json
✅ Cell 13 — UltraVPR query embeddings
   /content/dri

In [22]:
# ============================================================
# NOTEBOOK 04 — CELL 1 — FASTEST METADATA DISCOVERY
# No deep recursive scan
# ============================================================

from pathlib import Path
import os
import pandas as pd
import re

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

SEARCH_ROOTS = [
    MYDRIVE / "vps_dataset",
    MYDRIVE / "vps_dataset_subset",
    MYDRIVE / "mobilegeo_clean",
    MYDRIVE / "mobilegeo_final",
]

SEARCH_ROOTS = [
    p for p in SEARCH_ROOTS
    if p.is_dir()
]

print("=" * 80)
print("FAST GEOREFERENCE METADATA SEARCH")
print("=" * 80)

for root in SEARCH_ROOTS:
    print("✅", root)

# ------------------------------------------------------------
# 1. Only metadata-type files
# ------------------------------------------------------------

VALID_SUFFIXES = {
    ".csv",
    ".tsv",
    ".json",
    ".geojson",
    ".txt",
    ".kml",
    ".xml",
    ".parquet",
}

candidates = []

# ------------------------------------------------------------
# 2. Maximum depth = 2 only
# ------------------------------------------------------------

for root in SEARCH_ROOTS:

    root_depth = len(
        root.parts
    )

    for current_root, dirs, files in os.walk(
        root
    ):

        current_path = Path(
            current_root
        )

        depth = (
            len(current_path.parts)
            - root_depth
        )

        if depth >= 2:
            dirs[:] = []

        for filename in files:

            path = (
                current_path
                / filename
            )

            if (
                path.suffix.lower()
                in VALID_SUFFIXES
            ):

                candidates.append(
                    path
                )

print("\nMetadata files found:")
print(
    len(candidates)
)

for path in candidates[:200]:
    print("✅", path)

# ------------------------------------------------------------
# 3. Coordinate column vocabulary
# ------------------------------------------------------------

LAT_NAMES = {
    "lat",
    "latitude",
    "gps_lat",
    "gps_latitude",
    "query_lat",
    "query_latitude",
    "drone_lat",
    "drone_latitude",
}

LON_NAMES = {
    "lon",
    "lng",
    "longitude",
    "gps_lon",
    "gps_lng",
    "gps_longitude",
    "query_lon",
    "query_lng",
    "query_longitude",
    "drone_lon",
    "drone_longitude",
}

def normalize(
    value
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).lower()
    ).strip("_")

# ------------------------------------------------------------
# 4. Inspect only CSV / TSV / Parquet headers
# ------------------------------------------------------------

coordinate_tables = []

for path in candidates:

    suffix = (
        path.suffix.lower()
    )

    if suffix not in {
        ".csv",
        ".tsv",
        ".parquet",
    }:
        continue

    try:

        if suffix == ".csv":

            df = pd.read_csv(
                path,
                nrows=3,
            )

        elif suffix == ".tsv":

            df = pd.read_csv(
                path,
                sep="\t",
                nrows=3,
            )

        else:

            df = pd.read_parquet(
                path
            ).head(3)

        normalized_columns = {
            normalize(col): col
            for col in df.columns
        }

        lat_cols = [
            original
            for key, original
            in normalized_columns.items()
            if key in LAT_NAMES
        ]

        lon_cols = [
            original
            for key, original
            in normalized_columns.items()
            if key in LON_NAMES
        ]

        if (
            lat_cols
            and lon_cols
        ):

            coordinate_tables.append({
                "path": str(path),
                "latitude": lat_cols,
                "longitude": lon_cols,
                "columns": list(
                    df.columns
                ),
            })

    except Exception:
        pass

# ------------------------------------------------------------
# 5. Search text filenames/content only for obvious geo terms
# ------------------------------------------------------------

geo_text_files = []

TERMS = [
    "latitude",
    "longitude",
    "gps",
    "coordinate",
    "wgs84",
    "utm",
]

for path in candidates:

    if path.suffix.lower() not in {
        ".json",
        ".geojson",
        ".txt",
        ".kml",
        ".xml",
    }:
        continue

    try:

        if path.stat().st_size > (
            2 * 1024 * 1024
        ):
            continue

        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).lower()

        hits = [
            term
            for term in TERMS
            if term in text
        ]

        if hits:

            geo_text_files.append(
                (
                    path,
                    hits,
                )
            )

    except Exception:
        pass

# ------------------------------------------------------------
# 6. Final output
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("✅ FAST GEOREFERENCE SEARCH COMPLETE")
print("=" * 80)

print(
    "\nExplicit latitude/longitude tables:"
)

print(
    len(
        coordinate_tables
    )
)

for record in coordinate_tables:

    print("\n✅", record["path"])
    print(
        "   Latitude:",
        record["latitude"]
    )
    print(
        "   Longitude:",
        record["longitude"]
    )
    print(
        "   Columns:",
        record["columns"]
    )

print(
    "\nGeo-related JSON/TXT/KML files:"
)

print(
    len(
        geo_text_files
    )
)

for path, hits in (
    geo_text_files[:50]
):

    print(
        "✅",
        path,
        "->",
        hits,
    )

print("\nDone.")

FAST GEOREFERENCE METADATA SEARCH
✅ /content/drive/MyDrive/vps_dataset
✅ /content/drive/MyDrive/vps_dataset_subset
✅ /content/drive/MyDrive/mobilegeo_clean
✅ /content/drive/MyDrive/mobilegeo_final

Metadata files found:
47
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000023/labels.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000023/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000017/labels.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000017/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000016/labels.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000016/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000008/labels.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000008/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000004/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000004/labels.json
✅ /cont

In [23]:
# ============================================================
# NOTEBOOK 04 — CELL 2
# Direct inspection of GPS_info.json and existing manifests
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import re

import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

VPS_SUBSET_ROOT = (
    MYDRIVE
    / "vps_dataset_subset"
)

MOBILEGEO_FINAL = (
    MYDRIVE
    / "mobilegeo_final"
)

# ------------------------------------------------------------
# 2. Find GPS_info.json files
# ------------------------------------------------------------

gps_files = sorted(
    VPS_SUBSET_ROOT.glob(
        "*/GPS_info.json"
    )
)

label_files = sorted(
    VPS_SUBSET_ROOT.glob(
        "*/labels.json"
    )
)

print("=" * 84)
print("NOTEBOOK 04 — DIRECT GPS METADATA INSPECTION")
print("=" * 84)

print("\nGPS_info.json files:")
print(
    len(gps_files)
)

for path in gps_files:
    print("✅", path)

print("\nlabels.json files:")
print(
    len(label_files)
)

# ------------------------------------------------------------
# 3. Recursive JSON flattener
# ------------------------------------------------------------

def flatten_json(
    obj,
    prefix="",
):
    records = []

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            new_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            records.extend(
                flatten_json(
                    value,
                    new_prefix,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            new_prefix = (
                f"{prefix}[{index}]"
            )

            records.extend(
                flatten_json(
                    value,
                    new_prefix,
                )
            )

    else:

        records.append(
            (
                prefix,
                obj,
            )
        )

    return records

# ------------------------------------------------------------
# 4. Coordinate-key patterns
# ------------------------------------------------------------

LAT_PATTERNS = [
    r"(^|[._])lat($|[._])",
    r"latitude",
]

LON_PATTERNS = [
    r"(^|[._])lon($|[._])",
    r"(^|[._])lng($|[._])",
    r"longitude",
]

ALT_PATTERNS = [
    r"altitude",
    r"(^|[._])alt($|[._])",
    r"height",
]

GPS_PATTERNS = [
    r"gps",
    r"position",
    r"coordinate",
    r"location",
]

ORIENTATION_PATTERNS = [
    r"yaw",
    r"pitch",
    r"roll",
    r"heading",
    r"bearing",
    r"azimuth",
]

def matches_any(
    key,
    patterns,
):
    key = str(
        key
    ).lower()

    return any(
        re.search(
            pattern,
            key,
        )
        for pattern in patterns
    )

# ------------------------------------------------------------
# 5. Inspect GPS_info.json files
# ------------------------------------------------------------

gps_records = []

for path in gps_files:

    try:

        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as error:

        gps_records.append({
            "file": str(path),
            "key": "",
            "value": "",
            "value_type": "",
            "category": "READ_ERROR",
            "error": repr(error),
        })

        continue

    flattened = flatten_json(
        data
    )

    for key, value in flattened:

        category = "other"

        if matches_any(
            key,
            LAT_PATTERNS,
        ):
            category = "latitude_key"

        elif matches_any(
            key,
            LON_PATTERNS,
        ):
            category = "longitude_key"

        elif matches_any(
            key,
            ALT_PATTERNS,
        ):
            category = "altitude_key"

        elif matches_any(
            key,
            ORIENTATION_PATTERNS,
        ):
            category = "orientation_key"

        elif matches_any(
            key,
            GPS_PATTERNS,
        ):
            category = "gps_related_key"

        numeric_value = None

        if isinstance(
            value,
            (
                int,
                float,
            ),
        ):

            numeric_value = float(
                value
            )

        elif isinstance(
            value,
            str,
        ):

            try:
                numeric_value = float(
                    value.strip()
                )
            except Exception:
                pass

        # Heuristic only — not yet accepted as coordinates.
        possible_lat = (
            numeric_value is not None
            and -90.0
            <= numeric_value
            <= 90.0
        )

        possible_lon = (
            numeric_value is not None
            and -180.0
            <= numeric_value
            <= 180.0
        )

        gps_records.append({
            "file": str(
                path
            ),

            "sample_id": (
                path.parent.name
            ),

            "key": str(
                key
            ),

            "value": str(
                value
            )[:500],

            "value_type": (
                type(
                    value
                ).__name__
            ),

            "numeric_value": (
                numeric_value
            ),

            "category": (
                category
            ),

            "possible_lat_range": bool(
                possible_lat
            ),

            "possible_lon_range": bool(
                possible_lon
            ),
        })

gps_df = pd.DataFrame(
    gps_records
)

# ------------------------------------------------------------
# 6. Show actual JSON structure
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("GPS_INFO.JSON CONTENT")
print("=" * 84)

for path in gps_files[:5]:

    print(
        f"\n--- {path.parent.name}/GPS_info.json ---"
    )

    try:

        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

    except Exception as error:

        print(
            "ERROR:",
            repr(error),
        )

# ------------------------------------------------------------
# 7. Show coordinate/GPS-related flattened keys
# ------------------------------------------------------------

if not gps_df.empty:

    interesting_gps = (
        gps_df[
            gps_df[
                "category"
            ]
            != "other"
        ]
        .copy()
    )

else:

    interesting_gps = (
        pd.DataFrame()
    )

print("\n" + "=" * 84)
print("GPS / COORDINATE / ORIENTATION KEYS")
print("=" * 84)

if not interesting_gps.empty:

    print(
        interesting_gps[
            [
                "sample_id",
                "key",
                "value",
                "category",
            ]
        ]
        .head(200)
        .to_string(
            index=False
        )
    )

else:

    print(
        "No obvious coordinate-key names found."
    )

# ------------------------------------------------------------
# 8. Inspect labels.json structure
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("LABELS.JSON SAMPLE CONTENT")
print("=" * 84)

for path in label_files[:3]:

    print(
        f"\n--- {path.parent.name}/labels.json ---"
    )

    try:

        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False,
            )[:5000]
        )

    except Exception as error:

        print(
            "ERROR:",
            repr(error),
        )

# ------------------------------------------------------------
# 9. Inspect existing MobileGeo final manifests
# ------------------------------------------------------------

candidate_manifest_paths = [
    MOBILEGEO_FINAL
    / "data"
    / "frame_manifest.csv",

    MOBILEGEO_FINAL
    / "data"
    / "tile_manifest.csv",

    MOBILEGEO_FINAL
    / "data"
    / "video_metadata.json",

    MOBILEGEO_FINAL
    / "cache"
    / "advanced_edge_geo_lpn_test_query_metadata.csv",

    MOBILEGEO_FINAL
    / "satellite_embeddings"
    / "advanced_edge_geo_lpn_tile_metadata.csv",
]

print("\n" + "=" * 84)
print("EXISTING MOBILEGEO_FINAL METADATA")
print("=" * 84)

manifest_records = []

for path in candidate_manifest_paths:

    print(
        f"\n{'✅' if path.is_file() else '❌'} "
        f"{path}"
    )

    if not path.is_file():
        continue

    if path.suffix.lower() == ".csv":

        try:

            df = pd.read_csv(
                path,
                nrows=5,
            )

            print(
                "Columns:"
            )

            print(
                df.columns.tolist()
            )

            print(
                "\nPreview:"
            )

            print(
                df.to_string(
                    index=False
                )
            )

            manifest_records.append({
                "path": str(
                    path
                ),

                "type": "csv",

                "columns": (
                    df.columns.tolist()
                ),
            })

        except Exception as error:

            print(
                "ERROR:",
                repr(error),
            )

    elif path.suffix.lower() == ".json":

        try:

            data = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

            print(
                json.dumps(
                    data,
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )

            flattened = flatten_json(
                data
            )

            manifest_records.append({
                "path": str(
                    path
                ),

                "type": "json",

                "flattened_keys": [
                    key
                    for key, _
                    in flattened[:500]
                ],
            })

        except Exception as error:

            print(
                "ERROR:",
                repr(error),
            )

# ------------------------------------------------------------
# 10. Save audit files
# ------------------------------------------------------------

GPS_FLAT_CSV = (
    AUDIT_ROOT
    / "gps_info_flattened_audit.csv"
)

DIRECT_INSPECTION_JSON = (
    AUDIT_ROOT
    / "direct_geospatial_metadata_inspection.json"
)

gps_df.to_csv(
    GPS_FLAT_CSV,
    index=False,
)

latitude_keys = []

longitude_keys = []

altitude_keys = []

orientation_keys = []

if not gps_df.empty:

    latitude_keys = sorted(
        gps_df.loc[
            gps_df[
                "category"
            ]
            == "latitude_key",
            "key",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    longitude_keys = sorted(
        gps_df.loc[
            gps_df[
                "category"
            ]
            == "longitude_key",
            "key",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    altitude_keys = sorted(
        gps_df.loc[
            gps_df[
                "category"
            ]
            == "altitude_key",
            "key",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    orientation_keys = sorted(
        gps_df.loc[
            gps_df[
                "category"
            ]
            == "orientation_key",
            "key",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

inspection_summary = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "gps_info_file_count": int(
        len(
            gps_files
        )
    ),

    "labels_file_count": int(
        len(
            label_files
        )
    ),

    "latitude_keys_found": (
        latitude_keys
    ),

    "longitude_keys_found": (
        longitude_keys
    ),

    "altitude_keys_found": (
        altitude_keys
    ),

    "orientation_keys_found": (
        orientation_keys
    ),

    "explicit_lat_lon_pair_found": bool(
        latitude_keys
        and longitude_keys
    ),

    "existing_manifest_records": (
        manifest_records
    ),

    "status": (
        "POTENTIAL_VERIFIED_COORDINATES_FOUND"
        if (
            latitude_keys
            and longitude_keys
        )
        else (
            "GPS_INFO_PRESENT_BUT_SCHEMA_REQUIRES_INTERPRETATION"
        )
    ),

    "restriction": (
        "Numeric range matching alone must not be "
        "treated as verified latitude/longitude."
    ),
}

DIRECT_INSPECTION_JSON.write_text(
    json.dumps(
        inspection_summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("✅ NOTEBOOK 04 CELL 2 DIRECT INSPECTION COMPLETE")
print("=" * 84)

print("\nLatitude keys found:")
print(
    latitude_keys
)

print("\nLongitude keys found:")
print(
    longitude_keys
)

print("\nAltitude keys found:")
print(
    altitude_keys
)

print("\nOrientation keys found:")
print(
    orientation_keys
)

print("\nExplicit lat/lon key pair found:")
print(
    bool(
        latitude_keys
        and longitude_keys
    )
)

print("\nGPS flattened audit:")
print(
    GPS_FLAT_CSV
)

print("\nInspection summary:")
print(
    DIRECT_INSPECTION_JSON
)

print("\nNext action:")

if (
    latitude_keys
    and longitude_keys
):

    print(
        "Validate coordinate values and map "
        "GPS records to actual UAV frames."
    )

else:

    print(
        "Interpret the GPS_info.json schema "
        "before deciding whether meter-level "
        "localization can be evaluated."
    )

NOTEBOOK 04 — DIRECT GPS METADATA INSPECTION

GPS_info.json files:
5
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000004/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000008/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000016/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000017/GPS_info.json
✅ /content/drive/MyDrive/vps_dataset_subset/Chuanmei_100_000023/GPS_info.json

labels.json files:
5

GPS_INFO.JSON CONTENT

--- Chuanmei_100_000004/GPS_info.json ---
{
  "UAV": {
    "E": 120.33623455555555,
    "N": 30.324269555555556
  },
  "Satellite": {
    "0.jpg": {
      "tl_E": 120.33444479849757,
      "tl_N": 30.32448718248934,
      "br_E": 120.33632310425557,
      "br_N": 30.322866556386703,
      "center_distribute_X": 0.7314285714285714,
      "center_distribute_Y": -0.9057142857142857,
      "map_size": 700
    },
    "1.jpg": {
      "tl_E": 120.33439649920665,
      "tl_N": 30.325566056437665,


In [24]:
# ============================================================
# NOTEBOOK 04 — CELL 3
# Validate VPS geographic coordinate schema using
# GPS bounds <-> pixel labels <-> normalized coordinates
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_SUBSET_ROOT = (
    MYDRIVE
    / "vps_dataset_subset"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# labels.json coordinates are consistent with a
# 768 x 768 satellite representation.
LABEL_IMAGE_SIZE = 768

# ------------------------------------------------------------
# 2. Locate sample metadata
# ------------------------------------------------------------

gps_files = sorted(
    VPS_SUBSET_ROOT.glob(
        "*/GPS_info.json"
    )
)

if not gps_files:
    raise FileNotFoundError(
        "No GPS_info.json files found."
    )

print("=" * 84)
print("NOTEBOOK 04 — VPS GEOREFERENCE SCHEMA VALIDATION")
print("=" * 84)

print("\nSamples:")
print(
    len(gps_files)
)

# ------------------------------------------------------------
# 3. Optional geographic distance helper
# ------------------------------------------------------------
# This is only diagnostic because the exact CRS has not
# yet been independently verified.

def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2,
):

    earth_radius_m = (
        6371008.8
    )

    phi1 = math.radians(
        lat1
    )

    phi2 = math.radians(
        lat2
    )

    dphi = math.radians(
        lat2 - lat1
    )

    dlambda = math.radians(
        lon2 - lon1
    )

    a = (
        math.sin(
            dphi / 2
        ) ** 2
        +
        math.cos(
            phi1
        )
        * math.cos(
            phi2
        )
        * math.sin(
            dlambda / 2
        ) ** 2
    )

    return (
        2
        * earth_radius_m
        * math.asin(
            math.sqrt(
                a
            )
        )
    )

# ------------------------------------------------------------
# 4. Validate every sample and satellite map
# ------------------------------------------------------------

records = []

for gps_path in gps_files:

    sample_dir = (
        gps_path.parent
    )

    sample_id = (
        sample_dir.name
    )

    labels_path = (
        sample_dir
        / "labels.json"
    )

    if not labels_path.is_file():
        raise FileNotFoundError(
            labels_path
        )

    gps_data = json.loads(
        gps_path.read_text(
            encoding="utf-8"
        )
    )

    labels = json.loads(
        labels_path.read_text(
            encoding="utf-8"
        )
    )

    if "UAV" not in gps_data:
        raise KeyError(
            f"{sample_id}: UAV section missing."
        )

    if "Satellite" not in gps_data:
        raise KeyError(
            f"{sample_id}: Satellite section missing."
        )

    uav_e = float(
        gps_data["UAV"]["E"]
    )

    uav_n = float(
        gps_data["UAV"]["N"]
    )

    # Basic decimal-degree-like range check.
    if not (
        -180.0 <= uav_e <= 180.0
    ):
        raise RuntimeError(
            f"{sample_id}: invalid E range."
        )

    if not (
        -90.0 <= uav_n <= 90.0
    ):
        raise RuntimeError(
            f"{sample_id}: invalid N range."
        )

    for image_name, info in (
        gps_data[
            "Satellite"
        ].items()
    ):

        if image_name not in labels:
            raise KeyError(
                f"{sample_id}: "
                f"{image_name} missing from labels.json"
            )

        tl_e = float(
            info["tl_E"]
        )

        tl_n = float(
            info["tl_N"]
        )

        br_e = float(
            info["br_E"]
        )

        br_n = float(
            info["br_N"]
        )

        map_size = int(
            info["map_size"]
        )

        center_x_recorded = float(
            info[
                "center_distribute_X"
            ]
        )

        center_y_recorded = float(
            info[
                "center_distribute_Y"
            ]
        )

        # ----------------------------------------------------
        # 5. Validate bounding-box orientation
        # ----------------------------------------------------

        bbox_valid = bool(
            tl_e < br_e
            and tl_n > br_n
        )

        # ----------------------------------------------------
        # 6. Project UAV coordinate into native map pixels
        # ----------------------------------------------------

        native_x = (
            (
                uav_e - tl_e
            )
            /
            (
                br_e - tl_e
            )
            * map_size
        )

        native_y = (
            (
                tl_n - uav_n
            )
            /
            (
                tl_n - br_n
            )
            * map_size
        )

        inside_bbox = bool(
            0.0 <= native_x <= map_size
            and
            0.0 <= native_y <= map_size
        )

        # ----------------------------------------------------
        # 7. Convert to labels.json 768x768 convention
        # ----------------------------------------------------

        predicted_col_float = (
            native_x
            * LABEL_IMAGE_SIZE
            / map_size
        )

        predicted_row_float = (
            native_y
            * LABEL_IMAGE_SIZE
            / map_size
        )

        # Dataset labels match integer truncation/floor.
        predicted_col = int(
            math.floor(
                predicted_col_float
            )
        )

        predicted_row = int(
            math.floor(
                predicted_row_float
            )
        )

        recorded_label = (
            labels[
                image_name
            ]
        )

        if (
            not isinstance(
                recorded_label,
                list,
            )
            or len(
                recorded_label
            ) != 2
        ):
            raise RuntimeError(
                f"{sample_id}/{image_name}: "
                "unexpected labels.json format."
            )

        recorded_row = int(
            recorded_label[0]
        )

        recorded_col = int(
            recorded_label[1]
        )

        row_error_px = (
            predicted_row
            - recorded_row
        )

        col_error_px = (
            predicted_col
            - recorded_col
        )

        max_label_error_px = max(
            abs(
                row_error_px
            ),
            abs(
                col_error_px
            ),
        )

        # ----------------------------------------------------
        # 8. Reconstruct center_distribute_X/Y
        # ----------------------------------------------------
        #
        # Dataset convention:
        #
        # X uses vertical/native-y axis
        # Y uses horizontal/native-x axis
        #
        # Both are normalized around map center.

        predicted_center_x = (
            1.0
            - (
                2.0
                * native_y
                / map_size
            )
        )

        predicted_center_y = (
            1.0
            - (
                2.0
                * native_x
                / map_size
            )
        )

        center_x_error = (
            predicted_center_x
            - center_x_recorded
        )

        center_y_error = (
            predicted_center_y
            - center_y_recorded
        )

        # ----------------------------------------------------
        # 9. Geographic center of satellite crop
        # ----------------------------------------------------

        satellite_center_e = (
            (
                tl_e
                + br_e
            )
            / 2.0
        )

        satellite_center_n = (
            (
                tl_n
                + br_n
            )
            / 2.0
        )

        approximate_center_distance_m = (
            haversine_m(
                uav_n,
                uav_e,
                satellite_center_n,
                satellite_center_e,
            )
        )

        # ----------------------------------------------------
        # 10. Save validation row
        # ----------------------------------------------------

        records.append({
            "sample_id": (
                sample_id
            ),

            "satellite_image": (
                image_name
            ),

            "uav_E": (
                uav_e
            ),

            "uav_N": (
                uav_n
            ),

            "tl_E": (
                tl_e
            ),

            "tl_N": (
                tl_n
            ),

            "br_E": (
                br_e
            ),

            "br_N": (
                br_n
            ),

            "map_size": (
                map_size
            ),

            "bbox_valid": (
                bbox_valid
            ),

            "uav_inside_bbox": (
                inside_bbox
            ),

            "native_x": float(
                native_x
            ),

            "native_y": float(
                native_y
            ),

            "predicted_label_row": (
                predicted_row
            ),

            "predicted_label_col": (
                predicted_col
            ),

            "recorded_label_row": (
                recorded_row
            ),

            "recorded_label_col": (
                recorded_col
            ),

            "row_error_px": (
                row_error_px
            ),

            "col_error_px": (
                col_error_px
            ),

            "max_label_error_px": (
                max_label_error_px
            ),

            "recorded_center_distribute_X": (
                center_x_recorded
            ),

            "predicted_center_distribute_X": (
                predicted_center_x
            ),

            "center_X_error": float(
                center_x_error
            ),

            "recorded_center_distribute_Y": (
                center_y_recorded
            ),

            "predicted_center_distribute_Y": (
                predicted_center_y
            ),

            "center_Y_error": float(
                center_y_error
            ),

            "satellite_center_E": (
                satellite_center_e
            ),

            "satellite_center_N": (
                satellite_center_n
            ),

            "approximate_uav_to_crop_center_m": (
                approximate_center_distance_m
            ),
        })

validation_df = pd.DataFrame(
    records
)

# ------------------------------------------------------------
# 11. Overall validation
# ------------------------------------------------------------

expected_rows = (
    len(
        gps_files
    )
    * 12
)

if len(
    validation_df
) != expected_rows:

    raise RuntimeError(
        f"Expected {expected_rows} records, "
        f"found {len(validation_df)}."
    )

all_bbox_valid = bool(
    validation_df[
        "bbox_valid"
    ].all()
)

all_uav_inside = bool(
    validation_df[
        "uav_inside_bbox"
    ].all()
)

max_label_error = int(
    validation_df[
        "max_label_error_px"
    ].max()
)

max_center_error = float(
    max(
        validation_df[
            "center_X_error"
        ].abs().max(),

        validation_df[
            "center_Y_error"
        ].abs().max(),
    )
)

pixel_roundtrip_pass = bool(
    max_label_error <= 1
)

normalized_coordinate_pass = bool(
    max_center_error
    < 1e-8
)

SCHEMA_VALIDATED = bool(
    all_bbox_valid
    and all_uav_inside
    and pixel_roundtrip_pass
    and normalized_coordinate_pass
)

# ------------------------------------------------------------
# 12. Build compact georeference manifest
# ------------------------------------------------------------

manifest_columns = [
    "sample_id",
    "satellite_image",

    "uav_E",
    "uav_N",

    "tl_E",
    "tl_N",
    "br_E",
    "br_N",

    "satellite_center_E",
    "satellite_center_N",

    "map_size",

    "recorded_label_row",
    "recorded_label_col",
]

georef_manifest_df = (
    validation_df[
        manifest_columns
    ]
    .copy()
)

# Explicit semantic aliases, while preserving original keys.
georef_manifest_df[
    "uav_longitude_like"
] = georef_manifest_df[
    "uav_E"
]

georef_manifest_df[
    "uav_latitude_like"
] = georef_manifest_df[
    "uav_N"
]

georef_manifest_df[
    "coordinate_schema"
] = (
    "GEOGRAPHIC_DEGREES_EAST_NORTH"
)

georef_manifest_df[
    "crs_status"
] = (
    "CRS_UNVERIFIED"
)

# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

VALIDATION_CSV = (
    AUDIT_ROOT
    / "vps_georeference_schema_validation.csv"
)

VALIDATION_JSON = (
    AUDIT_ROOT
    / "vps_georeference_schema_validation.json"
)

GEOREF_MANIFEST_CSV = (
    MANIFEST_ROOT
    / "vps_subset_georeference_manifest.csv"
)

validation_df.to_csv(
    VALIDATION_CSV,
    index=False,
)

georef_manifest_df.to_csv(
    GEOREF_MANIFEST_CSV,
    index=False,
)

summary = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sample_count": int(
        len(
            gps_files
        )
    ),

    "satellite_record_count": int(
        len(
            validation_df
        )
    ),

    "coordinate_field_interpretation": {
        "E": (
            "EAST_LONGITUDE_LIKE_"
            "GEOGRAPHIC_DEGREES"
        ),

        "N": (
            "NORTH_LATITUDE_LIKE_"
            "GEOGRAPHIC_DEGREES"
        ),

        "tl_E": (
            "SATELLITE_TOP_LEFT_EAST"
        ),

        "tl_N": (
            "SATELLITE_TOP_LEFT_NORTH"
        ),

        "br_E": (
            "SATELLITE_BOTTOM_RIGHT_EAST"
        ),

        "br_N": (
            "SATELLITE_BOTTOM_RIGHT_NORTH"
        ),
    },

    "crs": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "label_coordinate_order": (
        "ROW_COLUMN"
    ),

    "label_image_size": (
        LABEL_IMAGE_SIZE
    ),

    "all_bounding_boxes_valid": (
        all_bbox_valid
    ),

    "all_uav_coordinates_inside_satellite_bounds": (
        all_uav_inside
    ),

    "max_pixel_roundtrip_error": (
        max_label_error
    ),

    "pixel_roundtrip_pass": (
        pixel_roundtrip_pass
    ),

    "max_center_distribution_error": (
        max_center_error
    ),

    "normalized_coordinate_pass": (
        normalized_coordinate_pass
    ),

    "schema_internal_consistency_validated": (
        SCHEMA_VALIDATED
    ),

    "metric_distance_status": (
        "BLOCK_FINAL_GEODETIC_CLAIM_UNTIL_CRS_VERIFIED"
    ),

    "outputs": {
        "validation_csv": str(
            VALIDATION_CSV
        ),

        "georeference_manifest": str(
            GEOREF_MANIFEST_CSV
        ),
    },
}

VALIDATION_JSON.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 14. Final display
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("✅ NOTEBOOK 04 CELL 3 SCHEMA VALIDATION COMPLETE")
print("=" * 84)

print("\nSamples:")
print(
    len(
        gps_files
    )
)

print("\nSatellite records validated:")
print(
    len(
        validation_df
    )
)

print("\nAll bounding boxes valid:")
print(
    all_bbox_valid
)

print("\nAll UAV coordinates inside satellite bounds:")
print(
    all_uav_inside
)

print("\nMaximum label round-trip error:")
print(
    f"{max_label_error} px"
)

print("\nMaximum center-distribution error:")
print(
    f"{max_center_error:.12e}"
)

print("\nPixel round-trip validation:")
print(
    pixel_roundtrip_pass
)

print("\nNormalized-coordinate validation:")
print(
    normalized_coordinate_pass
)

print("\nSchema internal consistency:")
print(
    SCHEMA_VALIDATED
)

print("\nCoordinate interpretation:")
print(
    "E = east/longitude-like geographic degrees"
)

print(
    "N = north/latitude-like geographic degrees"
)

print("\nCRS:")
print(
    "UNVERIFIED — DO NOT CLAIM WGS84 YET"
)

print("\nFirst validation rows:")

print(
    validation_df[
        [
            "sample_id",
            "satellite_image",
            "map_size",
            "predicted_label_row",
            "predicted_label_col",
            "recorded_label_row",
            "recorded_label_col",
            "max_label_error_px",
        ]
    ]
    .head(12)
    .to_string(
        index=False
    )
)

print("\nValidation CSV:")
print(
    VALIDATION_CSV
)

print("\nGeoreference manifest:")
print(
    GEOREF_MANIFEST_CSV
)

print("\nValidation JSON:")
print(
    VALIDATION_JSON
)

print("\nNext action:")

if SCHEMA_VALIDATED:

    print(
        "Inspect the full vps_dataset using direct "
        "top-level sample access, then construct the "
        "full georeferenced benchmark manifest."
    )

else:

    print(
        "Resolve schema inconsistencies before "
        "building the full benchmark."
    )

NOTEBOOK 04 — VPS GEOREFERENCE SCHEMA VALIDATION

Samples:
5

✅ NOTEBOOK 04 CELL 3 SCHEMA VALIDATION COMPLETE

Samples:
5

Satellite records validated:
60

All bounding boxes valid:
True

All UAV coordinates inside satellite bounds:
True

Maximum label round-trip error:
1 px

Maximum center-distribution error:
1.062516741257e-11

Pixel round-trip validation:
True

Normalized-coordinate validation:
True

Schema internal consistency:
True

Coordinate interpretation:
E = east/longitude-like geographic degrees
N = north/latitude-like geographic degrees

CRS:
UNVERIFIED — DO NOT CLAIM WGS84 YET

First validation rows:
          sample_id satellite_image  map_size  predicted_label_row  predicted_label_col  recorded_label_row  recorded_label_col  max_label_error_px
Chuanmei_100_000004           0.jpg       700                  103                  731                 103                 731                   0
Chuanmei_100_000004           1.jpg       800                  537                 

In [26]:
# ============================================================
# NOTEBOOK 04 — CELL 4A
# VPS full-dataset structure + archive metadata audit
#
# NO extraction
# NO deep image scan
# ============================================================

from pathlib import Path
import zipfile
import json
import time

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

VPS_ROOT = (
    MYDRIVE
    / "vps_dataset"
)

VPS_SUBSET = (
    MYDRIVE
    / "vps_dataset_subset"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

print("=" * 88)
print("NOTEBOOK 04 — VPS DATASET STRUCTURE AUDIT")
print("=" * 88)

# ------------------------------------------------------------
# 1. Basic existence
# ------------------------------------------------------------

print("\nvps_dataset:")
print(
    "✅" if VPS_ROOT.exists() else "❌",
    VPS_ROOT
)

print("\nvps_dataset_subset:")
print(
    "✅" if VPS_SUBSET.exists() else "❌",
    VPS_SUBSET
)

print("\nvps_dataset_archive.zip:")
print(
    "✅" if VPS_ARCHIVE.is_file() else "❌",
    VPS_ARCHIVE
)

# ------------------------------------------------------------
# 2. Top-level structure only
# ------------------------------------------------------------

def safe_listdir(
    root,
    limit=100,
):
    try:
        items = sorted(
            root.iterdir(),
            key=lambda p: p.name
        )
    except Exception as e:
        print(
            "ERROR:",
            repr(e)
        )
        return []

    return items[:limit]


print("\n" + "=" * 88)
print("VPS_DATASET TOP-LEVEL")
print("=" * 88)

top_items = safe_listdir(
    VPS_ROOT,
    100,
)

print(
    "Shown entries:",
    len(top_items)
)

for item in top_items:
    print(
        "[DIR] " if item.is_dir() else "[FILE]",
        item.name
    )

# ------------------------------------------------------------
# 3. Inspect first-level directories only
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("FIRST-LEVEL DIRECTORY STRUCTURE")
print("=" * 88)

first_level_dirs = [
    p
    for p in top_items
    if p.is_dir()
][:25]

for directory in first_level_dirs:

    print(
        f"\n--- {directory.name} ---"
    )

    children = safe_listdir(
        directory,
        30,
    )

    for child in children:
        print(
            "   [DIR] "
            if child.is_dir()
            else "   [FILE]",
            child.name
        )

# ------------------------------------------------------------
# 4. Check for obvious metadata filenames
#    without recursive scanning
# ------------------------------------------------------------

metadata_name_hits = []

keywords = [
    "gps",
    "label",
    "meta",
    "coord",
    "position",
    "location",
]

for item in top_items:

    if item.is_file():

        name_lower = (
            item.name.lower()
        )

        if any(
            k in name_lower
            for k in keywords
        ):
            metadata_name_hits.append(
                str(item)
            )

    elif item.is_dir():

        children = safe_listdir(
            item,
            100,
        )

        for child in children:

            name_lower = (
                child.name.lower()
            )

            if (
                child.is_file()
                and any(
                    k in name_lower
                    for k in keywords
                )
            ):
                metadata_name_hits.append(
                    str(child)
                )

print("\n" + "=" * 88)
print("DIRECT METADATA FILENAME HITS")
print("=" * 88)

print(
    "Count:",
    len(metadata_name_hits)
)

for path in metadata_name_hits[:100]:
    print("✅", path)

# ------------------------------------------------------------
# 5. Inspect ZIP CENTRAL DIRECTORY ONLY
# ------------------------------------------------------------
# Does NOT extract 26.6 GB archive.
# We only inspect file names stored in ZIP index.
# ------------------------------------------------------------

archive_gps = []
archive_labels = []
archive_metadata = []

archive_total_files = None
archive_error = None

if VPS_ARCHIVE.is_file():

    print("\n" + "=" * 88)
    print("ARCHIVE INDEX INSPECTION")
    print("=" * 88)

    print(
        "Reading ZIP central directory..."
    )

    start = time.time()

    try:

        with zipfile.ZipFile(
            VPS_ARCHIVE,
            "r",
        ) as zf:

            infos = (
                zf.infolist()
            )

            archive_total_files = (
                len(infos)
            )

            for info in infos:

                name = (
                    info.filename
                )

                lower = (
                    name.lower()
                )

                if lower.endswith(
                    "gps_info.json"
                ):
                    archive_gps.append(
                        name
                    )

                if lower.endswith(
                    "labels.json"
                ):
                    archive_labels.append(
                        name
                    )

                basename = (
                    Path(name)
                    .name
                    .lower()
                )

                if any(
                    keyword
                    in basename
                    for keyword
                    in keywords
                ):
                    archive_metadata.append(
                        name
                    )

        elapsed = (
            time.time()
            - start
        )

        print(
            f"✅ Archive index read in "
            f"{elapsed:.2f} sec"
        )

    except Exception as error:

        archive_error = repr(
            error
        )

        print(
            "❌ Archive inspection failed:"
        )

        print(
            archive_error
        )

# ------------------------------------------------------------
# 6. Archive summary
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("ARCHIVE METADATA SUMMARY")
print("=" * 88)

print(
    "Total archive entries:",
    archive_total_files
)

print(
    "\nGPS_info.json files inside archive:",
    len(
        archive_gps
    )
)

for name in archive_gps[:30]:
    print("✅", name)

print(
    "\nlabels.json files inside archive:",
    len(
        archive_labels
    )
)

for name in archive_labels[:20]:
    print("✅", name)

print(
    "\nOther metadata-like archive files:",
    len(
        archive_metadata
    )
)

for name in archive_metadata[:30]:
    print("✅", name)

# ------------------------------------------------------------
# 7. Compare subset IDs against archive
# ------------------------------------------------------------

subset_ids = []

if VPS_SUBSET.is_dir():

    subset_ids = sorted(
        p.name
        for p in VPS_SUBSET.iterdir()
        if p.is_dir()
    )

print("\n" + "=" * 88)
print("SUBSET SAMPLE IDS")
print("=" * 88)

print(
    "Count:",
    len(subset_ids)
)

for sample_id in subset_ids[:30]:
    print(
        "✅",
        sample_id
    )

# ------------------------------------------------------------
# 8. Determine next route
# ------------------------------------------------------------

if archive_gps:

    status = (
        "FULL_GPS_METADATA_FOUND_IN_ARCHIVE"
    )

elif metadata_name_hits:

    status = (
        "POSSIBLE_METADATA_FOUND_IN_DIRECTORY"
    )

else:

    status = (
        "FULL_DATASET_GPS_METADATA_NOT_YET_FOUND"
    )

print("\n" + "=" * 88)
print("✅ NOTEBOOK 04 CELL 4A COMPLETE")
print("=" * 88)

print("\nStatus:")
print(
    status
)

print("\nGPS_info.json in archive:")
print(
    len(
        archive_gps
    )
)

print("\nlabels.json in archive:")
print(
    len(
        archive_labels
    )
)

print("\nDirect-directory metadata hits:")
print(
    len(
        metadata_name_hits
    )
)

print("\nNext action:")

if archive_gps:

    print(
        "Use GPS_info.json and labels.json "
        "directly from the ZIP archive without "
        "extracting the complete 26.6 GB dataset."
    )

elif metadata_name_hits:

    print(
        "Inspect the discovered metadata files "
        "before searching deeper."
    )

else:

    print(
        "Determine how vps_dataset is organized "
        "from the printed directory structure."
    )

NOTEBOOK 04 — VPS DATASET STRUCTURE AUDIT

vps_dataset:
✅ /content/drive/MyDrive/vps_dataset

vps_dataset_subset:
✅ /content/drive/MyDrive/vps_dataset_subset

vps_dataset_archive.zip:
✅ /content/drive/MyDrive/vps_dataset_archive.zip

VPS_DATASET TOP-LEVEL
Shown entries: 1
[DIR]  map2019

FIRST-LEVEL DIRECTORY STRUCTURE

--- map2019 ---
   [DIR]  merge_test_700-1800_cr0.95_stride100
   [DIR]  old_val
   [DIR]  train

DIRECT METADATA FILENAME HITS
Count: 0

ARCHIVE INDEX INSPECTION
Reading ZIP central directory...
✅ Archive index read in 3.86 sec

ARCHIVE METADATA SUMMARY
Total archive entries: 94685

GPS_info.json files inside archive: 2680
✅ map2019/merge_test_700-1800_cr0.95_stride100/Ligong_100_000002/GPS_info.json
✅ map2019/merge_test_700-1800_cr0.95_stride100/Hangdian_90_000004/GPS_info.json
✅ map2019/merge_test_700-1800_cr0.95_stride100/Ligong_90_000182/GPS_info.json
✅ map2019/merge_test_700-1800_cr0.95_stride100/Hangdian_90_000157/GPS_info.json
✅ map2019/merge_test_700-1800_cr0.9

In [27]:
# ============================================================
# NOTEBOOK 04 — CELL 4B
# Build FULL VPS georeferenced manifest DIRECTLY FROM ZIP
#
# - No 26.6 GB extraction
# - Reads only ZIP metadata + filenames
# - Validates GPS <-> labels consistency
# - Discovers actual UAV/satellite image members
# - CRS remains UNVERIFIED
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import zipfile
import json
import math
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

for directory in [
    AUDIT_ROOT,
    MANIFEST_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if not VPS_ARCHIVE.is_file():
    raise FileNotFoundError(
        VPS_ARCHIVE
    )


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

LABEL_IMAGE_SIZE = 768

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------

def get_suffix(
    name
):
    return Path(
        name
    ).suffix.lower()


def parse_sample_tokens(
    sample_id
):
    parts = (
        sample_id.split("_")
    )

    result = {
        "site_token": None,
        "nominal_level_token": None,
        "sequence_token": None,
    }

    if len(parts) >= 3:

        result[
            "site_token"
        ] = parts[0]

        result[
            "nominal_level_token"
        ] = parts[-2]

        result[
            "sequence_token"
        ] = parts[-1]

    return result


def projected_position(
    uav_e,
    uav_n,
    tl_e,
    tl_n,
    br_e,
    br_n,
    map_size,
):

    native_x = (
        (uav_e - tl_e)
        /
        (br_e - tl_e)
        *
        map_size
    )

    native_y = (
        (tl_n - uav_n)
        /
        (tl_n - br_n)
        *
        map_size
    )

    label_col_float = (
        native_x
        * LABEL_IMAGE_SIZE
        / map_size
    )

    label_row_float = (
        native_y
        * LABEL_IMAGE_SIZE
        / map_size
    )

    return (
        native_x,
        native_y,
        label_row_float,
        label_col_float,
    )


# ------------------------------------------------------------
# 4. Read ZIP central directory
# ------------------------------------------------------------

print("=" * 92)
print("NOTEBOOK 04 — FULL VPS GEOREFERENCE MANIFEST FROM ZIP")
print("=" * 92)

print("\nArchive:")
print(
    VPS_ARCHIVE
)

start_total = time.time()

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    infos = (
        zf.infolist()
    )

    member_names = [
        info.filename
        for info in infos
        if not info.is_dir()
    ]

    member_set = set(
        member_names
    )

    print("\nArchive files:")
    print(
        f"{len(member_names):,}"
    )

    gps_members = sorted(
        name
        for name in member_names
        if name.lower().endswith(
            "/gps_info.json"
        )
    )

    label_members = set(
        name
        for name in member_names
        if name.lower().endswith(
            "/labels.json"
        )
    )

    print("\nGPS_info.json:")
    print(
        f"{len(gps_members):,}"
    )

    print("\nlabels.json:")
    print(
        f"{len(label_members):,}"
    )

    if not gps_members:
        raise RuntimeError(
            "No GPS_info.json files found."
        )

    # --------------------------------------------------------
    # 5. Group direct image members by sample directory
    # --------------------------------------------------------

    images_by_parent = defaultdict(
        list
    )

    for name in member_names:

        if (
            get_suffix(name)
            not in IMAGE_SUFFIXES
        ):
            continue

        parent = str(
            Path(name).parent
        )

        images_by_parent[
            parent
        ].append(
            name
        )

    # --------------------------------------------------------
    # 6. Parse all GPS records
    # --------------------------------------------------------

    georef_records = []
    sample_records = []
    errors = []

    split_counter = Counter()

    for index, gps_member in enumerate(
        gps_members,
        start=1,
    ):

        parent = str(
            Path(
                gps_member
            ).parent
        )

        sample_id = (
            Path(
                parent
            ).name
        )

        path_parts = (
            Path(
                gps_member
            ).parts
        )

        dataset_partition = (
            path_parts[1]
            if len(
                path_parts
            ) > 1
            else "UNKNOWN"
        )

        split_counter[
            dataset_partition
        ] += 1

        label_member = (
            parent
            + "/labels.json"
        )

        labels_available = (
            label_member
            in label_members
        )

        # ----------------------------------------------------
        # Read GPS JSON
        # ----------------------------------------------------

        try:

            gps_data = json.loads(
                zf.read(
                    gps_member
                ).decode(
                    "utf-8"
                )
            )

        except Exception as error:

            errors.append({
                "gps_member": (
                    gps_member
                ),
                "error_type": (
                    "GPS_READ_ERROR"
                ),
                "error": repr(
                    error
                ),
            })

            continue

        # ----------------------------------------------------
        # Read labels JSON
        # ----------------------------------------------------

        labels = {}

        if labels_available:

            try:

                labels = json.loads(
                    zf.read(
                        label_member
                    ).decode(
                        "utf-8"
                    )
                )

            except Exception as error:

                errors.append({
                    "gps_member": (
                        gps_member
                    ),
                    "error_type": (
                        "LABEL_READ_ERROR"
                    ),
                    "error": repr(
                        error
                    ),
                })

                labels = {}

        # ----------------------------------------------------
        # Validate main schema
        # ----------------------------------------------------

        try:

            uav_e = float(
                gps_data[
                    "UAV"
                ][
                    "E"
                ]
            )

            uav_n = float(
                gps_data[
                    "UAV"
                ][
                    "N"
                ]
            )

            satellite_info = (
                gps_data[
                    "Satellite"
                ]
            )

        except Exception as error:

            errors.append({
                "gps_member": (
                    gps_member
                ),
                "error_type": (
                    "GPS_SCHEMA_ERROR"
                ),
                "error": repr(
                    error
                ),
            })

            continue

        # ----------------------------------------------------
        # Image inventory for sample
        # ----------------------------------------------------

        direct_images = sorted(
            images_by_parent.get(
                parent,
                []
            )
        )

        satellite_names = set(
            satellite_info.keys()
        )

        satellite_members = []

        for image_name in (
            satellite_names
        ):

            candidate = (
                parent
                + "/"
                + image_name
            )

            if candidate in member_set:

                satellite_members.append(
                    candidate
                )

        uav_candidate_images = [
            image_member
            for image_member
            in direct_images
            if Path(
                image_member
            ).name
            not in satellite_names
        ]

        tokens = parse_sample_tokens(
            sample_id
        )

        # ----------------------------------------------------
        # Parse satellite entries
        # ----------------------------------------------------

        valid_satellite_records = 0
        inside_count = 0
        matched_label_count = 0

        sample_max_pixel_error = np.nan
        sample_max_center_error = np.nan

        pixel_errors = []
        center_errors = []

        for (
            satellite_image,
            info
        ) in satellite_info.items():

            try:

                tl_e = float(
                    info[
                        "tl_E"
                    ]
                )

                tl_n = float(
                    info[
                        "tl_N"
                    ]
                )

                br_e = float(
                    info[
                        "br_E"
                    ]
                )

                br_n = float(
                    info[
                        "br_N"
                    ]
                )

                map_size = int(
                    info[
                        "map_size"
                    ]
                )

                recorded_center_x = float(
                    info[
                        "center_distribute_X"
                    ]
                )

                recorded_center_y = float(
                    info[
                        "center_distribute_Y"
                    ]
                )

            except Exception as error:

                errors.append({
                    "gps_member": (
                        gps_member
                    ),
                    "error_type": (
                        "SATELLITE_SCHEMA_ERROR"
                    ),
                    "satellite_image": (
                        satellite_image
                    ),
                    "error": repr(
                        error
                    ),
                })

                continue

            bbox_valid = bool(
                tl_e < br_e
                and
                tl_n > br_n
                and
                map_size > 0
            )

            if not bbox_valid:

                errors.append({
                    "gps_member": (
                        gps_member
                    ),
                    "error_type": (
                        "INVALID_BBOX"
                    ),
                    "satellite_image": (
                        satellite_image
                    ),
                })

                continue

            (
                native_x,
                native_y,
                predicted_row_float,
                predicted_col_float,
            ) = projected_position(
                uav_e,
                uav_n,
                tl_e,
                tl_n,
                br_e,
                br_n,
                map_size,
            )

            inside_bbox = bool(
                0.0
                <= native_x
                <= map_size
                and
                0.0
                <= native_y
                <= map_size
            )

            if inside_bbox:
                inside_count += 1

            # ------------------------------------------------
            # Normalized coordinate consistency
            # ------------------------------------------------

            predicted_center_x = (
                1.0
                -
                (
                    2.0
                    * native_y
                    / map_size
                )
            )

            predicted_center_y = (
                1.0
                -
                (
                    2.0
                    * native_x
                    / map_size
                )
            )

            center_x_error = abs(
                predicted_center_x
                - recorded_center_x
            )

            center_y_error = abs(
                predicted_center_y
                - recorded_center_y
            )

            max_center_error = max(
                center_x_error,
                center_y_error,
            )

            center_errors.append(
                max_center_error
            )

            # ------------------------------------------------
            # labels.json consistency
            # ------------------------------------------------

            label_available = bool(
                satellite_image
                in labels
            )

            recorded_row = np.nan
            recorded_col = np.nan
            pixel_error = np.nan

            if label_available:

                label = labels[
                    satellite_image
                ]

                if (
                    isinstance(
                        label,
                        list,
                    )
                    and
                    len(label) == 2
                ):

                    recorded_row = int(
                        label[0]
                    )

                    recorded_col = int(
                        label[1]
                    )

                    predicted_row = int(
                        math.floor(
                            predicted_row_float
                        )
                    )

                    predicted_col = int(
                        math.floor(
                            predicted_col_float
                        )
                    )

                    pixel_error = max(
                        abs(
                            predicted_row
                            - recorded_row
                        ),
                        abs(
                            predicted_col
                            - recorded_col
                        ),
                    )

                    pixel_errors.append(
                        pixel_error
                    )

                    matched_label_count += 1

            # ------------------------------------------------
            # Satellite member existence
            # ------------------------------------------------

            satellite_member = (
                parent
                + "/"
                + satellite_image
            )

            satellite_image_exists = (
                satellite_member
                in member_set
            )

            # ------------------------------------------------
            # Add georeference row
            # ------------------------------------------------

            georef_records.append({
                "dataset_partition": (
                    dataset_partition
                ),

                "sample_id": (
                    sample_id
                ),

                "site_token": (
                    tokens[
                        "site_token"
                    ]
                ),

                "nominal_level_token": (
                    tokens[
                        "nominal_level_token"
                    ]
                ),

                "sequence_token": (
                    tokens[
                        "sequence_token"
                    ]
                ),

                "gps_member": (
                    gps_member
                ),

                "labels_member": (
                    label_member
                    if labels_available
                    else None
                ),

                "satellite_image": (
                    satellite_image
                ),

                "satellite_member": (
                    satellite_member
                ),

                "satellite_image_exists": (
                    satellite_image_exists
                ),

                "uav_E": (
                    uav_e
                ),

                "uav_N": (
                    uav_n
                ),

                "tl_E": (
                    tl_e
                ),

                "tl_N": (
                    tl_n
                ),

                "br_E": (
                    br_e
                ),

                "br_N": (
                    br_n
                ),

                "satellite_center_E": (
                    (
                        tl_e
                        + br_e
                    )
                    / 2.0
                ),

                "satellite_center_N": (
                    (
                        tl_n
                        + br_n
                    )
                    / 2.0
                ),

                "map_size": (
                    map_size
                ),

                "uav_inside_bbox": (
                    inside_bbox
                ),

                "predicted_label_row_float": (
                    predicted_row_float
                ),

                "predicted_label_col_float": (
                    predicted_col_float
                ),

                "recorded_label_row": (
                    recorded_row
                ),

                "recorded_label_col": (
                    recorded_col
                ),

                "label_available": (
                    label_available
                ),

                "pixel_roundtrip_error_px": (
                    pixel_error
                ),

                "recorded_center_distribute_X": (
                    recorded_center_x
                ),

                "recorded_center_distribute_Y": (
                    recorded_center_y
                ),

                "predicted_center_distribute_X": (
                    predicted_center_x
                ),

                "predicted_center_distribute_Y": (
                    predicted_center_y
                ),

                "center_distribution_error": (
                    max_center_error
                ),

                "coordinate_schema": (
                    "GEOGRAPHIC_DEGREES_"
                    "EAST_NORTH"
                ),

                "crs_status": (
                    "CRS_UNVERIFIED"
                ),
            })

            valid_satellite_records += 1

        # ----------------------------------------------------
        # Sample-level validation summary
        # ----------------------------------------------------

        if pixel_errors:

            sample_max_pixel_error = float(
                max(
                    pixel_errors
                )
            )

        if center_errors:

            sample_max_center_error = float(
                max(
                    center_errors
                )
            )

        sample_records.append({
            "dataset_partition": (
                dataset_partition
            ),

            "sample_id": (
                sample_id
            ),

            "site_token": (
                tokens[
                    "site_token"
                ]
            ),

            "nominal_level_token": (
                tokens[
                    "nominal_level_token"
                ]
            ),

            "sequence_token": (
                tokens[
                    "sequence_token"
                ]
            ),

            "uav_E": (
                uav_e
            ),

            "uav_N": (
                uav_n
            ),

            "gps_member": (
                gps_member
            ),

            "labels_available": (
                labels_available
            ),

            "satellite_records_declared": int(
                len(
                    satellite_info
                )
            ),

            "satellite_records_valid": (
                valid_satellite_records
            ),

            "satellite_members_found": int(
                len(
                    satellite_members
                )
            ),

            "inside_bbox_count": (
                inside_count
            ),

            "matched_label_count": (
                matched_label_count
            ),

            "max_pixel_roundtrip_error_px": (
                sample_max_pixel_error
            ),

            "max_center_distribution_error": (
                sample_max_center_error
            ),

            "direct_image_count": int(
                len(
                    direct_images
                )
            ),

            "uav_candidate_image_count": int(
                len(
                    uav_candidate_images
                )
            ),

            "uav_candidate_members": (
                " | ".join(
                    uav_candidate_images
                )
            ),
        })

        if (
            index % 250 == 0
            or
            index == len(
                gps_members
            )
        ):

            print(
                f"Parsed "
                f"{index:,} / "
                f"{len(gps_members):,} "
                f"GPS samples"
            )


# ------------------------------------------------------------
# 7. Build DataFrames
# ------------------------------------------------------------

georef_df = pd.DataFrame(
    georef_records
)

samples_df = pd.DataFrame(
    sample_records
)

errors_df = pd.DataFrame(
    errors
)

if georef_df.empty:
    raise RuntimeError(
        "No georeferenced records produced."
    )


# ------------------------------------------------------------
# 8. Global validation
# ------------------------------------------------------------

valid_sample_count = int(
    len(
        samples_df
    )
)

satellite_record_count = int(
    len(
        georef_df
    )
)

samples_with_labels = int(
    samples_df[
        "labels_available"
    ].sum()
)

samples_without_labels = (
    valid_sample_count
    - samples_with_labels
)

all_inside = bool(
    georef_df[
        "uav_inside_bbox"
    ].all()
)

all_satellite_images_found = bool(
    georef_df[
        "satellite_image_exists"
    ].all()
)

label_rows = (
    georef_df[
        georef_df[
            "label_available"
        ]
        == True
    ]
)

label_validation_count = int(
    len(
        label_rows
    )
)

if (
    label_validation_count
    > 0
):

    max_pixel_error = float(
        label_rows[
            "pixel_roundtrip_error_px"
        ]
        .dropna()
        .max()
    )

else:

    max_pixel_error = None

max_center_error = float(
    georef_df[
        "center_distribution_error"
    ].max()
)

pixel_validation_pass = bool(
    max_pixel_error is None
    or
    max_pixel_error <= 1
)

center_validation_pass = bool(
    max_center_error
    < 1e-8
)

schema_consistency = bool(
    all_inside
    and
    pixel_validation_pass
    and
    center_validation_pass
)


# ------------------------------------------------------------
# 9. Image inventory summary
# ------------------------------------------------------------

samples_one_uav_candidate = int(
    (
        samples_df[
            "uav_candidate_image_count"
        ]
        == 1
    ).sum()
)

samples_zero_uav_candidate = int(
    (
        samples_df[
            "uav_candidate_image_count"
        ]
        == 0
    ).sum()
)

samples_multiple_uav_candidates = int(
    (
        samples_df[
            "uav_candidate_image_count"
        ]
        > 1
    ).sum()
)


# ------------------------------------------------------------
# 10. Partition / site summaries
# ------------------------------------------------------------

partition_summary = (
    samples_df
    .groupby(
        "dataset_partition",
        dropna=False,
    )
    .agg(
        samples=(
            "sample_id",
            "count",
        ),

        samples_with_labels=(
            "labels_available",
            "sum",
        ),

        uav_candidate_images=(
            "uav_candidate_image_count",
            "sum",
        ),
    )
    .reset_index()
)

site_summary = (
    samples_df
    .groupby(
        "site_token",
        dropna=False,
    )
    .size()
    .reset_index(
        name="samples"
    )
    .sort_values(
        "samples",
        ascending=False,
    )
)


# ------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------

FULL_GEOREF_CSV = (
    MANIFEST_ROOT
    / "vps_archive_full_georeference_manifest.csv"
)

SAMPLE_MANIFEST_CSV = (
    MANIFEST_ROOT
    / "vps_archive_uav_sample_manifest.csv"
)

PARTITION_CSV = (
    AUDIT_ROOT
    / "vps_archive_partition_summary.csv"
)

SITE_CSV = (
    AUDIT_ROOT
    / "vps_archive_site_summary.csv"
)

ERROR_CSV = (
    AUDIT_ROOT
    / "vps_archive_metadata_errors.csv"
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "vps_archive_full_georeference_audit.json"
)

georef_df.to_csv(
    FULL_GEOREF_CSV,
    index=False,
)

samples_df.to_csv(
    SAMPLE_MANIFEST_CSV,
    index=False,
)

partition_summary.to_csv(
    PARTITION_CSV,
    index=False,
)

site_summary.to_csv(
    SITE_CSV,
    index=False,
)

errors_df.to_csv(
    ERROR_CSV,
    index=False,
)

runtime_seconds = (
    time.time()
    - start_total
)

audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_archive": str(
        VPS_ARCHIVE
    ),

    "source_mode": (
        "ZIP_CENTRAL_DIRECTORY_AND_"
        "DIRECT_METADATA_READ_NO_FULL_EXTRACTION"
    ),

    "archive_file_count": int(
        len(
            member_names
        )
    ),

    "gps_info_count": int(
        len(
            gps_members
        )
    ),

    "labels_json_count": int(
        len(
            label_members
        )
    ),

    "valid_gps_sample_count": (
        valid_sample_count
    ),

    "georeferenced_satellite_record_count": (
        satellite_record_count
    ),

    "samples_with_labels": (
        samples_with_labels
    ),

    "samples_without_labels": (
        samples_without_labels
    ),

    "label_validation_record_count": (
        label_validation_count
    ),

    "all_uav_coordinates_inside_bounds": (
        all_inside
    ),

    "all_referenced_satellite_images_found": (
        all_satellite_images_found
    ),

    "maximum_pixel_roundtrip_error_px": (
        max_pixel_error
    ),

    "maximum_center_distribution_error": (
        max_center_error
    ),

    "pixel_validation_pass": (
        pixel_validation_pass
    ),

    "normalized_coordinate_validation_pass": (
        center_validation_pass
    ),

    "schema_internal_consistency": (
        schema_consistency
    ),

    "samples_with_exactly_one_uav_candidate_image": (
        samples_one_uav_candidate
    ),

    "samples_with_zero_uav_candidate_images": (
        samples_zero_uav_candidate
    ),

    "samples_with_multiple_uav_candidate_images": (
        samples_multiple_uav_candidates
    ),

    "metadata_error_count": int(
        len(
            errors_df
        )
    ),

    "coordinate_interpretation": {
        "E": (
            "EAST_LONGITUDE_LIKE_"
            "GEOGRAPHIC_DEGREES"
        ),

        "N": (
            "NORTH_LATITUDE_LIKE_"
            "GEOGRAPHIC_DEGREES"
        ),
    },

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "metric_distance_claim_status": (
        "BLOCKED_UNTIL_CRS_PROVENANCE_VERIFIED"
    ),

    "nominal_level_token_policy": (
        "PARSED_FROM_SAMPLE_NAME_ONLY_"
        "NOT_CLAIMED_AS_VERIFIED_AGL_OR_MSL"
    ),

    "retrieval_policy": (
        "RETRIEVAL_RESULT_IS_NOT_VERIFIED_UAV_POSE"
    ),

    "runtime_seconds": (
        runtime_seconds
    ),

    "outputs": {
        "georeference_manifest": str(
            FULL_GEOREF_CSV
        ),

        "sample_manifest": str(
            SAMPLE_MANIFEST_CSV
        ),

        "partition_summary": str(
            PARTITION_CSV
        ),

        "site_summary": str(
            SITE_CSV
        ),

        "metadata_errors": str(
            ERROR_CSV
        ),
    },
}

AUDIT_JSON.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("✅ NOTEBOOK 04 CELL 4B COMPLETE")
print("=" * 92)

print("\nValid GPS samples:")
print(
    f"{valid_sample_count:,}"
)

print("\nGeoreferenced satellite records:")
print(
    f"{satellite_record_count:,}"
)

print("\nSamples with labels.json:")
print(
    f"{samples_with_labels:,}"
)

print("\nSamples without labels.json:")
print(
    f"{samples_without_labels:,}"
)

print("\nLabel-validated satellite records:")
print(
    f"{label_validation_count:,}"
)

print("\nAll UAV positions inside satellite bounds:")
print(
    all_inside
)

print("\nMaximum pixel round-trip error:")
print(
    max_pixel_error
)

print("\nMaximum center-distribution error:")
print(
    f"{max_center_error:.12e}"
)

print("\nSchema internal consistency:")
print(
    schema_consistency
)

print("\nAll referenced satellite images found:")
print(
    all_satellite_images_found
)

print("\nSamples with exactly ONE UAV candidate image:")
print(
    f"{samples_one_uav_candidate:,}"
)

print("\nSamples with ZERO UAV candidate images:")
print(
    f"{samples_zero_uav_candidate:,}"
)

print("\nSamples with MULTIPLE UAV candidate images:")
print(
    f"{samples_multiple_uav_candidates:,}"
)

print("\nMetadata errors:")
print(
    f"{len(errors_df):,}"
)

print("\nPartition summary:")
print(
    partition_summary.to_string(
        index=False
    )
)

print("\nSite summary:")
print(
    site_summary.to_string(
        index=False
    )
)

print("\nCRS:")
print(
    "UNVERIFIED — WGS84 NOT CLAIMED"
)

print("\nRuntime:")
print(
    f"{runtime_seconds:.2f} sec"
)

print("\nFull georeference manifest:")
print(
    FULL_GEOREF_CSV
)

print("\nUAV sample manifest:")
print(
    SAMPLE_MANIFEST_CSV
)

print("\nAudit JSON:")
print(
    AUDIT_JSON
)

print("\nNext action:")

if schema_consistency:

    print(
        "Metadata validation passed. "
        "Next: resolve the real UAV query image member "
        "and satellite gallery member structure, then "
        "prepare the georeferenced retrieval benchmark."
    )

else:

    print(
        "Review metadata inconsistencies before "
        "constructing the retrieval benchmark."
    )

NOTEBOOK 04 — FULL VPS GEOREFERENCE MANIFEST FROM ZIP

Archive:
/content/drive/MyDrive/vps_dataset_archive.zip

Archive files:
63,816

GPS_info.json:
2,680

labels.json:
3,520
Parsed 250 / 2,680 GPS samples
Parsed 500 / 2,680 GPS samples
Parsed 750 / 2,680 GPS samples
Parsed 1,000 / 2,680 GPS samples
Parsed 1,250 / 2,680 GPS samples
Parsed 1,500 / 2,680 GPS samples
Parsed 1,750 / 2,680 GPS samples
Parsed 2,000 / 2,680 GPS samples
Parsed 2,250 / 2,680 GPS samples
Parsed 2,500 / 2,680 GPS samples
Parsed 2,680 / 2,680 GPS samples

✅ NOTEBOOK 04 CELL 4B COMPLETE

Valid GPS samples:
2,680

Georeferenced satellite records:
32,160

Samples with labels.json:
2,680

Samples without labels.json:
0

Label-validated satellite records:
32,160

All UAV positions inside satellite bounds:
True

Maximum pixel round-trip error:
2.0

Maximum center-distribution error:
1.206179600644e-11

Schema internal consistency:
False

All referenced satellite images found:
False

Samples with exactly ONE UAV candida

In [28]:
# ============================================================
# NOTEBOOK 04 — CELL 4C
# Resolve actual image layout inside VPS ZIP
# + diagnose 2-pixel round-trip cases
#
# NO extraction
# NO GPS JSON reprocessing
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import zipfile
import json
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

GEOREF_CSV = (
    MANIFEST_ROOT
    / "vps_archive_full_georeference_manifest.csv"
)

SAMPLE_CSV = (
    MANIFEST_ROOT
    / "vps_archive_uav_sample_manifest.csv"
)

if not GEOREF_CSV.is_file():
    raise FileNotFoundError(
        GEOREF_CSV
    )

if not SAMPLE_CSV.is_file():
    raise FileNotFoundError(
        SAMPLE_CSV
    )

if not VPS_ARCHIVE.is_file():
    raise FileNotFoundError(
        VPS_ARCHIVE
    )


# ------------------------------------------------------------
# 2. Load existing manifests
# ------------------------------------------------------------

print("=" * 92)
print("NOTEBOOK 04 — VPS IMAGE-LAYOUT + PIXEL-ERROR AUDIT")
print("=" * 92)

georef_df = pd.read_csv(
    GEOREF_CSV
)

samples_df = pd.read_csv(
    SAMPLE_CSV
)

print("\nGeoreference rows:")
print(
    f"{len(georef_df):,}"
)

print("\nGPS samples:")
print(
    f"{len(samples_df):,}"
)


# ------------------------------------------------------------
# 3. Pixel round-trip error distribution
# ------------------------------------------------------------

pixel_errors = pd.to_numeric(
    georef_df[
        "pixel_roundtrip_error_px"
    ],
    errors="coerce",
)

pixel_error_counts = (
    pixel_errors
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 92)
print("PIXEL ROUND-TRIP ERROR DISTRIBUTION")
print("=" * 92)

total_labeled = int(
    pixel_errors.notna().sum()
)

for error_value, count in (
    pixel_error_counts.items()
):

    percentage = (
        100.0
        * count
        / total_labeled
    )

    print(
        f"{error_value:>2} px : "
        f"{count:>7,} "
        f"({percentage:6.3f}%)"
    )

count_le_1 = int(
    (
        pixel_errors
        <= 1
    ).sum()
)

count_eq_2 = int(
    (
        pixel_errors
        == 2
    ).sum()
)

count_gt_2 = int(
    (
        pixel_errors
        > 2
    ).sum()
)

print("\nRecords <= 1 px:")
print(
    f"{count_le_1:,} / "
    f"{total_labeled:,}"
)

print("\nRecords exactly 2 px:")
print(
    f"{count_eq_2:,}"
)

print("\nRecords > 2 px:")
print(
    f"{count_gt_2:,}"
)


# ------------------------------------------------------------
# 4. Show worst 2-pixel cases
# ------------------------------------------------------------

worst_pixel_rows = (
    georef_df[
        pixel_errors
        == pixel_errors.max()
    ]
    .copy()
)

print("\nWorst-error rows:")
print(
    len(
        worst_pixel_rows
    )
)

if not worst_pixel_rows.empty:

    columns_to_show = [
        "dataset_partition",
        "sample_id",
        "satellite_image",
        "map_size",
        "predicted_label_row_float",
        "predicted_label_col_float",
        "recorded_label_row",
        "recorded_label_col",
        "pixel_roundtrip_error_px",
    ]

    columns_to_show = [
        column
        for column in columns_to_show
        if column in worst_pixel_rows.columns
    ]

    print(
        worst_pixel_rows[
            columns_to_show
        ]
        .head(25)
        .to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 5. Build sample-parent lookup from GPS members
# ------------------------------------------------------------

sample_parent_to_id = {}

sample_id_to_parent = {}

for _, row in samples_df.iterrows():

    gps_member = str(
        row[
            "gps_member"
        ]
    )

    parent = str(
        Path(
            gps_member
        ).parent
    )

    sample_id = str(
        row[
            "sample_id"
        ]
    )

    sample_parent_to_id[
        parent
    ] = sample_id

    sample_id_to_parent[
        sample_id
    ] = parent

gps_parent_set = set(
    sample_parent_to_id.keys()
)


# ------------------------------------------------------------
# 6. Read ZIP central directory ONCE
# ------------------------------------------------------------

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

start = time.time()

images_by_sample = defaultdict(
    list
)

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    infos = zf.infolist()

    for info in infos:

        if info.is_dir():
            continue

        member = (
            info.filename
        )

        if (
            Path(
                member
            ).suffix.lower()
            not in IMAGE_SUFFIXES
        ):
            continue

        # Walk upward only a few levels until
        # the known GPS sample directory is found.
        candidate_parent = Path(
            member
        ).parent

        matched_parent = None

        for _ in range(5):

            candidate_string = str(
                candidate_parent
            )

            if (
                candidate_string
                in gps_parent_set
            ):

                matched_parent = (
                    candidate_string
                )

                break

            if (
                candidate_parent
                == candidate_parent.parent
            ):
                break

            candidate_parent = (
                candidate_parent.parent
            )

        if matched_parent is None:
            continue

        relative_path = str(
            Path(
                member
            ).relative_to(
                matched_parent
            )
        )

        sample_id = (
            sample_parent_to_id[
                matched_parent
            ]
        )

        images_by_sample[
            sample_id
        ].append(
            relative_path
        )

zip_index_runtime = (
    time.time()
    - start
)

print("\n" + "=" * 92)
print("ZIP IMAGE LAYOUT")
print("=" * 92)

print(
    f"ZIP index processed in "
    f"{zip_index_runtime:.2f} sec"
)

print("\nGPS samples with image descendants:")
print(
    f"{len(images_by_sample):,}"
)


# ------------------------------------------------------------
# 7. Summarize first relative image folder/component
# ------------------------------------------------------------

folder_sample_sets = defaultdict(
    set
)

folder_image_counts = Counter()

for sample_id, relative_paths in (
    images_by_sample.items()
):

    for relative_path in relative_paths:

        parts = Path(
            relative_path
        ).parts

        if len(parts) > 1:

            first_component = (
                parts[0]
            )

        else:

            first_component = (
                "<DIRECT_SAMPLE_ROOT>"
            )

        folder_sample_sets[
            first_component
        ].add(
            sample_id
        )

        folder_image_counts[
            first_component
        ] += 1

folder_rows = []

for folder_name in sorted(
    folder_image_counts,
    key=lambda name: (
        -len(
            folder_sample_sets[
                name
            ]
        ),
        name,
    ),
):

    folder_rows.append({
        "relative_component": (
            folder_name
        ),

        "samples_containing_component": int(
            len(
                folder_sample_sets[
                    folder_name
                ]
            )
        ),

        "image_file_count": int(
            folder_image_counts[
                folder_name
            ]
        ),
    })

folder_df = pd.DataFrame(
    folder_rows
)

print("\nImage-folder/component summary:")

if not folder_df.empty:

    print(
        folder_df
        .head(30)
        .to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 8. Show actual layout for first samples
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("EXAMPLE SAMPLE IMAGE MEMBERS")
print("=" * 92)

example_sample_ids = (
    samples_df[
        "sample_id"
    ]
    .astype(str)
    .head(8)
    .tolist()
)

for sample_id in example_sample_ids:

    print(
        f"\n--- {sample_id} ---"
    )

    relative_paths = sorted(
        images_by_sample.get(
            sample_id,
            []
        )
    )

    print(
        f"Image members: "
        f"{len(relative_paths)}"
    )

    for relative_path in (
        relative_paths[:40]
    ):

        print(
            "  ",
            relative_path
        )


# ------------------------------------------------------------
# 9. Resolve satellite image paths by basename
# ------------------------------------------------------------

# Build:
# sample -> basename -> relative paths

basename_map = {}

for sample_id, relative_paths in (
    images_by_sample.items()
):

    local_map = defaultdict(
        list
    )

    for relative_path in (
        relative_paths
    ):

        basename = (
            Path(
                relative_path
            ).name
        )

        local_map[
            basename
        ].append(
            relative_path
        )

    basename_map[
        sample_id
    ] = local_map


satellite_resolution_records = []

for row_index, row in (
    georef_df.iterrows()
):

    sample_id = str(
        row[
            "sample_id"
        ]
    )

    satellite_image = str(
        row[
            "satellite_image"
        ]
    )

    matches = (
        basename_map
        .get(
            sample_id,
            {}
        )
        .get(
            satellite_image,
            []
        )
    )

    satellite_resolution_records.append({
        "row_index": int(
            row_index
        ),

        "sample_id": (
            sample_id
        ),

        "satellite_image": (
            satellite_image
        ),

        "candidate_count": int(
            len(
                matches
            )
        ),

        "candidate_relative_paths": (
            " | ".join(
                matches
            )
        ),
    })

sat_resolution_df = pd.DataFrame(
    satellite_resolution_records
)

sat_candidate_distribution = (
    sat_resolution_df[
        "candidate_count"
    ]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 92)
print("SATELLITE BASENAME RESOLUTION")
print("=" * 92)

print(
    sat_candidate_distribution.to_string()
)

unique_satellite_rows = int(
    (
        sat_resolution_df[
            "candidate_count"
        ]
        == 1
    ).sum()
)

zero_satellite_rows = int(
    (
        sat_resolution_df[
            "candidate_count"
        ]
        == 0
    ).sum()
)

ambiguous_satellite_rows = int(
    (
        sat_resolution_df[
            "candidate_count"
        ]
        > 1
    ).sum()
)

print("\nUnique satellite matches:")
print(
    f"{unique_satellite_rows:,}"
)

print("\nMissing satellite matches:")
print(
    f"{zero_satellite_rows:,}"
)

print("\nAmbiguous satellite matches:")
print(
    f"{ambiguous_satellite_rows:,}"
)


# ------------------------------------------------------------
# 10. Determine satellite relative parent components
# ------------------------------------------------------------

unique_sat_rows = (
    sat_resolution_df[
        sat_resolution_df[
            "candidate_count"
        ]
        == 1
    ]
    .copy()
)

satellite_parent_counter = Counter()

for relative_path in (
    unique_sat_rows[
        "candidate_relative_paths"
    ].tolist()
):

    parent = str(
        Path(
            relative_path
        ).parent
    )

    satellite_parent_counter[
        parent
    ] += 1

print("\nSatellite relative-parent frequencies:")

for parent, count in (
    satellite_parent_counter.most_common(
        20
    )
):

    print(
        f"{count:>7,}  {parent}"
    )


# ------------------------------------------------------------
# 11. Resolve UAV candidate image per sample
# ------------------------------------------------------------

# For each sample:
# - collect all image paths
# - remove the paths that correspond to the
#   known 12 satellite metadata image basenames
# - inspect what remains

expected_satellite_by_sample = (
    georef_df
    .groupby(
        "sample_id"
    )[
        "satellite_image"
    ]
    .apply(
        lambda values: set(
            values.astype(str)
        )
    )
    .to_dict()
)

uav_resolution_records = []

for _, sample_row in (
    samples_df.iterrows()
):

    sample_id = str(
        sample_row[
            "sample_id"
        ]
    )

    image_paths = sorted(
        images_by_sample.get(
            sample_id,
            []
        )
    )

    expected_sat_names = (
        expected_satellite_by_sample
        .get(
            sample_id,
            set()
        )
    )

    remaining = []

    for relative_path in image_paths:

        basename = (
            Path(
                relative_path
            ).name
        )

        if basename not in (
            expected_sat_names
        ):

            remaining.append(
                relative_path
            )

    uav_resolution_records.append({
        "sample_id": (
            sample_id
        ),

        "total_image_members": int(
            len(
                image_paths
            )
        ),

        "non_satellite_candidate_count": int(
            len(
                remaining
            )
        ),

        "non_satellite_candidate_paths": (
            " | ".join(
                remaining
            )
        ),
    })

uav_resolution_df = pd.DataFrame(
    uav_resolution_records
)

uav_candidate_distribution = (
    uav_resolution_df[
        "non_satellite_candidate_count"
    ]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 92)
print("UAV QUERY IMAGE CANDIDATE RESOLUTION")
print("=" * 92)

print(
    uav_candidate_distribution.to_string()
)

exactly_one_uav = int(
    (
        uav_resolution_df[
            "non_satellite_candidate_count"
        ]
        == 1
    ).sum()
)

zero_uav = int(
    (
        uav_resolution_df[
            "non_satellite_candidate_count"
        ]
        == 0
    ).sum()
)

multiple_uav = int(
    (
        uav_resolution_df[
            "non_satellite_candidate_count"
        ]
        > 1
    ).sum()
)

print("\nSamples with exactly one candidate:")
print(
    f"{exactly_one_uav:,}"
)

print("\nSamples with zero candidates:")
print(
    f"{zero_uav:,}"
)

print("\nSamples with multiple candidates:")
print(
    f"{multiple_uav:,}"
)


# ------------------------------------------------------------
# 12. UAV candidate parent frequencies
# ------------------------------------------------------------

uav_parent_counter = Counter()

single_uav_rows = (
    uav_resolution_df[
        uav_resolution_df[
            "non_satellite_candidate_count"
        ]
        == 1
    ]
)

for relative_path in (
    single_uav_rows[
        "non_satellite_candidate_paths"
    ].tolist()
):

    parent = str(
        Path(
            relative_path
        ).parent
    )

    uav_parent_counter[
        parent
    ] += 1

print("\nUAV candidate relative-parent frequencies:")

for parent, count in (
    uav_parent_counter.most_common(
        20
    )
):

    print(
        f"{count:>7,}  {parent}"
    )


# ------------------------------------------------------------
# 13. Save diagnostics
# ------------------------------------------------------------

PIXEL_ERROR_CSV = (
    AUDIT_ROOT
    / "vps_pixel_roundtrip_error_distribution.csv"
)

FOLDER_LAYOUT_CSV = (
    AUDIT_ROOT
    / "vps_zip_image_folder_layout.csv"
)

SAT_RESOLUTION_CSV = (
    AUDIT_ROOT
    / "vps_satellite_member_resolution.csv"
)

UAV_RESOLUTION_CSV = (
    AUDIT_ROOT
    / "vps_uav_member_resolution.csv"
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "vps_image_layout_resolution_audit.json"
)

pixel_error_counts.rename_axis(
    "pixel_error_px"
).reset_index(
    name="count"
).to_csv(
    PIXEL_ERROR_CSV,
    index=False,
)

folder_df.to_csv(
    FOLDER_LAYOUT_CSV,
    index=False,
)

sat_resolution_df.to_csv(
    SAT_RESOLUTION_CSV,
    index=False,
)

uav_resolution_df.to_csv(
    UAV_RESOLUTION_CSV,
    index=False,
)

layout_fully_resolved = bool(
    unique_satellite_rows
    == len(
        georef_df
    )
    and
    exactly_one_uav
    == len(
        samples_df
    )
)

audit = {
    "pixel_error": {
        "label_record_count": (
            total_labeled
        ),

        "records_le_1_px": (
            count_le_1
        ),

        "records_equal_2_px": (
            count_eq_2
        ),

        "records_gt_2_px": (
            count_gt_2
        ),

        "maximum_px": float(
            pixel_errors.max()
        ),

        "tolerance_decision": (
            "PENDING_INTERPRETATION"
        ),
    },

    "image_layout": {
        "samples_with_image_descendants": int(
            len(
                images_by_sample
            )
        ),

        "satellite_rows_unique": (
            unique_satellite_rows
        ),

        "satellite_rows_missing": (
            zero_satellite_rows
        ),

        "satellite_rows_ambiguous": (
            ambiguous_satellite_rows
        ),

        "uav_samples_exactly_one_candidate": (
            exactly_one_uav
        ),

        "uav_samples_zero_candidates": (
            zero_uav
        ),

        "uav_samples_multiple_candidates": (
            multiple_uav
        ),

        "fully_resolved": (
            layout_fully_resolved
        ),
    },

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "runtime_seconds": (
        zip_index_runtime
    ),
}

AUDIT_JSON.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 14. Final
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("✅ NOTEBOOK 04 CELL 4C COMPLETE")
print("=" * 92)

print("\nPixel errors <= 1 px:")
print(
    f"{count_le_1:,} / "
    f"{total_labeled:,}"
)

print("\nPixel errors exactly 2 px:")
print(
    f"{count_eq_2:,}"
)

print("\nPixel errors > 2 px:")
print(
    f"{count_gt_2:,}"
)

print("\nUnique satellite member resolutions:")
print(
    f"{unique_satellite_rows:,} / "
    f"{len(georef_df):,}"
)

print("\nSamples with exactly one UAV candidate:")
print(
    f"{exactly_one_uav:,} / "
    f"{len(samples_df):,}"
)

print("\nImage layout fully resolved:")
print(
    layout_fully_resolved
)

print("\nCRS:")
print(
    "UNVERIFIED — WGS84 NOT CLAIMED"
)

print("\nAudit:")
print(
    AUDIT_JSON
)

print("\nNext action:")

if layout_fully_resolved:

    print(
        "Image paths are fully resolved. "
        "Next: create the executable georeferenced "
        "query/gallery benchmark manifest."
    )

else:

    print(
        "Use the printed folder/path structure to "
        "resolve the remaining UAV/satellite ambiguity."
    )

NOTEBOOK 04 — VPS IMAGE-LAYOUT + PIXEL-ERROR AUDIT

Georeference rows:
32,160

GPS samples:
2,680

PIXEL ROUND-TRIP ERROR DISTRIBUTION
 0 px :  27,638 (85.939%)
 1 px :   4,430 (13.775%)
 2 px :      92 ( 0.286%)

Records <= 1 px:
32,068 / 32,160

Records exactly 2 px:
92

Records > 2 px:
0

Worst-error rows:
92
                   dataset_partition          sample_id satellite_image  map_size  predicted_label_row_float  predicted_label_col_float  recorded_label_row  recorded_label_col  pixel_roundtrip_error_px
merge_test_700-1800_cr0.95_stride100 Chuanmei_80_000165           0.jpg       700                  80.091429                 301.714286                  78                 301                         2
merge_test_700-1800_cr0.95_stride100 Chuanmei_90_000009           0.jpg       700                  80.091429                 301.714286                  78                 301                         2
merge_test_700-1800_cr0.95_stride100 Chuanmei_90_000017           0.jpg       70

In [29]:
# ============================================================
# NOTEBOOK 04 — CELL 4D
# FIX partition/sample-ID collision
# Resolve exact UAV + Satellite ZIP members
# Build executable query/gallery manifests
#
# NO image extraction
# NO GPS JSON re-read
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import zipfile
import json
import time

import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

GEOREF_CSV = (
    MANIFEST_ROOT
    / "vps_archive_full_georeference_manifest.csv"
)

SAMPLE_CSV = (
    MANIFEST_ROOT
    / "vps_archive_uav_sample_manifest.csv"
)

for path in [
    VPS_ARCHIVE,
    GEOREF_CSV,
    SAMPLE_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Load existing metadata manifests
# ------------------------------------------------------------

print("=" * 94)
print("NOTEBOOK 04 — PARTITION-SAFE IMAGE MEMBER RESOLUTION")
print("=" * 94)

georef_df = pd.read_csv(
    GEOREF_CSV
)

samples_df = pd.read_csv(
    SAMPLE_CSV
)

print("\nGeoreference rows:")
print(
    f"{len(georef_df):,}"
)

print("\nGPS sample rows:")
print(
    f"{len(samples_df):,}"
)


# ------------------------------------------------------------
# 3. Create TRUE unique sample key
# ------------------------------------------------------------
# sample_id alone is NOT globally unique.
#
# Correct key:
# full parent path of GPS_info.json inside ZIP.
# ------------------------------------------------------------

georef_df[
    "sample_key"
] = georef_df[
    "gps_member"
].astype(str).map(
    lambda x: str(
        Path(x).parent
    )
)

samples_df[
    "sample_key"
] = samples_df[
    "gps_member"
].astype(str).map(
    lambda x: str(
        Path(x).parent
    )
)

unique_sample_keys = int(
    samples_df[
        "sample_key"
    ].nunique()
)

duplicate_sample_key_rows = int(
    samples_df[
        "sample_key"
    ].duplicated(
        keep=False
    ).sum()
)

# ------------------------------------------------------------
# 4. Diagnose sample_id collisions
# ------------------------------------------------------------

sample_id_counts = (
    samples_df[
        "sample_id"
    ]
    .astype(str)
    .value_counts()
)

duplicate_sample_ids = (
    sample_id_counts[
        sample_id_counts > 1
    ]
)

duplicate_sample_id_count = int(
    len(
        duplicate_sample_ids
    )
)

rows_with_duplicate_sample_id = int(
    duplicate_sample_ids.sum()
)

print("\n" + "=" * 94)
print("SAMPLE-ID COLLISION DIAGNOSIS")
print("=" * 94)

print("\nUnique full sample keys:")
print(
    f"{unique_sample_keys:,}"
)

print("\nDuplicate FULL sample-key rows:")
print(
    duplicate_sample_key_rows
)

print("\nRepeated plain sample_id values:")
print(
    f"{duplicate_sample_id_count:,}"
)

print("\nRows affected by repeated sample_id:")
print(
    f"{rows_with_duplicate_sample_id:,}"
)

print(
    "\nExpected explanation for previous ambiguity:"
)

print(
    "Plain sample_id merged different dataset partitions."
)


# ------------------------------------------------------------
# 5. Read ZIP index once
# ------------------------------------------------------------

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

sample_key_set = set(
    samples_df[
        "sample_key"
    ].astype(str)
)

images_by_sample_key = defaultdict(
    list
)

start = time.time()

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    for info in zf.infolist():

        if info.is_dir():
            continue

        member = (
            info.filename
        )

        if (
            Path(
                member
            ).suffix.lower()
            not in IMAGE_SUFFIXES
        ):
            continue

        candidate_parent = (
            Path(member).parent
        )

        matched_key = None

        # Search upward only a few levels.
        for _ in range(5):

            candidate_string = str(
                candidate_parent
            )

            if (
                candidate_string
                in sample_key_set
            ):

                matched_key = (
                    candidate_string
                )

                break

            if (
                candidate_parent
                == candidate_parent.parent
            ):
                break

            candidate_parent = (
                candidate_parent.parent
            )

        if matched_key is None:
            continue

        relative_member = str(
            Path(member).relative_to(
                matched_key
            )
        )

        images_by_sample_key[
            matched_key
        ].append(
            relative_member
        )

zip_runtime = (
    time.time()
    - start
)

print("\nZIP image index runtime:")
print(
    f"{zip_runtime:.2f} sec"
)


# ------------------------------------------------------------
# 6. Resolve exact UAV + satellite folders
# ------------------------------------------------------------

sample_resolution_records = []

for _, row in samples_df.iterrows():

    sample_key = str(
        row[
            "sample_key"
        ]
    )

    relative_images = sorted(
        images_by_sample_key.get(
            sample_key,
            []
        )
    )

    uav_images = []

    satellite_images = []

    other_images = []

    for relative_path in relative_images:

        parts = (
            Path(
                relative_path
            ).parts
        )

        first_component = (
            parts[0].lower()
            if parts
            else ""
        )

        if first_component == "uav":

            uav_images.append(
                relative_path
            )

        elif (
            first_component
            == "satellite"
        ):

            satellite_images.append(
                relative_path
            )

        else:

            other_images.append(
                relative_path
            )

    resolved_uav_member = (
        sample_key
        + "/"
        + uav_images[0]
        if len(
            uav_images
        ) == 1
        else None
    )

    sample_resolution_records.append({
        "sample_key": (
            sample_key
        ),

        "dataset_partition": (
            row[
                "dataset_partition"
            ]
        ),

        "sample_id": (
            row[
                "sample_id"
            ]
        ),

        "uav_image_count": int(
            len(
                uav_images
            )
        ),

        "satellite_image_count": int(
            len(
                satellite_images
            )
        ),

        "other_image_count": int(
            len(
                other_images
            )
        ),

        "uav_relative_member": (
            uav_images[0]
            if len(
                uav_images
            ) == 1
            else None
        ),

        "uav_member": (
            resolved_uav_member
        ),
    })

resolution_df = pd.DataFrame(
    sample_resolution_records
)


# ------------------------------------------------------------
# 7. Resolution statistics
# ------------------------------------------------------------

exact_one_uav = int(
    (
        resolution_df[
            "uav_image_count"
        ] == 1
    ).sum()
)

zero_uav = int(
    (
        resolution_df[
            "uav_image_count"
        ] == 0
    ).sum()
)

multiple_uav = int(
    (
        resolution_df[
            "uav_image_count"
        ] > 1
    ).sum()
)

exact_12_sat = int(
    (
        resolution_df[
            "satellite_image_count"
        ] == 12
    ).sum()
)

non_12_sat = int(
    (
        resolution_df[
            "satellite_image_count"
        ] != 12
    ).sum()
)

print("\n" + "=" * 94)
print("PARTITION-SAFE SAMPLE IMAGE RESOLUTION")
print("=" * 94)

print("\nExactly one UAV image:")
print(
    f"{exact_one_uav:,} / "
    f"{len(resolution_df):,}"
)

print("\nZero UAV images:")
print(
    f"{zero_uav:,}"
)

print("\nMultiple UAV images:")
print(
    f"{multiple_uav:,}"
)

print("\nExactly 12 satellite images:")
print(
    f"{exact_12_sat:,} / "
    f"{len(resolution_df):,}"
)

print("\nSamples not having 12 satellite images:")
print(
    f"{non_12_sat:,}"
)


# ------------------------------------------------------------
# 8. Resolve each georeference satellite member
# ------------------------------------------------------------

# Build exact map:
# sample_key -> satellite basename -> member
# ------------------------------------------------------------

satellite_map = {}

for sample_key, relative_images in (
    images_by_sample_key.items()
):

    local = defaultdict(
        list
    )

    for relative_path in (
        relative_images
    ):

        parts = (
            Path(
                relative_path
            ).parts
        )

        if (
            not parts
            or
            parts[0].lower()
            != "satellite"
        ):
            continue

        basename = (
            Path(
                relative_path
            ).name
        )

        local[
            basename
        ].append(
            relative_path
        )

    satellite_map[
        sample_key
    ] = local


resolved_satellite_members = []

satellite_candidate_counts = []

for _, row in georef_df.iterrows():

    sample_key = str(
        row[
            "sample_key"
        ]
    )

    satellite_image = str(
        row[
            "satellite_image"
        ]
    )

    matches = (
        satellite_map
        .get(
            sample_key,
            {}
        )
        .get(
            satellite_image,
            []
        )
    )

    satellite_candidate_counts.append(
        len(
            matches
        )
    )

    if len(
        matches
    ) == 1:

        resolved_satellite_members.append(
            sample_key
            + "/"
            + matches[0]
        )

    else:

        resolved_satellite_members.append(
            None
        )

georef_df[
    "satellite_candidate_count"
] = (
    satellite_candidate_counts
)

georef_df[
    "satellite_member_resolved"
] = (
    resolved_satellite_members
)

unique_satellite_resolution = int(
    (
        georef_df[
            "satellite_candidate_count"
        ] == 1
    ).sum()
)

missing_satellite_resolution = int(
    (
        georef_df[
            "satellite_candidate_count"
        ] == 0
    ).sum()
)

ambiguous_satellite_resolution = int(
    (
        georef_df[
            "satellite_candidate_count"
        ] > 1
    ).sum()
)


# ------------------------------------------------------------
# 9. Attach UAV member to every georef row
# ------------------------------------------------------------

uav_member_lookup = (
    resolution_df
    .set_index(
        "sample_key"
    )[
        "uav_member"
    ]
    .to_dict()
)

georef_df[
    "uav_member"
] = georef_df[
    "sample_key"
].map(
    uav_member_lookup
)

missing_uav_in_georef = int(
    georef_df[
        "uav_member"
    ].isna().sum()
)


# ------------------------------------------------------------
# 10. Pixel-label tolerance diagnosis
# ------------------------------------------------------------

pixel_errors = pd.to_numeric(
    georef_df[
        "pixel_roundtrip_error_px"
    ],
    errors="coerce",
)

label_records = int(
    pixel_errors.notna().sum()
)

within_1px = int(
    (
        pixel_errors <= 1
    ).sum()
)

exactly_2px = int(
    (
        pixel_errors == 2
    ).sum()
)

greater_2px = int(
    (
        pixel_errors > 2
    ).sum()
)

within_2px = int(
    (
        pixel_errors <= 2
    ).sum()
)

within_2px_fraction = (
    within_2px
    / label_records
    if label_records
    else 0.0
)

# Important:
# exact normalized coordinate consistency is the stronger
# schema check; labels are discretized raster coordinates.

center_errors = pd.to_numeric(
    georef_df[
        "center_distribution_error"
    ],
    errors="coerce",
)

max_center_error = float(
    center_errors.max()
)

normalized_schema_pass = bool(
    max_center_error
    < 1e-8
)

label_rasterization_pass_2px = bool(
    greater_2px == 0
)


# ------------------------------------------------------------
# 11. Build executable QUERY manifest
# ------------------------------------------------------------

query_df = (
    samples_df[
        [
            "sample_key",
            "dataset_partition",
            "sample_id",
            "site_token",
            "nominal_level_token",
            "sequence_token",
            "uav_E",
            "uav_N",
            "gps_member",
        ]
    ]
    .copy()
)

query_df[
    "uav_member"
] = query_df[
    "sample_key"
].map(
    uav_member_lookup
)

query_df[
    "coordinate_schema"
] = (
    "GEOGRAPHIC_DEGREES_EAST_NORTH"
)

query_df[
    "crs_status"
] = (
    "CRS_UNVERIFIED"
)

query_df[
    "nominal_level_status"
] = (
    "NOT_VERIFIED_AS_AGL_OR_MSL"
)


# ------------------------------------------------------------
# 12. Build executable GALLERY manifest
# ------------------------------------------------------------

gallery_columns = [
    "sample_key",
    "dataset_partition",
    "sample_id",
    "site_token",
    "nominal_level_token",
    "sequence_token",

    "satellite_image",
    "satellite_member_resolved",

    "tl_E",
    "tl_N",
    "br_E",
    "br_N",

    "satellite_center_E",
    "satellite_center_N",

    "map_size",

    "recorded_label_row",
    "recorded_label_col",

    "uav_E",
    "uav_N",

    "pixel_roundtrip_error_px",
    "center_distribution_error",
]

gallery_df = (
    georef_df[
        gallery_columns
    ]
    .copy()
)

gallery_df = gallery_df.rename(
    columns={
        "satellite_member_resolved":
        "satellite_member"
    }
)

gallery_df[
    "coordinate_schema"
] = (
    "GEOGRAPHIC_DEGREES_EAST_NORTH"
)

gallery_df[
    "crs_status"
] = (
    "CRS_UNVERIFIED"
)


# ------------------------------------------------------------
# 13. Final resolution pass
# ------------------------------------------------------------

all_query_members_resolved = bool(
    query_df[
        "uav_member"
    ].notna().all()
)

all_gallery_members_resolved = bool(
    gallery_df[
        "satellite_member"
    ].notna().all()
)

sample_keys_unique = bool(
    query_df[
        "sample_key"
    ].is_unique
)

full_image_layout_resolved = bool(
    sample_keys_unique
    and
    all_query_members_resolved
    and
    all_gallery_members_resolved
    and
    exact_one_uav
    == len(
        query_df
    )
    and
    exact_12_sat
    == len(
        query_df
    )
)

metadata_schema_validated = bool(
    normalized_schema_pass
    and
    label_rasterization_pass_2px
)


# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

RESOLUTION_CSV = (
    AUDIT_ROOT
    / "vps_partition_safe_image_resolution.csv"
)

EXEC_QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_executable_query_manifest.csv"
)

EXEC_GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_executable_gallery_manifest.csv"
)

RESOLVED_GEOREF_CSV = (
    MANIFEST_ROOT
    / "vps_archive_resolved_georeference_manifest.csv"
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "vps_partition_safe_resolution_audit.json"
)

resolution_df.to_csv(
    RESOLUTION_CSV,
    index=False,
)

query_df.to_csv(
    EXEC_QUERY_CSV,
    index=False,
)

gallery_df.to_csv(
    EXEC_GALLERY_CSV,
    index=False,
)

georef_df.to_csv(
    RESOLVED_GEOREF_CSV,
    index=False,
)

audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "root_cause_previous_ambiguity": (
        "PLAIN_SAMPLE_ID_COLLISION_ACROSS_"
        "DATASET_PARTITIONS"
    ),

    "unique_full_sample_keys": (
        unique_sample_keys
    ),

    "repeated_plain_sample_ids": (
        duplicate_sample_id_count
    ),

    "rows_affected_by_repeated_sample_ids": (
        rows_with_duplicate_sample_id
    ),

    "image_resolution": {
        "exactly_one_uav_image_samples": (
            exact_one_uav
        ),

        "exactly_12_satellite_image_samples": (
            exact_12_sat
        ),

        "unique_satellite_record_resolutions": (
            unique_satellite_resolution
        ),

        "missing_satellite_resolutions": (
            missing_satellite_resolution
        ),

        "ambiguous_satellite_resolutions": (
            ambiguous_satellite_resolution
        ),

        "full_image_layout_resolved": (
            full_image_layout_resolved
        ),
    },

    "pixel_label_validation": {
        "records": (
            label_records
        ),

        "within_1px": (
            within_1px
        ),

        "exactly_2px": (
            exactly_2px
        ),

        "greater_than_2px": (
            greater_2px
        ),

        "within_2px_fraction": (
            within_2px_fraction
        ),

        "interpretation": (
            "RASTER_LABEL_DISCRETIZATION_AUDIT"
        ),

        "tolerance_used_for_internal_consistency_px": (
            2
        ),
    },

    "normalized_coordinate_validation": {
        "maximum_error": (
            max_center_error
        ),

        "pass": (
            normalized_schema_pass
        ),
    },

    "metadata_schema_validated": (
        metadata_schema_validated
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "metric_geodetic_claim_status": (
        "BLOCKED_UNTIL_CRS_PROVENANCE_VERIFIED"
    ),

    "retrieval_is_verified_uav_pose": (
        False
    ),

    "outputs": {
        "query_manifest": str(
            EXEC_QUERY_CSV
        ),

        "gallery_manifest": str(
            EXEC_GALLERY_CSV
        ),

        "resolved_georeference_manifest": str(
            RESOLVED_GEOREF_CSV
        ),
    },
}

AUDIT_JSON.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 15. Final output
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("✅ NOTEBOOK 04 CELL 4D COMPLETE")
print("=" * 94)

print("\nRepeated plain sample IDs:")
print(
    f"{duplicate_sample_id_count:,}"
)

print("\nRows affected by duplicate sample IDs:")
print(
    f"{rows_with_duplicate_sample_id:,}"
)

print("\nUnique full sample keys:")
print(
    f"{unique_sample_keys:,} / "
    f"{len(samples_df):,}"
)

print("\nSamples with exactly ONE UAV image:")
print(
    f"{exact_one_uav:,} / "
    f"{len(samples_df):,}"
)

print("\nSamples with exactly 12 Satellite images:")
print(
    f"{exact_12_sat:,} / "
    f"{len(samples_df):,}"
)

print("\nUnique satellite record resolutions:")
print(
    f"{unique_satellite_resolution:,} / "
    f"{len(georef_df):,}"
)

print("\nMissing satellite resolutions:")
print(
    f"{missing_satellite_resolution:,}"
)

print("\nAmbiguous satellite resolutions:")
print(
    f"{ambiguous_satellite_resolution:,}"
)

print("\nPixel records <=1 px:")
print(
    f"{within_1px:,} / "
    f"{label_records:,}"
)

print("\nPixel records exactly 2 px:")
print(
    f"{exactly_2px:,}"
)

print("\nPixel records >2 px:")
print(
    f"{greater_2px:,}"
)

print("\nMaximum normalized-coordinate error:")
print(
    f"{max_center_error:.12e}"
)

print("\nMetadata schema validated:")
print(
    metadata_schema_validated
)

print("\nFull image layout resolved:")
print(
    full_image_layout_resolved
)

print("\nExecutable query rows:")
print(
    f"{len(query_df):,}"
)

print("\nExecutable gallery rows:")
print(
    f"{len(gallery_df):,}"
)

print("\nCRS:")
print(
    "UNVERIFIED — DO NOT CLAIM WGS84"
)

print("\nQuery manifest:")
print(
    EXEC_QUERY_CSV
)

print("\nGallery manifest:")
print(
    EXEC_GALLERY_CSV
)

print("\nAudit:")
print(
    AUDIT_JSON
)

print("\nNext action:")

if (
    full_image_layout_resolved
    and metadata_schema_validated
):

    print(
        "Executable georeferenced manifests are ready. "
        "Next: define the leakage-safe VPS evaluation "
        "protocol and benchmark partition."
    )

else:

    print(
        "Review unresolved image members or metadata "
        "validation before benchmark execution."
    )

NOTEBOOK 04 — PARTITION-SAFE IMAGE MEMBER RESOLUTION

Georeference rows:
32,160

GPS sample rows:
2,680

SAMPLE-ID COLLISION DIAGNOSIS

Unique full sample keys:
2,680

Duplicate FULL sample-key rows:
0

Repeated plain sample_id values:
349

Rows affected by repeated sample_id:
698

Expected explanation for previous ambiguity:
Plain sample_id merged different dataset partitions.

ZIP image index runtime:
6.58 sec

PARTITION-SAFE SAMPLE IMAGE RESOLUTION

Exactly one UAV image:
2,680 / 2,680

Zero UAV images:
0

Multiple UAV images:
0

Exactly 12 satellite images:
2,680 / 2,680

Samples not having 12 satellite images:
0

✅ NOTEBOOK 04 CELL 4D COMPLETE

Repeated plain sample IDs:
349

Rows affected by duplicate sample IDs:
698

Unique full sample keys:
2,680 / 2,680

Samples with exactly ONE UAV image:
2,680 / 2,680

Samples with exactly 12 Satellite images:
2,680 / 2,680

Unique satellite record resolutions:
32,160 / 32,160

Missing satellite resolutions:
0

Ambiguous satellite resolution

In [30]:
# ============================================================
# NOTEBOOK 04 — CELL 5
# Leakage audit + leakage-safe VPS benchmark protocol
#
# Goals:
# 1. Verify 349 cross-partition repeated sample IDs
# 2. Compare UAV coordinates + ZIP CRCs
# 3. Detect exact val/test duplication
# 4. If verified, exclude duplicated VAL samples
# 5. Build scale-controlled evaluation manifests
#
# NO image decoding
# NO archive extraction
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import zipfile
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

PROTOCOL_ROOT = (
    PHASE4_ROOT
    / "protocol"
)

for directory in [
    AUDIT_ROOT,
    MANIFEST_ROOT,
    PROTOCOL_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_executable_query_manifest.csv"
)

GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_executable_gallery_manifest.csv"
)

for path in [
    VPS_ARCHIVE,
    QUERY_CSV,
    GALLERY_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Load resolved manifests
# ------------------------------------------------------------

print("=" * 96)
print("NOTEBOOK 04 — VPS LEAKAGE AUDIT + BENCHMARK PROTOCOL")
print("=" * 96)

query_df = pd.read_csv(
    QUERY_CSV
)

gallery_df = pd.read_csv(
    GALLERY_CSV
)

print("\nExecutable queries:")
print(
    f"{len(query_df):,}"
)

print("\nExecutable gallery records:")
print(
    f"{len(gallery_df):,}"
)

print("\nPartitions:")
print(
    query_df[
        "dataset_partition"
    ].value_counts().to_string()
)


# ------------------------------------------------------------
# 3. Identify cross-partition repeated plain sample IDs
# ------------------------------------------------------------

sample_partition_counts = (
    query_df
    .groupby(
        "sample_id"
    )[
        "dataset_partition"
    ]
    .nunique()
)

cross_partition_ids = (
    sample_partition_counts[
        sample_partition_counts > 1
    ]
    .index
    .astype(str)
    .tolist()
)

print("\n" + "=" * 96)
print("CROSS-PARTITION SAMPLE-ID AUDIT")
print("=" * 96)

print("\nRepeated IDs across partitions:")
print(
    f"{len(cross_partition_ids):,}"
)


# ------------------------------------------------------------
# 4. ZIP CRC lookup
# ------------------------------------------------------------
# CRC + file size is used only as an archive-level
# exact-content identity check.
# ------------------------------------------------------------

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    zip_info = {
        info.filename: {
            "crc": int(
                info.CRC
            ),
            "file_size": int(
                info.file_size
            ),
            "compress_size": int(
                info.compress_size
            ),
        }
        for info in zf.infolist()
        if not info.is_dir()
    }


# ------------------------------------------------------------
# 5. Attach UAV CRC / size
# ------------------------------------------------------------

def zip_identity(
    member
):

    if pd.isna(
        member
    ):
        return (
            None,
            None,
        )

    info = zip_info.get(
        str(
            member
        )
    )

    if info is None:
        return (
            None,
            None,
        )

    return (
        info[
            "crc"
        ],
        info[
            "file_size"
        ],
    )


uav_crc = []

uav_size = []

for member in query_df[
    "uav_member"
]:

    crc, size = zip_identity(
        member
    )

    uav_crc.append(
        crc
    )

    uav_size.append(
        size
    )

query_df[
    "uav_crc32"
] = uav_crc

query_df[
    "uav_file_size"
] = uav_size


# ------------------------------------------------------------
# 6. Attach satellite CRC / size
# ------------------------------------------------------------

sat_crc = []

sat_size = []

for member in gallery_df[
    "satellite_member"
]:

    crc, size = zip_identity(
        member
    )

    sat_crc.append(
        crc
    )

    sat_size.append(
        size
    )

gallery_df[
    "satellite_crc32"
] = sat_crc

gallery_df[
    "satellite_file_size"
] = sat_size


# ------------------------------------------------------------
# 7. Build per-sample satellite identity signature
# ------------------------------------------------------------

gallery_sorted = (
    gallery_df
    .sort_values(
        [
            "sample_key",
            "map_size",
            "satellite_image",
        ]
    )
)

satellite_signatures = {}

for sample_key, group in (
    gallery_sorted.groupby(
        "sample_key"
    )
):

    signature = []

    for _, row in group.iterrows():

        signature.append(
            (
                int(
                    row[
                        "map_size"
                    ]
                ),
                str(
                    row[
                        "satellite_image"
                    ]
                ),
                int(
                    row[
                        "satellite_crc32"
                    ]
                ),
                int(
                    row[
                        "satellite_file_size"
                    ]
                ),
            )
        )

    satellite_signatures[
        str(
            sample_key
        )
    ] = tuple(
        signature
    )


# ------------------------------------------------------------
# 8. Compare duplicated IDs across partitions
# ------------------------------------------------------------

duplicate_audit_records = []

for sample_id in (
    cross_partition_ids
):

    rows = (
        query_df[
            query_df[
                "sample_id"
            ].astype(str)
            == sample_id
        ]
        .copy()
        .sort_values(
            "dataset_partition"
        )
    )

    if len(
        rows
    ) != 2:

        duplicate_audit_records.append({
            "sample_id": sample_id,
            "row_count": int(
                len(
                    rows
                )
            ),
            "exact_duplicate": False,
            "reason": (
                "EXPECTED_EXACTLY_TWO_ROWS"
            ),
        })

        continue

    row_a = rows.iloc[
        0
    ]

    row_b = rows.iloc[
        1
    ]

    coordinate_match = bool(
        np.isclose(
            float(
                row_a[
                    "uav_E"
                ]
            ),
            float(
                row_b[
                    "uav_E"
                ]
            ),
            atol=1e-12,
            rtol=0.0,
        )
        and
        np.isclose(
            float(
                row_a[
                    "uav_N"
                ]
            ),
            float(
                row_b[
                    "uav_N"
                ]
            ),
            atol=1e-12,
            rtol=0.0,
        )
    )

    uav_crc_match = bool(
        int(
            row_a[
                "uav_crc32"
            ]
        )
        ==
        int(
            row_b[
                "uav_crc32"
            ]
        )
    )

    uav_size_match = bool(
        int(
            row_a[
                "uav_file_size"
            ]
        )
        ==
        int(
            row_b[
                "uav_file_size"
            ]
        )
    )

    sig_a = (
        satellite_signatures.get(
            str(
                row_a[
                    "sample_key"
                ]
            )
        )
    )

    sig_b = (
        satellite_signatures.get(
            str(
                row_b[
                    "sample_key"
                ]
            )
        )
    )

    satellite_signature_match = bool(
        sig_a is not None
        and
        sig_b is not None
        and
        sig_a == sig_b
    )

    exact_duplicate = bool(
        coordinate_match
        and
        uav_crc_match
        and
        uav_size_match
        and
        satellite_signature_match
    )

    duplicate_audit_records.append({
        "sample_id": (
            sample_id
        ),

        "partition_a": str(
            row_a[
                "dataset_partition"
            ]
        ),

        "partition_b": str(
            row_b[
                "dataset_partition"
            ]
        ),

        "sample_key_a": str(
            row_a[
                "sample_key"
            ]
        ),

        "sample_key_b": str(
            row_b[
                "sample_key"
            ]
        ),

        "coordinate_match": (
            coordinate_match
        ),

        "uav_crc_match": (
            uav_crc_match
        ),

        "uav_size_match": (
            uav_size_match
        ),

        "satellite_signature_match": (
            satellite_signature_match
        ),

        "exact_duplicate": (
            exact_duplicate
        ),
    })

duplicate_audit_df = pd.DataFrame(
    duplicate_audit_records
)


# ------------------------------------------------------------
# 9. Leakage audit statistics
# ------------------------------------------------------------

if not duplicate_audit_df.empty:

    exact_duplicate_count = int(
        duplicate_audit_df[
            "exact_duplicate"
        ].fillna(
            False
        ).sum()
    )

else:

    exact_duplicate_count = 0

non_exact_duplicate_count = (
    len(
        duplicate_audit_df
    )
    - exact_duplicate_count
)

all_cross_partition_exact = bool(
    len(
        duplicate_audit_df
    ) > 0
    and
    exact_duplicate_count
    == len(
        duplicate_audit_df
    )
)

print("\nCross-partition repeated IDs:")
print(
    f"{len(duplicate_audit_df):,}"
)

print("\nVerified exact duplicates:")
print(
    f"{exact_duplicate_count:,}"
)

print("\nNon-exact repeated IDs:")
print(
    f"{non_exact_duplicate_count:,}"
)

print("\nAll repeated IDs are exact duplicates:")
print(
    all_cross_partition_exact
)


# ------------------------------------------------------------
# 10. Determine evaluation partition
# ------------------------------------------------------------

partition_counts = (
    query_df[
        "dataset_partition"
    ]
    .value_counts()
    .to_dict()
)

MERGE_TEST_PARTITION = (
    "merge_test_700-1800_cr0.95_stride100"
)

VAL_PARTITION = (
    "val"
)

merge_test_exists = (
    MERGE_TEST_PARTITION
    in partition_counts
)

val_exists = (
    VAL_PARTITION
    in partition_counts
)

protocol_ready = bool(
    merge_test_exists
    and
    all_cross_partition_exact
)


# ------------------------------------------------------------
# 11. Build leakage-safe core evaluation data
# ------------------------------------------------------------
# If VAL is an exact duplicate subset, it must not be
# treated as an independent calibration/evaluation set.
#
# Core protocol:
#
# Query:
#   merge_test UAV images only
#
# Gallery:
#   merge_test satellite images only
#
# Scale-controlled evaluation:
#   one gallery image per query/sample for each map_size
#
# This produces 12 independent retrieval benchmarks:
# 700, 800, ... 1800.
# ------------------------------------------------------------

CORE_QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_queries.csv"
)

CORE_GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_gallery_all_scales.csv"
)

PAIR_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_positive_pairs.csv"
)

SCALE_INDEX_CSV = (
    PROTOCOL_ROOT
    / "vps_scale_benchmark_index.csv"
)

scale_index_records = []

if protocol_ready:

    core_query_df = (
        query_df[
            query_df[
                "dataset_partition"
            ]
            == MERGE_TEST_PARTITION
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    core_gallery_df = (
        gallery_df[
            gallery_df[
                "dataset_partition"
            ]
            == MERGE_TEST_PARTITION
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    # Unique query identifier.
    core_query_df[
        "query_id"
    ] = [
        f"q_{index:06d}"
        for index in range(
            len(
                core_query_df
            )
        )
    ]

    query_id_lookup = (
        core_query_df
        .set_index(
            "sample_key"
        )[
            "query_id"
        ]
        .to_dict()
    )

    core_gallery_df[
        "query_id"
    ] = core_gallery_df[
        "sample_key"
    ].map(
        query_id_lookup
    )

    # Positive pairing is exact sample_key identity.
    core_gallery_df[
        "is_positive_for_query"
    ] = True

    # --------------------------------------------------------
    # Verify one gallery item per sample per scale
    # --------------------------------------------------------

    scale_counts = (
        core_gallery_df
        .groupby(
            "map_size"
        )
        .agg(
            gallery_items=(
                "sample_key",
                "count",
            ),

            unique_samples=(
                "sample_key",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            "map_size"
        )
    )

    expected_queries = int(
        len(
            core_query_df
        )
    )

    for _, row in (
        scale_counts.iterrows()
    ):

        map_size = int(
            row[
                "map_size"
            ]
        )

        gallery_items = int(
            row[
                "gallery_items"
            ]
        )

        unique_samples = int(
            row[
                "unique_samples"
            ]
        )

        scale_valid = bool(
            gallery_items
            == expected_queries
            and
            unique_samples
            == expected_queries
        )

        scale_manifest_path = (
            MANIFEST_ROOT
            / (
                f"vps_core_eval_gallery_"
                f"map{map_size}.csv"
            )
        )

        scale_gallery_df = (
            core_gallery_df[
                core_gallery_df[
                    "map_size"
                ]
                == map_size
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        scale_gallery_df[
            "gallery_id"
        ] = [
            f"g_{map_size}_{index:06d}"
            for index in range(
                len(
                    scale_gallery_df
                )
            )
        ]

        scale_gallery_df.to_csv(
            scale_manifest_path,
            index=False,
        )

        scale_index_records.append({
            "map_size": (
                map_size
            ),

            "query_count": (
                expected_queries
            ),

            "gallery_count": (
                gallery_items
            ),

            "unique_sample_count": (
                unique_samples
            ),

            "one_positive_per_query": (
                scale_valid
            ),

            "gallery_manifest": str(
                scale_manifest_path
            ),
        })

    scale_index_df = pd.DataFrame(
        scale_index_records
    )

    all_scales_valid = bool(
        scale_index_df[
            "one_positive_per_query"
        ].all()
    )

    # --------------------------------------------------------
    # Positive-pair manifest
    # --------------------------------------------------------

    pair_df = (
        core_gallery_df[
            [
                "query_id",
                "sample_key",
                "map_size",
                "uav_E",
                "uav_N",
                "satellite_member",
                "recorded_label_row",
                "recorded_label_col",
            ]
        ]
        .copy()
    )

    pair_df = pair_df.rename(
        columns={
            "satellite_member":
            "positive_satellite_member"
        }
    )

    core_query_df.to_csv(
        CORE_QUERY_CSV,
        index=False,
    )

    core_gallery_df.to_csv(
        CORE_GALLERY_CSV,
        index=False,
    )

    pair_df.to_csv(
        PAIR_CSV,
        index=False,
    )

    scale_index_df.to_csv(
        SCALE_INDEX_CSV,
        index=False,
    )

else:

    core_query_df = pd.DataFrame()
    core_gallery_df = pd.DataFrame()
    pair_df = pd.DataFrame()
    scale_index_df = pd.DataFrame()

    all_scales_valid = False


# ------------------------------------------------------------
# 12. Save duplicate audit
# ------------------------------------------------------------

DUPLICATE_AUDIT_CSV = (
    AUDIT_ROOT
    / "vps_cross_partition_duplicate_audit.csv"
)

duplicate_audit_df.to_csv(
    DUPLICATE_AUDIT_CSV,
    index=False,
)


# ------------------------------------------------------------
# 13. Protocol definition
# ------------------------------------------------------------

PROTOCOL_JSON = (
    PROTOCOL_ROOT
    / "vps_leakage_safe_protocol.json"
)

protocol = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_name": (
        "PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "source_archive": str(
        VPS_ARCHIVE
    ),

    "cross_partition_audit": {
        "repeated_plain_sample_ids": int(
            len(
                duplicate_audit_df
            )
        ),

        "verified_exact_duplicates": (
            exact_duplicate_count
        ),

        "non_exact_repeated_ids": (
            non_exact_duplicate_count
        ),

        "all_repeated_ids_exact": (
            all_cross_partition_exact
        ),
    },

    "partition_policy": {
        "core_evaluation_partition": (
            MERGE_TEST_PARTITION
        ),

        "excluded_partition": (
            VAL_PARTITION
            if all_cross_partition_exact
            else None
        ),

        "excluded_partition_reason": (
            "EXACT_DUPLICATE_SUBSET_OF_CORE_"
            "EVALUATION_PARTITION"
            if all_cross_partition_exact
            else (
                "NOT_YET_VERIFIED"
            )
        ),

        "val_allowed_for_threshold_calibration": (
            False
            if all_cross_partition_exact
            else None
        ),
    },

    "query_definition": (
        "ONE_UAV_IMAGE_PER_UNIQUE_FULL_SAMPLE_KEY"
    ),

    "gallery_definition": (
        "ONE_SATELLITE_IMAGE_PER_SAMPLE_AT_EACH_"
        "MAP_SIZE_FOR_SCALE_CONTROLLED_EVALUATION"
    ),

    "positive_definition": (
        "QUERY_AND_GALLERY_SHARE_EXACT_FULL_SAMPLE_KEY"
    ),

    "map_sizes": (
        scale_index_df[
            "map_size"
        ].astype(
            int
        ).tolist()
        if not scale_index_df.empty
        else []
    ),

    "query_count": (
        int(
            len(
                core_query_df
            )
        )
        if protocol_ready
        else None
    ),

    "gallery_count_per_scale": (
        int(
            len(
                core_query_df
            )
        )
        if protocol_ready
        else None
    ),

    "all_scale_manifests_valid": (
        all_scales_valid
    ),

    "coordinate_schema": (
        "GEOGRAPHIC_DEGREES_EAST_NORTH"
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "meter_level_geodetic_claim_allowed": (
        False
    ),

    "nominal_level_policy": (
        "DATASET_NAME_TOKEN_ONLY_"
        "NOT_VERIFIED_AS_AGL_OR_MSL"
    ),

    "retrieval_output_policy": (
        "RETRIEVAL_RESULT_IS_NOT_VERIFIED_UAV_POSE"
    ),

    "protocol_ready": (
        bool(
            protocol_ready
            and
            all_scales_valid
        )
    ),
}

PROTOCOL_JSON.write_text(
    json.dumps(
        protocol,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 14. Final output
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("✅ NOTEBOOK 04 CELL 5 COMPLETE")
print("=" * 96)

print("\nRepeated IDs across partitions:")
print(
    f"{len(duplicate_audit_df):,}"
)

print("\nVerified exact duplicates:")
print(
    f"{exact_duplicate_count:,}"
)

print("\nNon-exact repeated IDs:")
print(
    f"{non_exact_duplicate_count:,}"
)

print("\nAll repeated IDs exact:")
print(
    all_cross_partition_exact
)

if protocol_ready:

    print("\nCore evaluation partition:")
    print(
        MERGE_TEST_PARTITION
    )

    print("\nVAL excluded as independent set:")
    print(
        all_cross_partition_exact
    )

    print("\nCore query count:")
    print(
        f"{len(core_query_df):,}"
    )

    print("\nCore gallery records across all scales:")
    print(
        f"{len(core_gallery_df):,}"
    )

    print("\nScale-controlled benchmark:")
    print(
        scale_index_df[
            [
                "map_size",
                "query_count",
                "gallery_count",
                "one_positive_per_query",
            ]
        ].to_string(
            index=False
        )
    )

    print("\nAll scale manifests valid:")
    print(
        all_scales_valid
    )

else:

    print(
        "\n⚠ Protocol was NOT created because "
        "cross-partition leakage is not fully resolved."
    )

print("\nCRS:")
print(
    "UNVERIFIED — WGS84 NOT CLAIMED"
)

print("\nMeter-level geodetic claim:")
print(
    "BLOCKED UNTIL CRS PROVENANCE IS VERIFIED"
)

print("\nProtocol JSON:")
print(
    PROTOCOL_JSON
)

print("\nDuplicate audit:")
print(
    DUPLICATE_AUDIT_CSV
)

print("\n" + "=" * 96)

if (
    protocol_ready
    and
    all_scales_valid
):

    print(
        "✅ LEAKAGE-SAFE VPS BENCHMARK PROTOCOL READY"
    )

    print(
        "\nNext: extract/cache only the required "
        "core evaluation images from ZIP and run "
        "descriptor inference."
    )

else:

    print(
        "⚠ REVIEW REQUIRED BEFORE MODEL EVALUATION"
    )

print("=" * 96)

NOTEBOOK 04 — VPS LEAKAGE AUDIT + BENCHMARK PROTOCOL

Executable queries:
2,680

Executable gallery records:
32,160

Partitions:
dataset_partition
merge_test_700-1800_cr0.95_stride100    2331
val                                      349

CROSS-PARTITION SAMPLE-ID AUDIT

Repeated IDs across partitions:
349

Cross-partition repeated IDs:
349

Verified exact duplicates:
349

Non-exact repeated IDs:
0

All repeated IDs are exact duplicates:
True

✅ NOTEBOOK 04 CELL 5 COMPLETE

Repeated IDs across partitions:
349

Verified exact duplicates:
349

Non-exact repeated IDs:
0

All repeated IDs exact:
True

Core evaluation partition:
merge_test_700-1800_cr0.95_stride100

VAL excluded as independent set:
True

Core query count:
2,331

Core gallery records across all scales:
27,972

Scale-controlled benchmark:
 map_size  query_count  gallery_count  one_positive_per_query
      700         2331           2331                    True
      800         2331           2331                    True
     

In [31]:
# ============================================================
# NOTEBOOK 04 — CELL 6
# Core VPS evaluation cache planning
#
# Purpose:
# - Resolve exact required ZIP members
# - Calculate extraction size BEFORE extracting
# - Check Colab local disk capacity
# - Produce scale-by-scale cache plan
#
# NO IMAGE EXTRACTION YET
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import zipfile
import shutil
import json

import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

CACHE_PLAN_ROOT = (
    PHASE4_ROOT
    / "cache_plan"
)

for directory in [
    AUDIT_ROOT,
    CACHE_PLAN_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_queries.csv"
)

GALLERY_ALL_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_gallery_all_scales.csv"
)


for path in [
    VPS_ARCHIVE,
    QUERY_CSV,
    GALLERY_ALL_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Load core manifests
# ------------------------------------------------------------

print("=" * 94)
print("NOTEBOOK 04 — VPS CORE CACHE PLANNING")
print("=" * 94)

query_df = pd.read_csv(
    QUERY_CSV
)

gallery_df = pd.read_csv(
    GALLERY_ALL_CSV
)

print("\nCore queries:")
print(
    f"{len(query_df):,}"
)

print("\nGallery records:")
print(
    f"{len(gallery_df):,}"
)


# ------------------------------------------------------------
# 3. Basic manifest validation
# ------------------------------------------------------------

if not query_df[
    "uav_member"
].notna().all():

    raise RuntimeError(
        "Some UAV ZIP members are unresolved."
    )


if not gallery_df[
    "satellite_member"
].notna().all():

    raise RuntimeError(
        "Some satellite ZIP members are unresolved."
    )


if not query_df[
    "sample_key"
].is_unique:

    raise RuntimeError(
        "Query sample_key is not unique."
    )


map_sizes = sorted(
    gallery_df[
        "map_size"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print("\nMap sizes:")
print(
    map_sizes
)


# ------------------------------------------------------------
# 4. Build ZIP metadata lookup
# ------------------------------------------------------------

print("\nReading ZIP central directory...")

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    zip_lookup = {
        info.filename: {
            "file_size": int(
                info.file_size
            ),
            "compress_size": int(
                info.compress_size
            ),
            "crc32": int(
                info.CRC
            ),
        }
        for info in zf.infolist()
        if not info.is_dir()
    }


# ------------------------------------------------------------
# 5. Required query members
# ------------------------------------------------------------

query_members = (
    query_df[
        "uav_member"
    ]
    .astype(str)
    .tolist()
)

query_members_unique = sorted(
    set(
        query_members
    )
)

missing_query_members = [
    member
    for member in query_members_unique
    if member not in zip_lookup
]


# ------------------------------------------------------------
# 6. Required gallery members
# ------------------------------------------------------------

gallery_members = (
    gallery_df[
        "satellite_member"
    ]
    .astype(str)
    .tolist()
)

gallery_members_unique = sorted(
    set(
        gallery_members
    )
)

missing_gallery_members = [
    member
    for member in gallery_members_unique
    if member not in zip_lookup
]


if missing_query_members:

    raise RuntimeError(
        f"{len(missing_query_members)} "
        "query members missing from ZIP."
    )


if missing_gallery_members:

    raise RuntimeError(
        f"{len(missing_gallery_members)} "
        "gallery members missing from ZIP."
    )


# ------------------------------------------------------------
# 7. Size helper
# ------------------------------------------------------------

GB = 1024 ** 3
MB = 1024 ** 2


def member_size_summary(
    members,
):

    raw_bytes = sum(
        zip_lookup[
            member
        ][
            "file_size"
        ]
        for member in members
    )

    compressed_bytes = sum(
        zip_lookup[
            member
        ][
            "compress_size"
        ]
        for member in members
    )

    return {
        "count": int(
            len(
                members
            )
        ),

        "raw_bytes": int(
            raw_bytes
        ),

        "compressed_bytes": int(
            compressed_bytes
        ),

        "raw_gb": float(
            raw_bytes / GB
        ),

        "compressed_gb": float(
            compressed_bytes / GB
        ),
    }


query_size = member_size_summary(
    query_members_unique
)

gallery_size = member_size_summary(
    gallery_members_unique
)


# ------------------------------------------------------------
# 8. Scale-by-scale size plan
# ------------------------------------------------------------

scale_records = []

for map_size in map_sizes:

    scale_df = (
        gallery_df[
            gallery_df[
                "map_size"
            ].astype(int)
            == map_size
        ]
    )

    members = sorted(
        set(
            scale_df[
                "satellite_member"
            ].astype(str)
        )
    )

    size_info = (
        member_size_summary(
            members
        )
    )

    scale_records.append({
        "map_size": int(
            map_size
        ),

        "gallery_count": int(
            len(
                scale_df
            )
        ),

        "unique_zip_members": int(
            len(
                members
            )
        ),

        "raw_bytes": (
            size_info[
                "raw_bytes"
            ]
        ),

        "raw_gb": (
            size_info[
                "raw_gb"
            ]
        ),

        "compressed_gb": (
            size_info[
                "compressed_gb"
            ]
        ),
    })


scale_plan_df = pd.DataFrame(
    scale_records
)


# ------------------------------------------------------------
# 9. Total required extraction size
# ------------------------------------------------------------

all_required_members = sorted(
    set(
        query_members_unique
        +
        gallery_members_unique
    )
)

total_size = member_size_summary(
    all_required_members
)


# ------------------------------------------------------------
# 10. Local disk availability
# ------------------------------------------------------------

disk = shutil.disk_usage(
    "/content"
)

disk_total_gb = (
    disk.total
    / GB
)

disk_used_gb = (
    disk.used
    / GB
)

disk_free_gb = (
    disk.free
    / GB
)


# ------------------------------------------------------------
# 11. Cache strategy
# ------------------------------------------------------------
# We require headroom for:
# - PyTorch
# - extracted images
# - embeddings
# - temporary arrays
#
# Keep at least 15 GB free after extraction.
# ------------------------------------------------------------

SAFETY_RESERVE_GB = 15.0

full_cache_required_gb = (
    total_size[
        "raw_gb"
    ]
)

full_cache_safe = bool(
    disk_free_gb
    >
    (
        full_cache_required_gb
        +
        SAFETY_RESERVE_GB
    )
)


largest_scale_raw_gb = float(
    scale_plan_df[
        "raw_gb"
    ].max()
)

streaming_cache_required_gb = (
    query_size[
        "raw_gb"
    ]
    +
    largest_scale_raw_gb
)

scale_streaming_safe = bool(
    disk_free_gb
    >
    (
        streaming_cache_required_gb
        +
        SAFETY_RESERVE_GB
    )
)


if full_cache_safe:

    recommended_strategy = (
        "FULL_LOCAL_CACHE"
    )

elif scale_streaming_safe:

    recommended_strategy = (
        "QUERY_CACHE_PLUS_ONE_SCALE_AT_A_TIME"
    )

else:

    recommended_strategy = (
        "DIRECT_ZIP_STREAMING_OR_REDUCED_CACHE"
    )


# ------------------------------------------------------------
# 12. Save cache plan
# ------------------------------------------------------------

SCALE_PLAN_CSV = (
    CACHE_PLAN_ROOT
    / "vps_scale_cache_plan.csv"
)

CACHE_PLAN_JSON = (
    CACHE_PLAN_ROOT
    / "vps_cache_plan.json"
)

scale_plan_df.to_csv(
    SCALE_PLAN_CSV,
    index=False,
)


cache_plan = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "core_query_count": int(
        len(
            query_df
        )
    ),

    "core_gallery_record_count": int(
        len(
            gallery_df
        )
    ),

    "unique_query_members": int(
        len(
            query_members_unique
        )
    ),

    "unique_gallery_members": int(
        len(
            gallery_members_unique
        )
    ),

    "unique_required_members_total": int(
        len(
            all_required_members
        )
    ),

    "query_cache_raw_gb": (
        query_size[
            "raw_gb"
        ]
    ),

    "gallery_all_scales_raw_gb": (
        gallery_size[
            "raw_gb"
        ]
    ),

    "full_cache_raw_gb": (
        full_cache_required_gb
    ),

    "largest_single_scale_raw_gb": (
        largest_scale_raw_gb
    ),

    "query_plus_largest_scale_raw_gb": (
        streaming_cache_required_gb
    ),

    "local_disk": {
        "total_gb": (
            disk_total_gb
        ),

        "used_gb": (
            disk_used_gb
        ),

        "free_gb": (
            disk_free_gb
        ),
    },

    "safety_reserve_gb": (
        SAFETY_RESERVE_GB
    ),

    "full_cache_safe": (
        full_cache_safe
    ),

    "scale_streaming_safe": (
        scale_streaming_safe
    ),

    "recommended_strategy": (
        recommended_strategy
    ),

    "note": (
        "No images were extracted by this cell."
    ),
}


CACHE_PLAN_JSON.write_text(
    json.dumps(
        cache_plan,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 13. Final output
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("✅ NOTEBOOK 04 CELL 6 COMPLETE")
print("=" * 94)

print("\nUnique UAV query images:")
print(
    f"{len(query_members_unique):,}"
)

print("\nUnique satellite images:")
print(
    f"{len(gallery_members_unique):,}"
)

print("\nTotal unique required images:")
print(
    f"{len(all_required_members):,}"
)

print("\nQuery cache size:")
print(
    f"{query_size['raw_gb']:.3f} GB"
)

print("\nAll gallery scales size:")
print(
    f"{gallery_size['raw_gb']:.3f} GB"
)

print("\nFull required cache size:")
print(
    f"{full_cache_required_gb:.3f} GB"
)

print("\nScale-by-scale gallery size:")
print(
    scale_plan_df[
        [
            "map_size",
            "gallery_count",
            "raw_gb",
            "compressed_gb",
        ]
    ].to_string(
        index=False
    )
)

print("\nLocal /content disk:")
print(
    f"Total : {disk_total_gb:.2f} GB"
)

print(
    f"Used  : {disk_used_gb:.2f} GB"
)

print(
    f"Free  : {disk_free_gb:.2f} GB"
)

print("\nFull cache safe:")
print(
    full_cache_safe
)

print("\nQuery + one-scale cache safe:")
print(
    scale_streaming_safe
)

print("\nRECOMMENDED STRATEGY:")
print(
    recommended_strategy
)

print("\nCache plan:")
print(
    CACHE_PLAN_JSON
)

print("\n" + "=" * 94)

if (
    recommended_strategy
    == "FULL_LOCAL_CACHE"
):

    print(
        "✅ Enough disk for complete local "
        "evaluation cache."
    )

elif (
    recommended_strategy
    == "QUERY_CACHE_PLUS_ONE_SCALE_AT_A_TIME"
):

    print(
        "✅ Use query cache once, then process "
        "one satellite scale at a time."
    )

else:

    print(
        "⚠ Avoid bulk extraction. "
        "Use direct ZIP streaming."
    )

print("=" * 94)

NOTEBOOK 04 — VPS CORE CACHE PLANNING

Core queries:
2,331

Gallery records:
27,972

Map sizes:
[700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800]

Reading ZIP central directory...

✅ NOTEBOOK 04 CELL 6 COMPLETE

Unique UAV query images:
2,331

Unique satellite images:
27,972

Total unique required images:
30,303

Query cache size:
0.095 GB

All gallery scales size:
3.957 GB

Full required cache size:
4.052 GB

Scale-by-scale gallery size:
 map_size  gallery_count   raw_gb  compressed_gb
      700           2331 0.282234       0.281958
      800           2331 0.301457       0.301140
      900           2331 0.318768       0.318357
     1000           2331 0.338026       0.337652
     1100           2331 0.352142       0.351699
     1200           2331 0.358399       0.357401
     1300           2331 0.358001       0.356433
     1400           2331 0.334120       0.330780
     1500           2331 0.350292       0.347038
     1600           2331 0.324425       0.31979

In [32]:
# ============================================================
# NOTEBOOK 04 — CELL 7
# Extract complete leakage-safe VPS evaluation cache locally
#
# 2,331 UAV queries
# 27,972 satellite images (12 scales)
# Total raw cache ~4.05 GB
#
# RESUMABLE:
# Existing files with correct byte size are skipped.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import zipfile
import shutil
import json
import time

import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "mobilegeo_project"
)

VPS_ARCHIVE = (
    MYDRIVE
    / "vps_dataset_archive.zip"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

CACHE_ROOT = Path(
    "/content/vps_core_eval_cache"
)

QUERY_CACHE_ROOT = (
    CACHE_ROOT
    / "query"
)

GALLERY_CACHE_ROOT = (
    CACHE_ROOT
    / "gallery"
)

QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_queries.csv"
)

GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_core_eval_gallery_all_scales.csv"
)

for path in [
    VPS_ARCHIVE,
    QUERY_CSV,
    GALLERY_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )

for directory in [
    CACHE_ROOT,
    QUERY_CACHE_ROOT,
    GALLERY_CACHE_ROOT,
    AUDIT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 2. Load benchmark manifests
# ------------------------------------------------------------

print("=" * 96)
print("NOTEBOOK 04 — FULL LOCAL VPS EVALUATION CACHE")
print("=" * 96)

query_df = pd.read_csv(
    QUERY_CSV
)

gallery_df = pd.read_csv(
    GALLERY_CSV
)

print("\nQueries:")
print(
    f"{len(query_df):,}"
)

print("\nGallery records:")
print(
    f"{len(gallery_df):,}"
)

print("\nTotal required images:")
print(
    f"{len(query_df) + len(gallery_df):,}"
)


# ------------------------------------------------------------
# 3. Validate required fields
# ------------------------------------------------------------

required_query_columns = {
    "query_id",
    "sample_key",
    "uav_member",
}

required_gallery_columns = {
    "query_id",
    "sample_key",
    "map_size",
    "satellite_member",
}

missing_query_columns = (
    required_query_columns
    - set(
        query_df.columns
    )
)

missing_gallery_columns = (
    required_gallery_columns
    - set(
        gallery_df.columns
    )
)

if missing_query_columns:
    raise RuntimeError(
        "Missing query columns: "
        f"{sorted(missing_query_columns)}"
    )

if missing_gallery_columns:
    raise RuntimeError(
        "Missing gallery columns: "
        f"{sorted(missing_gallery_columns)}"
    )

if not query_df[
    "query_id"
].is_unique:
    raise RuntimeError(
        "query_id is not unique."
    )


# ------------------------------------------------------------
# 4. Build deterministic local paths
# ------------------------------------------------------------

def suffix_or_default(
    member,
    default=".jpg",
):
    suffix = (
        Path(
            str(member)
        ).suffix
        .lower()
    )

    return (
        suffix
        if suffix
        else default
    )


query_local_paths = []

for _, row in query_df.iterrows():

    query_id = str(
        row[
            "query_id"
        ]
    )

    extension = suffix_or_default(
        row[
            "uav_member"
        ]
    )

    local_path = (
        QUERY_CACHE_ROOT
        / f"{query_id}{extension}"
    )

    query_local_paths.append(
        str(
            local_path
        )
    )

query_df[
    "local_path"
] = query_local_paths


gallery_local_paths = []

gallery_ids = []

for _, row in gallery_df.iterrows():

    query_id = str(
        row[
            "query_id"
        ]
    )

    map_size = int(
        row[
            "map_size"
        ]
    )

    extension = suffix_or_default(
        row[
            "satellite_member"
        ]
    )

    scale_root = (
        GALLERY_CACHE_ROOT
        / f"map_{map_size}"
    )

    local_path = (
        scale_root
        / f"{query_id}{extension}"
    )

    gallery_local_paths.append(
        str(
            local_path
        )
    )

    gallery_ids.append(
        f"g_{map_size}_{query_id}"
    )

gallery_df[
    "gallery_id"
] = gallery_ids

gallery_df[
    "local_path"
] = gallery_local_paths


# ------------------------------------------------------------
# 5. Check path uniqueness
# ------------------------------------------------------------

if not query_df[
    "local_path"
].is_unique:
    raise RuntimeError(
        "Query local paths are not unique."
    )

if not gallery_df[
    "local_path"
].is_unique:
    raise RuntimeError(
        "Gallery local paths are not unique."
    )


# ------------------------------------------------------------
# 6. Create scale directories
# ------------------------------------------------------------

map_sizes = sorted(
    gallery_df[
        "map_size"
    ]
    .astype(int)
    .unique()
    .tolist()
)

for map_size in map_sizes:

    (
        GALLERY_CACHE_ROOT
        / f"map_{map_size}"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 7. Build extraction jobs
# ------------------------------------------------------------

jobs = []

for _, row in query_df.iterrows():

    jobs.append({
        "kind": "query",

        "member": str(
            row[
                "uav_member"
            ]
        ),

        "destination": str(
            row[
                "local_path"
            ]
        ),
    })


for _, row in gallery_df.iterrows():

    jobs.append({
        "kind": "gallery",

        "member": str(
            row[
                "satellite_member"
            ]
        ),

        "destination": str(
            row[
                "local_path"
            ]
        ),
    })


# ------------------------------------------------------------
# 8. Extract required files
# ------------------------------------------------------------

total_jobs = len(
    jobs
)

extracted = 0
skipped = 0
failed = []

bytes_written = 0

start_time = time.time()

print("\n" + "=" * 96)
print("EXTRACTING REQUIRED IMAGES")
print("=" * 96)

with zipfile.ZipFile(
    VPS_ARCHIVE,
    "r",
) as zf:

    zip_info = {
        info.filename: info
        for info in zf.infolist()
        if not info.is_dir()
    }

    for index, job in enumerate(
        jobs,
        start=1,
    ):

        member = (
            job[
                "member"
            ]
        )

        destination = Path(
            job[
                "destination"
            ]
        )

        info = zip_info.get(
            member
        )

        if info is None:

            failed.append({
                "member": member,
                "destination": str(
                    destination
                ),
                "error": (
                    "ZIP_MEMBER_NOT_FOUND"
                ),
            })

            continue

        expected_size = int(
            info.file_size
        )

        # ----------------------------------------------------
        # Resume behavior
        # ----------------------------------------------------

        if (
            destination.is_file()
            and
            destination.stat().st_size
            == expected_size
        ):

            skipped += 1

        else:

            try:

                destination.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                temporary_path = (
                    destination.with_suffix(
                        destination.suffix
                        + ".part"
                    )
                )

                if temporary_path.exists():
                    temporary_path.unlink()

                with zf.open(
                    member,
                    "r",
                ) as source:

                    with open(
                        temporary_path,
                        "wb",
                    ) as target:

                        shutil.copyfileobj(
                            source,
                            target,
                            length=1024 * 1024,
                        )

                actual_size = (
                    temporary_path
                    .stat()
                    .st_size
                )

                if (
                    actual_size
                    != expected_size
                ):

                    temporary_path.unlink(
                        missing_ok=True
                    )

                    raise RuntimeError(
                        "SIZE_MISMATCH: "
                        f"{actual_size} != "
                        f"{expected_size}"
                    )

                temporary_path.replace(
                    destination
                )

                extracted += 1

                bytes_written += (
                    expected_size
                )

            except Exception as error:

                failed.append({
                    "member": member,
                    "destination": str(
                        destination
                    ),
                    "error": repr(
                        error
                    ),
                })

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            index % 1000 == 0
            or
            index == total_jobs
        ):

            elapsed = (
                time.time()
                - start_time
            )

            rate = (
                index
                / elapsed
                if elapsed > 0
                else 0.0
            )

            remaining = (
                total_jobs
                - index
            )

            eta_min = (
                remaining
                / rate
                / 60.0
                if rate > 0
                else 0.0
            )

            print(
                f"{index:>6,} / "
                f"{total_jobs:,} | "
                f"new={extracted:,} | "
                f"skipped={skipped:,} | "
                f"failed={len(failed):,} | "
                f"ETA≈{eta_min:.1f} min"
            )


# ------------------------------------------------------------
# 9. Verify complete local cache
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("VERIFYING LOCAL CACHE")
print("=" * 96)

query_exists = (
    query_df[
        "local_path"
    ]
    .map(
        lambda p: Path(
            p
        ).is_file()
    )
)

gallery_exists = (
    gallery_df[
        "local_path"
    ]
    .map(
        lambda p: Path(
            p
        ).is_file()
    )
)

query_df[
    "local_file_exists"
] = query_exists

gallery_df[
    "local_file_exists"
] = gallery_exists


query_ready_count = int(
    query_exists.sum()
)

gallery_ready_count = int(
    gallery_exists.sum()
)

all_queries_ready = bool(
    query_exists.all()
)

all_gallery_ready = bool(
    gallery_exists.all()
)

full_cache_ready = bool(
    all_queries_ready
    and
    all_gallery_ready
    and
    len(
        failed
    ) == 0
)


# ------------------------------------------------------------
# 10. Local cache size
# ------------------------------------------------------------

cache_size_bytes = 0

for path in CACHE_ROOT.rglob(
    "*"
):

    if path.is_file():

        cache_size_bytes += (
            path.stat().st_size
        )

cache_size_gb = (
    cache_size_bytes
    / (
        1024 ** 3
    )
)


# ------------------------------------------------------------
# 11. Save LOCAL executable manifests
# ------------------------------------------------------------

LOCAL_QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_cached_query_manifest.csv"
)

LOCAL_GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_cached_gallery_manifest.csv"
)

FAILED_CSV = (
    AUDIT_ROOT
    / "vps_cache_extraction_failures.csv"
)

CACHE_AUDIT_JSON = (
    AUDIT_ROOT
    / "vps_local_cache_audit.json"
)

query_df.to_csv(
    LOCAL_QUERY_CSV,
    index=False,
)

gallery_df.to_csv(
    LOCAL_GALLERY_CSV,
    index=False,
)

pd.DataFrame(
    failed
).to_csv(
    FAILED_CSV,
    index=False,
)


# ------------------------------------------------------------
# 12. Per-scale verification
# ------------------------------------------------------------

scale_records = []

for map_size in map_sizes:

    scale_df = (
        gallery_df[
            gallery_df[
                "map_size"
            ].astype(int)
            == map_size
        ]
    )

    ready = int(
        scale_df[
            "local_file_exists"
        ].sum()
    )

    scale_records.append({
        "map_size": (
            map_size
        ),

        "expected": int(
            len(
                scale_df
            )
        ),

        "ready": (
            ready
        ),

        "complete": bool(
            ready
            == len(
                scale_df
            )
        ),
    })

scale_status_df = pd.DataFrame(
    scale_records
)


# ------------------------------------------------------------
# 13. Save audit
# ------------------------------------------------------------

runtime_seconds = (
    time.time()
    - start_time
)

audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "cache_root": str(
        CACHE_ROOT
    ),

    "query_expected": int(
        len(
            query_df
        )
    ),

    "query_ready": (
        query_ready_count
    ),

    "gallery_expected": int(
        len(
            gallery_df
        )
    ),

    "gallery_ready": (
        gallery_ready_count
    ),

    "new_files_extracted": (
        extracted
    ),

    "existing_files_skipped": (
        skipped
    ),

    "failed_files": int(
        len(
            failed
        )
    ),

    "cache_size_gb": (
        cache_size_gb
    ),

    "runtime_seconds": (
        runtime_seconds
    ),

    "full_cache_ready": (
        full_cache_ready
    ),

    "protocol": (
        "PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "outputs": {
        "query_manifest": str(
            LOCAL_QUERY_CSV
        ),

        "gallery_manifest": str(
            LOCAL_GALLERY_CSV
        ),

        "failure_log": str(
            FAILED_CSV
        ),
    },
}

CACHE_AUDIT_JSON.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 14. Disk status after extraction
# ------------------------------------------------------------

disk = shutil.disk_usage(
    "/content"
)

free_gb = (
    disk.free
    / (
        1024 ** 3
    )
)


# ------------------------------------------------------------
# 15. Final output
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("✅ NOTEBOOK 04 CELL 7 COMPLETE")
print("=" * 96)

print("\nQueries cached:")
print(
    f"{query_ready_count:,} / "
    f"{len(query_df):,}"
)

print("\nGallery images cached:")
print(
    f"{gallery_ready_count:,} / "
    f"{len(gallery_df):,}"
)

print("\nNew files extracted:")
print(
    f"{extracted:,}"
)

print("\nExisting valid files skipped:")
print(
    f"{skipped:,}"
)

print("\nFailures:")
print(
    f"{len(failed):,}"
)

print("\nLocal cache size:")
print(
    f"{cache_size_gb:.3f} GB"
)

print("\nPer-scale status:")
print(
    scale_status_df.to_string(
        index=False
    )
)

print("\nFull cache ready:")
print(
    full_cache_ready
)

print("\nRemaining /content disk:")
print(
    f"{free_gb:.2f} GB"
)

print("\nCached query manifest:")
print(
    LOCAL_QUERY_CSV
)

print("\nCached gallery manifest:")
print(
    LOCAL_GALLERY_CSV
)

print("\nCache audit:")
print(
    CACHE_AUDIT_JSON
)

print("\n" + "=" * 96)

if full_cache_ready:

    print(
        "✅ VPS CORE EVALUATION CACHE READY"
    )

    print(
        "\nNext: run descriptor extraction on the "
        "2,331 UAV queries and 27,972 satellite images."
    )

else:

    print(
        "⚠ CACHE INCOMPLETE — rerun this cell. "
        "It will resume and skip valid files."
    )

print("=" * 96)

NOTEBOOK 04 — FULL LOCAL VPS EVALUATION CACHE

Queries:
2,331

Gallery records:
27,972

Total required images:
30,303

EXTRACTING REQUIRED IMAGES
 1,000 / 30,303 | new=1,000 | skipped=0 | failed=0 | ETA≈7.7 min
 2,000 / 30,303 | new=2,000 | skipped=0 | failed=0 | ETA≈6.4 min
 3,000 / 30,303 | new=3,000 | skipped=0 | failed=0 | ETA≈4.9 min
 4,000 / 30,303 | new=4,000 | skipped=0 | failed=0 | ETA≈4.1 min
 5,000 / 30,303 | new=5,000 | skipped=0 | failed=0 | ETA≈3.4 min
 6,000 / 30,303 | new=6,000 | skipped=0 | failed=0 | ETA≈3.2 min
 7,000 / 30,303 | new=7,000 | skipped=0 | failed=0 | ETA≈3.1 min
 8,000 / 30,303 | new=8,000 | skipped=0 | failed=0 | ETA≈2.8 min
 9,000 / 30,303 | new=9,000 | skipped=0 | failed=0 | ETA≈2.5 min
10,000 / 30,303 | new=10,000 | skipped=0 | failed=0 | ETA≈2.4 min
11,000 / 30,303 | new=11,000 | skipped=0 | failed=0 | ETA≈2.1 min
12,000 / 30,303 | new=12,000 | skipped=0 | failed=0 | ETA≈1.9 min
13,000 / 30,303 | new=13,000 | skipped=0 | failed=0 | ETA≈1.9 min
14,00

In [33]:
# ============================================================
# NOTEBOOK 04 — CELL 8A
# Exact local-model implementation / checkpoint audit
#
# Purpose:
# - Locate the exact existing model definition
# - Inspect checkpoint structure and parameter names
# - Avoid guessing architecture before 30k-image inference
# ============================================================

from pathlib import Path
import json
import re
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CHECKPOINT = (
    PROJECT_ROOT
    / "baseline_v1"
    / "checkpoint"
    / "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth"
)

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        CHECKPOINT
    )

print("=" * 94)
print("NOTEBOOK 04 — LOCAL MODEL EXACT-LOADER AUDIT")
print("=" * 94)

print("\nCheckpoint:")
print(CHECKPOINT)

print("\nCheckpoint size:")
print(
    f"{CHECKPOINT.stat().st_size / (1024**2):.2f} MB"
)

# ------------------------------------------------------------
# 2. Load checkpoint on CPU only
# ------------------------------------------------------------

checkpoint_obj = torch.load(
    CHECKPOINT,
    map_location="cpu",
)

print("\nCheckpoint object type:")
print(
    type(checkpoint_obj).__name__
)

if isinstance(
    checkpoint_obj,
    dict,
):

    print("\nTop-level checkpoint keys:")
    print(
        list(
            checkpoint_obj.keys()
        )[:50]
    )

# ------------------------------------------------------------
# 3. Resolve state_dict
# ------------------------------------------------------------

def resolve_state_dict(
    obj
):
    if not isinstance(
        obj,
        dict,
    ):
        raise RuntimeError(
            "Checkpoint is not a dictionary."
        )

    preferred_keys = [
        "state_dict",
        "model_state_dict",
        "model",
        "net",
        "network",
    ]

    for key in preferred_keys:

        candidate = obj.get(
            key
        )

        if isinstance(
            candidate,
            dict,
        ):

            tensor_values = [
                value
                for value in candidate.values()
                if torch.is_tensor(
                    value
                )
            ]

            if tensor_values:
                return candidate, key

    tensor_values = [
        value
        for value in obj.values()
        if torch.is_tensor(
            value
        )
    ]

    if tensor_values:
        return obj, "<checkpoint_root>"

    raise RuntimeError(
        "Could not resolve state_dict."
    )


state_dict, state_source = resolve_state_dict(
    checkpoint_obj
)

print("\nResolved state_dict source:")
print(
    state_source
)

print("\nState tensors:")
print(
    len(
        state_dict
    )
)

# ------------------------------------------------------------
# 4. Parameter key + shape preview
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("CHECKPOINT PARAMETER PREVIEW")
print("=" * 94)

for index, (
    key,
    value,
) in enumerate(
    state_dict.items()
):

    if index >= 80:
        break

    if torch.is_tensor(
        value
    ):

        print(
            f"{index:03d}  "
            f"{key:<70} "
            f"{tuple(value.shape)}"
        )

# ------------------------------------------------------------
# 5. Detect 960 -> 512 head tensors
# ------------------------------------------------------------

head_weight_candidates = []

for key, value in (
    state_dict.items()
):

    if not torch.is_tensor(
        value
    ):
        continue

    if (
        value.ndim == 2
        and
        tuple(
            value.shape
        ) == (
            512,
            960,
        )
    ):

        head_weight_candidates.append(
            key
        )

print("\n" + "=" * 94)
print("960 -> 512 HEAD CANDIDATES")
print("=" * 94)

print(
    "Count:",
    len(
        head_weight_candidates
    )
)

for key in head_weight_candidates:
    print(
        "✅",
        key
    )

# ------------------------------------------------------------
# 6. Search existing project source/notebooks
# ------------------------------------------------------------

SEARCH_TERMS = [
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION",
    "AdaptiveAvgPool2d",
    "mobilenet_v3_large",
    "MobileNetV3",
]

SOURCE_SUFFIXES = {
    ".py",
    ".ipynb",
}

search_roots = [
    PROJECT_ROOT / "baseline_v1",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "scripts",
    PROJECT_ROOT,
]

candidate_files = []

seen = set()

for root in search_roots:

    if not root.exists():
        continue

    for path in root.rglob(
        "*"
    ):

        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in SOURCE_SUFFIXES
        ):
            continue

        # Avoid giant notebooks/files.
        try:
            if (
                path.stat().st_size
                > 10 * 1024 * 1024
            ):
                continue
        except Exception:
            continue

        resolved = str(
            path
        )

        if resolved in seen:
            continue

        seen.add(
            resolved
        )

        candidate_files.append(
            path
        )

print("\nSource/notebook files inspected:")
print(
    len(
        candidate_files
    )
)

matches = []

for path in candidate_files:

    try:

        text = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    except Exception:
        continue

    matched_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower()
        in text.lower()
    ]

    if matched_terms:

        matches.append({
            "path": str(
                path
            ),
            "matched_terms": (
                matched_terms
            ),
            "text": (
                text
            ),
        })

print("\n" + "=" * 94)
print("EXACT MODEL-CODE CANDIDATES")
print("=" * 94)

print(
    "Matching files:",
    len(
        matches
    )
)

for result in matches[:20]:

    print(
        "\n✅",
        result[
            "path"
        ]
    )

    print(
        "   terms:",
        result[
            "matched_terms"
        ]
    )

# ------------------------------------------------------------
# 7. Print useful snippets
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("MODEL DEFINITION SNIPPETS")
print("=" * 94)

snippet_count = 0

patterns = [
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION",
    "class AdvancedEdgeGeoLPN",
    "AdaptiveAvgPool2d",
    "mobilenet_v3_large",
]

for result in matches:

    text = result[
        "text"
    ]

    lines = (
        text.splitlines()
    )

    hit_indices = []

    for line_index, line in enumerate(
        lines
    ):

        if any(
            pattern.lower()
            in line.lower()
            for pattern in patterns
        ):
            hit_indices.append(
                line_index
            )

    if not hit_indices:
        continue

    print(
        "\nFILE:",
        result[
            "path"
        ]
    )

    for line_index in hit_indices[:4]:

        start = max(
            0,
            line_index - 8,
        )

        end = min(
            len(
                lines
            ),
            line_index + 35,
        )

        print(
            "\n--- snippet ---"
        )

        for i in range(
            start,
            end,
        ):

            print(
                f"{i+1:05d}: "
                f"{lines[i]}"
            )

        snippet_count += 1

        if snippet_count >= 8:
            break

    if snippet_count >= 8:
        break

# ------------------------------------------------------------
# 8. Save audit
# ------------------------------------------------------------

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
    / "audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "local_model_exact_loader_audit.json"
)

audit = {
    "checkpoint": str(
        CHECKPOINT
    ),

    "checkpoint_state_source": (
        state_source
    ),

    "state_tensor_count": int(
        len(
            state_dict
        )
    ),

    "head_960x512_candidates": (
        head_weight_candidates
    ),

    "source_candidates": [
        {
            "path": item[
                "path"
            ],

            "matched_terms": item[
                "matched_terms"
            ],
        }
        for item in matches
    ],

    "required_model_name": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance": (
        "LOCAL_REIMPLEMENTATION"
    ),
}

AUDIT_JSON.write_text(
    json.dumps(
        audit,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 9. Final
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("✅ NOTEBOOK 04 CELL 8A COMPLETE")
print("=" * 94)

print("\n960->512 head tensors:")
print(
    len(
        head_weight_candidates
    )
)

print("\nExact model-code candidate files:")
print(
    len(
        matches
    )
)

print("\nAudit:")
print(
    AUDIT_JSON
)

print("\nNext:")
print(
    "Use the verified implementation above for "
    "VPS descriptor extraction; do not substitute "
    "an assumed architecture."
)

NOTEBOOK 04 — LOCAL MODEL EXACT-LOADER AUDIT

Checkpoint:
/content/drive/MyDrive/mobilegeo_project/baseline_v1/checkpoint/AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth

Checkpoint size:
19.05 MB

Checkpoint object type:
OrderedDict

Top-level checkpoint keys:
['backbone.0.0.weight', 'backbone.0.1.weight', 'backbone.0.1.bias', 'backbone.0.1.running_mean', 'backbone.0.1.running_var', 'backbone.0.1.num_batches_tracked', 'backbone.1.block.0.0.weight', 'backbone.1.block.0.1.weight', 'backbone.1.block.0.1.bias', 'backbone.1.block.0.1.running_mean', 'backbone.1.block.0.1.running_var', 'backbone.1.block.0.1.num_batches_tracked', 'backbone.1.block.1.0.weight', 'backbone.1.block.1.1.weight', 'backbone.1.block.1.1.bias', 'backbone.1.block.1.1.running_mean', 'backbone.1.block.1.1.running_var', 'backbone.1.block.1.1.num_batches_tracked', 'backbone.2.block.0.0.weight', 'backbone.2.block.0.1.weight', 'backbone.2.block.0.1.bias', 'backbone.2.block.0.1.running_mean', 'backbone.2.block.0.1.running_var',

In [34]:
# ============================================================
# NOTEBOOK 04 — CELL 8B
# AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
# VPS descriptor extraction
#
# Provenance:
# LOCAL_REIMPLEMENTATION
#
# IMPORTANT:
# - NOT official MobileGeo inference
# - Colab runtime timing, NOT Jetson Orin Nano latency
# - 224x224 ImageNet normalization
# - 2048-D descriptor
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
import json
import math
import re
import time
import hashlib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_large,
)

from PIL import Image


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CHECKPOINT = (
    PROJECT_ROOT
    / "baseline_v1"
    / "checkpoint"
    / "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

EMBED_ROOT = (
    PHASE4_ROOT
    / "embeddings"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

EMBED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

QUERY_MANIFEST = (
    MANIFEST_ROOT
    / "vps_cached_query_manifest.csv"
)

GALLERY_MANIFEST = (
    MANIFEST_ROOT
    / "vps_cached_gallery_manifest.csv"
)

for path in [
    CHECKPOINT,
    QUERY_MANIFEST,
    GALLERY_MANIFEST,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Runtime
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this extraction."
    )

device = torch.device(
    "cuda"
)

gpu_name = torch.cuda.get_device_name(
    0
)

print("=" * 96)
print("NOTEBOOK 04 — LOCAL MODEL VPS DESCRIPTOR EXTRACTION")
print("=" * 96)

print("\nModel:")
print(
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
)

print("\nProvenance:")
print(
    "LOCAL_REIMPLEMENTATION"
)

print("\nGPU:")
print(
    gpu_name
)

print("\nTiming policy:")
print(
    "GOOGLE_COLAB_RUNTIME_NOT_JETSON_ORIN_NANO"
)


# ------------------------------------------------------------
# 3. Checkpoint SHA256
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


checkpoint_sha256 = sha256_file(
    CHECKPOINT
)

EXPECTED_SHA256 = (
    "adc64177582638690b841c1be390b7d05dfb27f4d2001b3d52fb74455020b807"
)

print("\nCheckpoint SHA256:")
print(
    checkpoint_sha256
)

print("\nExpected SHA256 match:")
print(
    checkpoint_sha256
    == EXPECTED_SHA256
)

if checkpoint_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Checkpoint SHA256 does not match "
        "the previously validated local baseline."
    )


# ------------------------------------------------------------
# 4. Load checkpoint
# ------------------------------------------------------------

checkpoint_obj = torch.load(
    CHECKPOINT,
    map_location="cpu",
)

def resolve_state_dict(
    obj
):

    if not isinstance(
        obj,
        dict,
    ):
        raise RuntimeError(
            "Unexpected checkpoint structure."
        )

    for key in [
        "state_dict",
        "model_state_dict",
        "model",
        "net",
        "network",
    ]:

        candidate = obj.get(
            key
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and any(
                torch.is_tensor(v)
                for v in candidate.values()
            )
        ):

            return candidate

    if any(
        torch.is_tensor(v)
        for v in obj.values()
    ):
        return obj

    raise RuntimeError(
        "Could not locate state_dict."
    )


state_dict = resolve_state_dict(
    checkpoint_obj
)


# ------------------------------------------------------------
# 5. Canonical verified architecture
# ------------------------------------------------------------

class AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION(
    nn.Module
):

    def __init__(
        self
    ):
        super().__init__()

        mobilenet = mobilenet_v3_large(
            weights=None
        )

        self.features = (
            mobilenet.features
        )

        self.pool = (
            nn.AdaptiveAvgPool2d(
                (
                    4,
                    1,
                )
            )
        )

        self.heads = nn.ModuleList([
            nn.Linear(
                960,
                512,
            )
            for _ in range(
                4
            )
        ])

    def forward(
        self,
        x,
    ):

        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        descriptors = []

        for index, head in enumerate(
            self.heads
        ):

            part = x[
                :,
                :,
                index,
                0,
            ]

            part = head(
                part
            )

            descriptors.append(
                part
            )

        descriptor = torch.cat(
            descriptors,
            dim=1,
        )

        descriptor = F.normalize(
            descriptor,
            p=2,
            dim=1,
        )

        return descriptor


model = (
    AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION()
)


# ------------------------------------------------------------
# 6. Resolve MobileNetV3 feature prefix automatically
# ------------------------------------------------------------

feature_state = (
    model.features.state_dict()
)

feature_keys = list(
    feature_state.keys()
)

first_feature_key = (
    feature_keys[0]
)

candidate_prefixes = []

for checkpoint_key in (
    state_dict.keys()
):

    if checkpoint_key.endswith(
        first_feature_key
    ):

        prefix = checkpoint_key[
            :
            len(
                checkpoint_key
            )
            -
            len(
                first_feature_key
            )
        ]

        candidate_prefixes.append(
            prefix
        )


valid_feature_prefixes = []

for prefix in (
    candidate_prefixes
):

    matches = True

    for key, value in (
        feature_state.items()
    ):

        checkpoint_key = (
            prefix
            + key
        )

        if checkpoint_key not in (
            state_dict
        ):

            matches = False
            break

        if tuple(
            state_dict[
                checkpoint_key
            ].shape
        ) != tuple(
            value.shape
        ):

            matches = False
            break

    if matches:

        valid_feature_prefixes.append(
            prefix
        )


valid_feature_prefixes = list(
    dict.fromkeys(
        valid_feature_prefixes
    )
)

if len(
    valid_feature_prefixes
) != 1:

    print(
        "\nCandidate feature prefixes:"
    )

    print(
        valid_feature_prefixes
    )

    raise RuntimeError(
        "Could not uniquely resolve "
        "MobileNetV3 feature checkpoint prefix."
    )


feature_prefix = (
    valid_feature_prefixes[0]
)

print("\nResolved feature prefix:")
print(
    repr(
        feature_prefix
    )
)


# ------------------------------------------------------------
# 7. Strict-load MobileNetV3 features
# ------------------------------------------------------------

resolved_feature_state = {
    key: state_dict[
        feature_prefix
        + key
    ]
    for key in feature_keys
}

feature_load_result = (
    model.features.load_state_dict(
        resolved_feature_state,
        strict=True,
    )
)

print("\nFeature strict load:")
print(
    feature_load_result
)


# ------------------------------------------------------------
# 8. Resolve four 960 -> 512 FC heads
# ------------------------------------------------------------

head_weight_keys = [
    key
    for key, value in (
        state_dict.items()
    )
    if (
        torch.is_tensor(
            value
        )
        and
        value.ndim == 2
        and
        tuple(
            value.shape
        ) == (
            512,
            960,
        )
    )
]

if len(
    head_weight_keys
) != 4:

    raise RuntimeError(
        "Expected exactly four 960->512 "
        f"head weights, found {len(head_weight_keys)}."
    )


def natural_key(
    text
):
    return [
        int(token)
        if token.isdigit()
        else token.lower()
        for token in re.split(
            r"(\d+)",
            text,
        )
    ]


head_weight_keys = sorted(
    head_weight_keys,
    key=natural_key,
)

head_bias_keys = []

for weight_key in (
    head_weight_keys
):

    if not weight_key.endswith(
        ".weight"
    ):
        raise RuntimeError(
            f"Unexpected head weight key: "
            f"{weight_key}"
        )

    bias_key = (
        weight_key[
            :-len(
                ".weight"
            )
        ]
        + ".bias"
    )

    if bias_key not in (
        state_dict
    ):
        raise RuntimeError(
            f"Missing head bias: "
            f"{bias_key}"
        )

    if tuple(
        state_dict[
            bias_key
        ].shape
    ) != (
        512,
    ):
        raise RuntimeError(
            f"Unexpected bias shape: "
            f"{bias_key}"
        )

    head_bias_keys.append(
        bias_key
    )


for index in range(
    4
):

    model.heads[
        index
    ].weight.data.copy_(
        state_dict[
            head_weight_keys[
                index
            ]
        ]
    )

    model.heads[
        index
    ].bias.data.copy_(
        state_dict[
            head_bias_keys[
                index
            ]
        ]
    )


print("\nResolved FC heads:")

for index, key in enumerate(
    head_weight_keys
):
    print(
        f"Head {index}: {key}"
    )


# ------------------------------------------------------------
# 9. Parameter-use audit
# ------------------------------------------------------------

used_checkpoint_keys = set(
    feature_prefix
    + key
    for key in feature_keys
)

used_checkpoint_keys.update(
    head_weight_keys
)

used_checkpoint_keys.update(
    head_bias_keys
)

tensor_checkpoint_keys = {
    key
    for key, value in (
        state_dict.items()
    )
    if torch.is_tensor(
        value
    )
}

unused_tensor_keys = sorted(
    tensor_checkpoint_keys
    - used_checkpoint_keys
)

print("\nCheckpoint tensor keys used:")
print(
    f"{len(used_checkpoint_keys)} / "
    f"{len(tensor_checkpoint_keys)}"
)

print("\nUnused checkpoint tensor keys:")
print(
    len(
        unused_tensor_keys
    )
)

if unused_tensor_keys:

    print(
        "\nFirst unused keys:"
    )

    for key in (
        unused_tensor_keys[:30]
    ):
        print(
            "  ",
            key
        )

    raise RuntimeError(
        "Unexpected checkpoint tensors remain unused. "
        "Stopping rather than silently using an "
        "assumed architecture."
    )


# ------------------------------------------------------------
# 10. Model finalization
# ------------------------------------------------------------

model = model.to(
    device
)

model.eval()

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("\nParameter count:")
print(
    f"{parameter_count:,}"
)


# ------------------------------------------------------------
# 11. Preprocessing
# ------------------------------------------------------------

preprocess = transforms.Compose([
    transforms.Resize(
        (
            224,
            224,
        ),
        interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406,
        ],
        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),
])

PREPROCESS_PROVENANCE = (
    "PROVISIONAL_IMAGENET_NORMALIZATION"
)


# ------------------------------------------------------------
# 12. Dataset
# ------------------------------------------------------------

class ImagePathDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

    def __len__(
        self
    ):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index,
    ):

        row = self.dataframe.iloc[
            index
        ]

        path = str(
            row[
                "local_path"
            ]
        )

        with Image.open(
            path
        ) as image:

            image = image.convert(
                "RGB"
            )

            tensor = preprocess(
                image
            )

        return (
            tensor,
            index,
        )


# ------------------------------------------------------------
# 13. Load manifests
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_MANIFEST
)

gallery_df = pd.read_csv(
    GALLERY_MANIFEST
)


if not query_df[
    "local_file_exists"
].astype(bool).all():

    raise RuntimeError(
        "Query cache is incomplete."
    )

if not gallery_df[
    "local_file_exists"
].astype(bool).all():

    raise RuntimeError(
        "Gallery cache is incomplete."
    )


print("\n" + "=" * 96)
print("INFERENCE DATA")
print("=" * 96)

print("\nQueries:")
print(
    f"{len(query_df):,}"
)

print("\nGallery:")
print(
    f"{len(gallery_df):,}"
)


# ------------------------------------------------------------
# 14. DataLoader configuration
# ------------------------------------------------------------

BATCH_SIZE = 128

NUM_WORKERS = min(
    4,
    max(
        1,
        (
            __import__(
                "os"
            ).cpu_count()
            or 2
        )
        // 2,
    ),
)

print("\nBatch size:")
print(
    BATCH_SIZE
)

print("\nWorkers:")
print(
    NUM_WORKERS
)


# ------------------------------------------------------------
# 15. Descriptor extraction function
# ------------------------------------------------------------

def extract_descriptors(
    dataframe,
    label,
):

    dataset = ImagePathDataset(
        dataframe
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
    )

    descriptor_chunks = []

    processed = 0

    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    start_time = time.time()

    with torch.inference_mode():

        for batch_index, (
            images,
            indices,
        ) in enumerate(
            loader,
            start=1,
        ):

            images = images.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):

                descriptors = model(
                    images
                )

            if descriptors.ndim != 2:

                raise RuntimeError(
                    "Descriptor output must be 2-D."
                )

            if descriptors.shape[
                1
            ] != 2048:

                raise RuntimeError(
                    "Expected 2048-D descriptors, "
                    f"got {descriptors.shape}."
                )

            descriptors = (
                descriptors
                .float()
                .cpu()
                .numpy()
            )

            descriptor_chunks.append(
                descriptors
            )

            processed += len(
                images
            )

            if (
                batch_index % 25 == 0
                or
                processed == len(
                    dataset
                )
            ):

                elapsed = (
                    time.time()
                    - start_time
                )

                rate = (
                    processed
                    / elapsed
                    if elapsed > 0
                    else 0.0
                )

                remaining = (
                    len(
                        dataset
                    )
                    - processed
                )

                eta_minutes = (
                    remaining
                    / rate
                    / 60
                    if rate > 0
                    else 0.0
                )

                print(
                    f"{label}: "
                    f"{processed:,} / "
                    f"{len(dataset):,} | "
                    f"{rate:.1f} img/s | "
                    f"ETA≈{eta_minutes:.1f} min"
                )

    torch.cuda.synchronize()

    elapsed = (
        time.time()
        - start_time
    )

    descriptors = np.concatenate(
        descriptor_chunks,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    if len(
        descriptors
    ) != len(
        dataframe
    ):

        raise RuntimeError(
            "Descriptor count mismatch."
        )

    norms = np.linalg.norm(
        descriptors,
        axis=1,
    )

    return (
        descriptors,
        {
            "images": int(
                len(
                    dataframe
                )
            ),

            "runtime_seconds": float(
                elapsed
            ),

            "images_per_second": float(
                len(
                    dataframe
                )
                / elapsed
            ),

            "descriptor_dim": int(
                descriptors.shape[
                    1
                ]
            ),

            "norm_min": float(
                norms.min()
            ),

            "norm_mean": float(
                norms.mean()
            ),

            "norm_max": float(
                norms.max()
            ),
        },
    )


# ------------------------------------------------------------
# 16. Smoke test
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("MODEL SMOKE TEST")
print("=" * 96)

sample_path = str(
    query_df.iloc[
        0
    ][
        "local_path"
    ]
)

with Image.open(
    sample_path
) as image:

    sample_tensor = preprocess(
        image.convert(
            "RGB"
        )
    ).unsqueeze(
        0
    ).to(
        device
    )


with torch.inference_mode():

    sample_descriptor = model(
        sample_tensor
    )


print("\nSmoke output shape:")
print(
    tuple(
        sample_descriptor.shape
    )
)

print("\nSmoke descriptor norm:")
print(
    float(
        sample_descriptor.norm(
            dim=1
        ).item()
    )
)

if tuple(
    sample_descriptor.shape
) != (
    1,
    2048,
):

    raise RuntimeError(
        "Smoke test failed."
    )


# ------------------------------------------------------------
# 17. Extract QUERY descriptors
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("QUERY DESCRIPTOR EXTRACTION")
print("=" * 96)

query_descriptors, query_runtime = (
    extract_descriptors(
        query_df,
        "QUERY",
    )
)


# ------------------------------------------------------------
# 18. Save query descriptors immediately
# ------------------------------------------------------------

QUERY_NPZ = (
    EMBED_ROOT
    / "vps_local_query_embeddings.npz"
)

np.savez_compressed(
    QUERY_NPZ,

    embeddings=(
        query_descriptors
    ),

    query_id=(
        query_df[
            "query_id"
        ].astype(str).to_numpy()
    ),

    sample_key=(
        query_df[
            "sample_key"
        ].astype(str).to_numpy()
    ),

    dataset_partition=(
        query_df[
            "dataset_partition"
        ].astype(str).to_numpy()
    ),

    uav_E=(
        query_df[
            "uav_E"
        ].to_numpy(
            dtype=np.float64
        )
    ),

    uav_N=(
        query_df[
            "uav_N"
        ].to_numpy(
            dtype=np.float64
        )
    ),

    local_path=(
        query_df[
            "local_path"
        ].astype(str).to_numpy()
    ),

    model_name=np.array(
        [
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ]
    ),

    provenance=np.array(
        [
            "LOCAL_REIMPLEMENTATION"
        ]
    ),
)


print("\n✅ Query embeddings saved:")
print(
    QUERY_NPZ
)


# ------------------------------------------------------------
# 19. Extract GALLERY descriptors
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("GALLERY DESCRIPTOR EXTRACTION")
print("=" * 96)

gallery_descriptors, gallery_runtime = (
    extract_descriptors(
        gallery_df,
        "GALLERY",
    )
)


# ------------------------------------------------------------
# 20. Save gallery descriptors
# ------------------------------------------------------------

GALLERY_NPZ = (
    EMBED_ROOT
    / "vps_local_gallery_embeddings.npz"
)

np.savez_compressed(
    GALLERY_NPZ,

    embeddings=(
        gallery_descriptors
    ),

    gallery_id=(
        gallery_df[
            "gallery_id"
        ].astype(str).to_numpy()
    ),

    query_id=(
        gallery_df[
            "query_id"
        ].astype(str).to_numpy()
    ),

    sample_key=(
        gallery_df[
            "sample_key"
        ].astype(str).to_numpy()
    ),

    map_size=(
        gallery_df[
            "map_size"
        ].to_numpy(
            dtype=np.int32
        )
    ),

    satellite_member=(
        gallery_df[
            "satellite_member"
        ].astype(str).to_numpy()
    ),

    local_path=(
        gallery_df[
            "local_path"
        ].astype(str).to_numpy()
    ),

    model_name=np.array(
        [
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ]
    ),

    provenance=np.array(
        [
            "LOCAL_REIMPLEMENTATION"
        ]
    ),
)


print("\n✅ Gallery embeddings saved:")
print(
    GALLERY_NPZ
)


# ------------------------------------------------------------
# 21. Final validation
# ------------------------------------------------------------

query_shape = tuple(
    query_descriptors.shape
)

gallery_shape = tuple(
    gallery_descriptors.shape
)

descriptor_validation = bool(
    query_shape
    == (
        2331,
        2048,
    )
    and
    gallery_shape
    == (
        27972,
        2048,
    )
    and
    np.isfinite(
        query_descriptors
    ).all()
    and
    np.isfinite(
        gallery_descriptors
    ).all()
)


# ------------------------------------------------------------
# 22. Save extraction report
# ------------------------------------------------------------

REPORT_JSON = (
    AUDIT_ROOT
    / "vps_local_descriptor_extraction_report.json"
)

report = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model_name": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "checkpoint": str(
        CHECKPOINT
    ),

    "checkpoint_sha256": (
        checkpoint_sha256
    ),

    "checkpoint_sha256_verified": bool(
        checkpoint_sha256
        == EXPECTED_SHA256
    ),

    "architecture": {
        "backbone": (
            "MobileNetV3-Large features"
        ),

        "feature_channels": (
            960
        ),

        "pooling": (
            "AdaptiveAvgPool2d((4,1))"
        ),

        "branch_count": (
            4
        ),

        "branch_projection": (
            "960_to_512"
        ),

        "descriptor_dim": (
            2048
        ),

        "final_normalization": (
            "L2"
        ),
    },

    "preprocessing": {
        "input_size": [
            224,
            224,
        ],

        "normalization": (
            PREPROCESS_PROVENANCE
        ),

        "mean": [
            0.485,
            0.456,
            0.406,
        ],

        "std": [
            0.229,
            0.224,
            0.225,
        ],
    },

    "query": (
        query_runtime
    ),

    "gallery": (
        gallery_runtime
    ),

    "query_shape": list(
        query_shape
    ),

    "gallery_shape": list(
        gallery_shape
    ),

    "descriptor_validation": (
        descriptor_validation
    ),

    "runtime_environment": {
        "device": (
            gpu_name
        ),

        "timing_scope": (
            "DESCRIPTOR_EXTRACTION_INCLUDING_IMAGE_"
            "DECODE_PREPROCESS_AND_MODEL_FORWARD"
        ),

        "deployment_claim": (
            "GOOGLE_COLAB_RUNTIME_NOT_"
            "JETSON_ORIN_NANO"
        ),
    },

    "protocol": (
        "PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "outputs": {
        "query_embeddings": str(
            QUERY_NPZ
        ),

        "gallery_embeddings": str(
            GALLERY_NPZ
        ),
    },
}

REPORT_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 23. Cleanup RAM/GPU only
# ------------------------------------------------------------

del query_descriptors
del gallery_descriptors

torch.cuda.empty_cache()


# ------------------------------------------------------------
# 24. Final output
# ------------------------------------------------------------

print("\n" + "=" * 96)
print("✅ NOTEBOOK 04 CELL 8B COMPLETE")
print("=" * 96)

print("\nQuery descriptor shape:")
print(
    query_shape
)

print("\nGallery descriptor shape:")
print(
    gallery_shape
)

print("\nQuery runtime:")
print(
    f"{query_runtime['runtime_seconds']:.2f} sec"
)

print("\nQuery throughput:")
print(
    f"{query_runtime['images_per_second']:.2f} img/s"
)

print("\nGallery runtime:")
print(
    f"{gallery_runtime['runtime_seconds']:.2f} sec"
)

print("\nGallery throughput:")
print(
    f"{gallery_runtime['images_per_second']:.2f} img/s"
)

print("\nDescriptor validation:")
print(
    descriptor_validation
)

print("\nModel provenance:")
print(
    "LOCAL_REIMPLEMENTATION"
)

print("\nTiming provenance:")
print(
    "GOOGLE_COLAB_RUNTIME_NOT_JETSON_ORIN_NANO"
)

print("\nQuery embeddings:")
print(
    QUERY_NPZ
)

print("\nGallery embeddings:")
print(
    GALLERY_NPZ
)

print("\nExtraction report:")
print(
    REPORT_JSON
)

print("\n" + "=" * 96)

if descriptor_validation:

    print(
        "✅ LOCAL VPS DESCRIPTOR EXTRACTION COMPLETE"
    )

    print(
        "\nNext: run the 12-scale retrieval benchmark "
        "and calculate R@1/R@5/R@10/mAP by map size."
    )

else:

    print(
        "⚠ DESCRIPTOR VALIDATION FAILED"
    )

print("=" * 96)

NOTEBOOK 04 — LOCAL MODEL VPS DESCRIPTOR EXTRACTION

Model:
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION

Provenance:
LOCAL_REIMPLEMENTATION

GPU:
Tesla T4

Timing policy:
GOOGLE_COLAB_RUNTIME_NOT_JETSON_ORIN_NANO

Checkpoint SHA256:
adc64177582638690b841c1be390b7d05dfb27f4d2001b3d52fb74455020b807

Expected SHA256 match:
True

Resolved feature prefix:
'backbone.'

Feature strict load:
<All keys matched successfully>

Resolved FC heads:
Head 0: fcs.0.weight
Head 1: fcs.1.weight
Head 2: fcs.2.weight
Head 3: fcs.3.weight

Checkpoint tensor keys used:
316 / 316

Unused checkpoint tensor keys:
0

Parameter count:
4,940,080

INFERENCE DATA

Queries:
2,331

Gallery:
27,972

Batch size:
128

Workers:
1

MODEL SMOKE TEST

Smoke output shape:
(1, 2048)

Smoke descriptor norm:
0.9999999403953552

QUERY DESCRIPTOR EXTRACTION
QUERY: 2,331 / 2,331 | 113.3 img/s | ETA≈0.0 min

✅ Query embeddings saved:
/content/drive/MyDrive/mobilegeo_project/results/georeferenced_benchmark/embeddings/vps_local_query_em

In [35]:
# ============================================================
# NOTEBOOK 04 — CELL 9
# 12-scale VPS retrieval evaluation
#
# Model:
# AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
#
# Protocol:
# PROJECT_DEFINED_VPS_LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL
#
# Metrics:
# - R@1
# - R@5
# - R@10
# - mAP
# - mean positive rank
# - median positive rank
#
# IMPORTANT:
# - One positive gallery item per query per scale
# - Therefore per-query AP = 1 / positive_rank
# - Retrieval is NOT verified UAV pose
# - CRS remains UNVERIFIED
# - Search timing excludes descriptor extraction
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

EMBED_ROOT = (
    PHASE4_ROOT
    / "embeddings"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

REPORT_ROOT = (
    PHASE4_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE4_ROOT
    / "rankings"
)

for directory in [
    REPORT_ROOT,
    RANKING_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


QUERY_NPZ = (
    EMBED_ROOT
    / "vps_local_query_embeddings.npz"
)

GALLERY_NPZ = (
    EMBED_ROOT
    / "vps_local_gallery_embeddings.npz"
)

QUERY_MANIFEST = (
    MANIFEST_ROOT
    / "vps_cached_query_manifest.csv"
)

GALLERY_MANIFEST = (
    MANIFEST_ROOT
    / "vps_cached_gallery_manifest.csv"
)


for path in [
    QUERY_NPZ,
    GALLERY_NPZ,
    QUERY_MANIFEST,
    GALLERY_MANIFEST,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Load embeddings
# ------------------------------------------------------------

print("=" * 98)
print("NOTEBOOK 04 — VPS 12-SCALE RETRIEVAL BENCHMARK")
print("=" * 98)

query_npz = np.load(
    QUERY_NPZ,
    allow_pickle=True,
)

gallery_npz = np.load(
    GALLERY_NPZ,
    allow_pickle=True,
)

query_embeddings = (
    query_npz[
        "embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

gallery_embeddings = (
    gallery_npz[
        "embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

print("\nQuery embeddings:")
print(
    query_embeddings.shape
)

print("\nGallery embeddings:")
print(
    gallery_embeddings.shape
)


# ------------------------------------------------------------
# 3. Load manifests
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_MANIFEST
)

gallery_df = pd.read_csv(
    GALLERY_MANIFEST
)

if len(
    query_df
) != len(
    query_embeddings
):
    raise RuntimeError(
        "Query manifest / embedding mismatch."
    )

if len(
    gallery_df
) != len(
    gallery_embeddings
):
    raise RuntimeError(
        "Gallery manifest / embedding mismatch."
    )


# ------------------------------------------------------------
# 4. Verify descriptor normalization
# ------------------------------------------------------------

query_norms = np.linalg.norm(
    query_embeddings,
    axis=1,
)

gallery_norms = np.linalg.norm(
    gallery_embeddings,
    axis=1,
)

normalization_valid = bool(
    np.allclose(
        query_norms,
        1.0,
        atol=1e-4,
    )
    and
    np.allclose(
        gallery_norms,
        1.0,
        atol=1e-4,
    )
)

print("\nDescriptor normalization valid:")
print(
    normalization_valid
)

if not normalization_valid:
    raise RuntimeError(
        "Embeddings are not consistently L2-normalized."
    )


# ------------------------------------------------------------
# 5. Verify query identity
# ------------------------------------------------------------

query_sample_keys = (
    query_df[
        "sample_key"
    ]
    .astype(str)
    .to_numpy()
)

query_ids = (
    query_df[
        "query_id"
    ]
    .astype(str)
    .to_numpy()
)

if len(
    np.unique(
        query_sample_keys
    )
) != len(
    query_sample_keys
):
    raise RuntimeError(
        "Query sample_key must be unique."
    )

if len(
    np.unique(
        query_ids
    )
) != len(
    query_ids
):
    raise RuntimeError(
        "Query IDs must be unique."
    )


# ------------------------------------------------------------
# 6. Map sizes
# ------------------------------------------------------------

map_sizes = sorted(
    gallery_df[
        "map_size"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print("\nMap sizes:")
print(
    map_sizes
)

if map_sizes != [
    700,
    800,
    900,
    1000,
    1100,
    1200,
    1300,
    1400,
    1500,
    1600,
    1700,
    1800,
]:
    raise RuntimeError(
        "Unexpected map-size set."
    )


# ------------------------------------------------------------
# 7. Evaluation containers
# ------------------------------------------------------------

metric_records = []

per_query_records = []

top5_records = []

search_times = []


# ------------------------------------------------------------
# 8. Evaluate each map scale independently
# ------------------------------------------------------------

for map_size in map_sizes:

    print("\n" + "-" * 98)
    print(
        f"EVALUATING MAP SIZE {map_size}"
    )
    print("-" * 98)

    scale_mask = (
        gallery_df[
            "map_size"
        ].astype(int)
        .to_numpy()
        == map_size
    )

    scale_indices = np.flatnonzero(
        scale_mask
    )

    scale_gallery_df = (
        gallery_df.iloc[
            scale_indices
        ]
        .reset_index(
            drop=True
        )
    )

    scale_gallery_embeddings = (
        gallery_embeddings[
            scale_indices
        ]
    )

    if len(
        scale_gallery_df
    ) != len(
        query_df
    ):
        raise RuntimeError(
            f"Map {map_size}: expected "
            f"{len(query_df)} gallery rows, "
            f"found {len(scale_gallery_df)}."
        )

    # --------------------------------------------------------
    # 8a. Verify one exact positive per query
    # --------------------------------------------------------

    scale_sample_keys = (
        scale_gallery_df[
            "sample_key"
        ]
        .astype(str)
        .to_numpy()
    )

    if len(
        np.unique(
            scale_sample_keys
        )
    ) != len(
        scale_sample_keys
    ):
        raise RuntimeError(
            f"Map {map_size}: duplicate sample keys "
            "in gallery."
        )

    sample_to_gallery_index = {
        sample_key: index
        for index, sample_key in enumerate(
            scale_sample_keys
        )
    }

    missing_positive_keys = [
        sample_key
        for sample_key in query_sample_keys
        if sample_key
        not in sample_to_gallery_index
    ]

    if missing_positive_keys:
        raise RuntimeError(
            f"Map {map_size}: "
            f"{len(missing_positive_keys)} "
            "queries have no positive gallery."
        )

    positive_indices = np.array(
        [
            sample_to_gallery_index[
                sample_key
            ]
            for sample_key in (
                query_sample_keys
            )
        ],
        dtype=np.int32,
    )

    # --------------------------------------------------------
    # 8b. Similarity + full sort timing
    # --------------------------------------------------------
    # Scope:
    # cosine similarity matrix because embeddings are
    # already L2-normalized, plus full descending argsort.
    #
    # Descriptor extraction is NOT included.
    # --------------------------------------------------------

    start_search = time.perf_counter()

    similarities = (
        query_embeddings
        @ scale_gallery_embeddings.T
    )

    ranking = np.argsort(
        -similarities,
        axis=1,
    )

    search_seconds = (
        time.perf_counter()
        - start_search
    )

    search_times.append(
        search_seconds
    )

    # --------------------------------------------------------
    # 8c. Positive rank
    # --------------------------------------------------------

    inverse_ranking = np.empty_like(
        ranking
    )

    row_ids = np.arange(
        len(
            ranking
        )
    )[:, None]

    inverse_ranking[
        row_ids,
        ranking
    ] = np.arange(
        ranking.shape[
            1
        ]
    )

    # Convert zero-based position -> one-based rank.
    positive_ranks = (
        inverse_ranking[
            np.arange(
                len(
                    positive_indices
                )
            ),
            positive_indices,
        ]
        + 1
    )

    positive_ranks = (
        positive_ranks.astype(
            np.int32
        )
    )

    # --------------------------------------------------------
    # 8d. Retrieval metrics
    # --------------------------------------------------------

    recall_1 = float(
        np.mean(
            positive_ranks
            <= 1
        )
    )

    recall_5 = float(
        np.mean(
            positive_ranks
            <= 5
        )
    )

    recall_10 = float(
        np.mean(
            positive_ranks
            <= 10
        )
    )

    # Exactly one positive per query:
    # AP_i = 1 / rank_i.
    reciprocal_ranks = (
        1.0
        / positive_ranks.astype(
            np.float64
        )
    )

    mean_ap = float(
        reciprocal_ranks.mean()
    )

    mean_positive_rank = float(
        positive_ranks.mean()
    )

    median_positive_rank = float(
        np.median(
            positive_ranks
        )
    )

    max_positive_rank = int(
        positive_ranks.max()
    )

    top1_failures = int(
        np.sum(
            positive_ranks
            > 1
        )
    )

    # --------------------------------------------------------
    # 8e. Search timing
    # --------------------------------------------------------

    query_count = int(
        len(
            query_df
        )
    )

    gallery_count = int(
        len(
            scale_gallery_df
        )
    )

    similarity_count = int(
        query_count
        * gallery_count
    )

    search_ms_per_query = float(
        search_seconds
        * 1000.0
        / query_count
    )

    # --------------------------------------------------------
    # 8f. Save metric row
    # --------------------------------------------------------

    metric_records.append({
        "model": (
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ),

        "provenance": (
            "LOCAL_REIMPLEMENTATION"
        ),

        "map_size": int(
            map_size
        ),

        "query_count": (
            query_count
        ),

        "gallery_count": (
            gallery_count
        ),

        "positive_count_per_query": (
            1
        ),

        "R@1": (
            recall_1
        ),

        "R@5": (
            recall_5
        ),

        "R@10": (
            recall_10
        ),

        "mAP": (
            mean_ap
        ),

        "mean_positive_rank": (
            mean_positive_rank
        ),

        "median_positive_rank": (
            median_positive_rank
        ),

        "max_positive_rank": (
            max_positive_rank
        ),

        "top1_failure_count": (
            top1_failures
        ),

        "search_seconds": (
            search_seconds
        ),

        "search_ms_per_query": (
            search_ms_per_query
        ),

        "similarity_values_computed": (
            similarity_count
        ),

        "search_timing_scope": (
            "SIMILARITY_MATRIX_PLUS_FULL_SORT_"
            "DESCRIPTOR_EXTRACTION_EXCLUDED"
        ),
    })

    # --------------------------------------------------------
    # 8g. Per-query results
    # --------------------------------------------------------

    top1_indices = (
        ranking[
            :,
            0
        ]
    )

    top1_scores = (
        similarities[
            np.arange(
                query_count
            ),
            top1_indices,
        ]
    )

    positive_scores = (
        similarities[
            np.arange(
                query_count
            ),
            positive_indices,
        ]
    )

    top1_sample_keys = (
        scale_gallery_df.iloc[
            top1_indices
        ][
            "sample_key"
        ]
        .astype(str)
        .to_numpy()
    )

    top1_gallery_members = (
        scale_gallery_df.iloc[
            top1_indices
        ][
            "satellite_member"
        ]
        .astype(str)
        .to_numpy()
    )

    for query_index in range(
        query_count
    ):

        per_query_records.append({
            "model": (
                "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
            ),

            "map_size": int(
                map_size
            ),

            "query_id": str(
                query_ids[
                    query_index
                ]
            ),

            "query_sample_key": str(
                query_sample_keys[
                    query_index
                ]
            ),

            "positive_rank": int(
                positive_ranks[
                    query_index
                ]
            ),

            "reciprocal_rank": float(
                reciprocal_ranks[
                    query_index
                ]
            ),

            "top1_correct": bool(
                positive_ranks[
                    query_index
                ]
                == 1
            ),

            "top1_sample_key": str(
                top1_sample_keys[
                    query_index
                ]
            ),

            "top1_gallery_member": str(
                top1_gallery_members[
                    query_index
                ]
            ),

            "top1_similarity": float(
                top1_scores[
                    query_index
                ]
            ),

            "positive_similarity": float(
                positive_scores[
                    query_index
                ]
            ),
        })

    # --------------------------------------------------------
    # 8h. Save top-5 ranking evidence
    # --------------------------------------------------------

    top5 = ranking[
        :,
        :5
    ]

    for query_index in range(
        query_count
    ):

        for position in range(
            5
        ):

            gallery_local_index = int(
                top5[
                    query_index,
                    position,
                ]
            )

            gallery_row = (
                scale_gallery_df.iloc[
                    gallery_local_index
                ]
            )

            top5_records.append({
                "map_size": int(
                    map_size
                ),

                "query_id": str(
                    query_ids[
                        query_index
                    ]
                ),

                "query_sample_key": str(
                    query_sample_keys[
                        query_index
                    ]
                ),

                "rank": int(
                    position
                    + 1
                ),

                "retrieved_sample_key": str(
                    gallery_row[
                        "sample_key"
                    ]
                ),

                "retrieved_gallery_member": str(
                    gallery_row[
                        "satellite_member"
                    ]
                ),

                "similarity": float(
                    similarities[
                        query_index,
                        gallery_local_index,
                    ]
                ),

                "is_positive": bool(
                    gallery_local_index
                    == positive_indices[
                        query_index
                    ]
                ),
            })

    print(
        f"R@1  : {recall_1 * 100:.2f}%"
    )

    print(
        f"R@5  : {recall_5 * 100:.2f}%"
    )

    print(
        f"R@10 : {recall_10 * 100:.2f}%"
    )

    print(
        f"mAP  : {mean_ap * 100:.2f}%"
    )

    print(
        f"Mean positive rank: "
        f"{mean_positive_rank:.3f}"
    )

    print(
        f"Top-1 failures: "
        f"{top1_failures:,}"
    )

    print(
        f"Search time: "
        f"{search_seconds:.4f} sec "
        f"({search_ms_per_query:.4f} ms/query)"
    )

    # Free scale-specific ranking memory.
    del similarities
    del ranking
    del inverse_ranking


# ------------------------------------------------------------
# 9. Build result DataFrames
# ------------------------------------------------------------

metrics_df = pd.DataFrame(
    metric_records
)

per_query_df = pd.DataFrame(
    per_query_records
)

top5_df = pd.DataFrame(
    top5_records
)


# ------------------------------------------------------------
# 10. Macro summary across 12 independent scales
# ------------------------------------------------------------

macro_summary = {
    "model": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "protocol": (
        "PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "scale_count": int(
        len(
            metrics_df
        )
    ),

    "queries_per_scale": int(
        len(
            query_df
        )
    ),

    "gallery_per_scale": int(
        len(
            query_df
        )
    ),

    "macro_R@1": float(
        metrics_df[
            "R@1"
        ].mean()
    ),

    "macro_R@5": float(
        metrics_df[
            "R@5"
        ].mean()
    ),

    "macro_R@10": float(
        metrics_df[
            "R@10"
        ].mean()
    ),

    "macro_mAP": float(
        metrics_df[
            "mAP"
        ].mean()
    ),

    "macro_mean_positive_rank": float(
        metrics_df[
            "mean_positive_rank"
        ].mean()
    ),

    "total_top1_failures_across_scale_trials": int(
        metrics_df[
            "top1_failure_count"
        ].sum()
    ),

    "total_search_seconds_12_scales": float(
        sum(
            search_times
        )
    ),

    "mean_search_ms_per_query_per_scale": float(
        metrics_df[
            "search_ms_per_query"
        ].mean()
    ),

    "search_timing_scope": (
        "SIMILARITY_MATRIX_PLUS_FULL_SORT_"
        "DESCRIPTOR_EXTRACTION_EXCLUDED"
    ),

    "descriptor_extraction_timing_scope": (
        "REPORTED_SEPARATELY_IN_CELL_8B"
    ),

    "runtime_environment": (
        "GOOGLE_COLAB_NOT_JETSON_ORIN_NANO"
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "wgs84_claim_allowed": (
        False
    ),

    "meter_level_geodetic_claim_allowed": (
        False
    ),

    "retrieval_is_verified_uav_pose": (
        False
    ),
}


# ------------------------------------------------------------
# 11. Site-level diagnostic
# ------------------------------------------------------------

query_site_lookup = (
    query_df
    .set_index(
        "query_id"
    )[
        "site_token"
    ]
    .astype(str)
    .to_dict()
)

per_query_df[
    "site_token"
] = per_query_df[
    "query_id"
].map(
    query_site_lookup
)

site_records = []

for (
    map_size,
    site_token
), group in (
    per_query_df.groupby(
        [
            "map_size",
            "site_token",
        ]
    )
):

    ranks = (
        group[
            "positive_rank"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    site_records.append({
        "map_size": int(
            map_size
        ),

        "site_token": str(
            site_token
        ),

        "query_count": int(
            len(
                group
            )
        ),

        "R@1": float(
            np.mean(
                ranks <= 1
            )
        ),

        "R@5": float(
            np.mean(
                ranks <= 5
            )
        ),

        "R@10": float(
            np.mean(
                ranks <= 10
            )
        ),

        "mAP": float(
            np.mean(
                1.0
                / ranks.astype(
                    np.float64
                )
            )
        ),

        "mean_positive_rank": float(
            ranks.mean()
        ),
    })

site_df = pd.DataFrame(
    site_records
)


# ------------------------------------------------------------
# 12. Nominal-level diagnostic
# ------------------------------------------------------------

query_level_lookup = (
    query_df
    .set_index(
        "query_id"
    )[
        "nominal_level_token"
    ]
    .astype(str)
    .to_dict()
)

per_query_df[
    "nominal_level_token"
] = (
    per_query_df[
        "query_id"
    ].map(
        query_level_lookup
    )
)

level_records = []

for (
    map_size,
    nominal_level
), group in (
    per_query_df.groupby(
        [
            "map_size",
            "nominal_level_token",
        ]
    )
):

    ranks = (
        group[
            "positive_rank"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    level_records.append({
        "map_size": int(
            map_size
        ),

        "nominal_level_token": str(
            nominal_level
        ),

        "level_status": (
            "DATASET_NAME_TOKEN_ONLY_"
            "NOT_VERIFIED_AS_AGL_OR_MSL"
        ),

        "query_count": int(
            len(
                group
            )
        ),

        "R@1": float(
            np.mean(
                ranks <= 1
            )
        ),

        "R@5": float(
            np.mean(
                ranks <= 5
            )
        ),

        "R@10": float(
            np.mean(
                ranks <= 10
            )
        ),

        "mAP": float(
            np.mean(
                1.0
                / ranks.astype(
                    np.float64
                )
            )
        ),

        "mean_positive_rank": float(
            ranks.mean()
        ),
    })

level_df = pd.DataFrame(
    level_records
)


# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

METRICS_CSV = (
    REPORT_ROOT
    / "vps_local_12scale_retrieval_metrics.csv"
)

METRICS_JSON = (
    REPORT_ROOT
    / "vps_local_12scale_retrieval_metrics.json"
)

PER_QUERY_CSV = (
    RANKING_ROOT
    / "vps_local_per_query_rankings.csv"
)

TOP5_CSV = (
    RANKING_ROOT
    / "vps_local_top5_rankings.csv"
)

SITE_CSV = (
    REPORT_ROOT
    / "vps_local_per_site_metrics.csv"
)

LEVEL_CSV = (
    REPORT_ROOT
    / "vps_local_per_nominal_level_metrics.csv"
)

metrics_df.to_csv(
    METRICS_CSV,
    index=False,
)

per_query_df.to_csv(
    PER_QUERY_CSV,
    index=False,
)

top5_df.to_csv(
    TOP5_CSV,
    index=False,
)

site_df.to_csv(
    SITE_CSV,
    index=False,
)

level_df.to_csv(
    LEVEL_CSV,
    index=False,
)

report = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "benchmark": (
        "VPS_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "model": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "protocol": (
        "PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
    ),

    "positive_definition": (
        "EXACT_FULL_SAMPLE_KEY_MATCH"
    ),

    "one_positive_per_query_per_scale": (
        True
    ),

    "metrics_note": (
        "With exactly one positive per query, "
        "per-query average precision equals reciprocal rank."
    ),

    "map_sizes": (
        map_sizes
    ),

    "per_scale_metrics": (
        metrics_df.to_dict(
            orient="records"
        )
    ),

    "macro_summary": (
        macro_summary
    ),

    "reporting_constraints": {
        "crs_status": (
            "UNVERIFIED"
        ),

        "wgs84_claim_allowed": (
            False
        ),

        "meter_level_geodetic_claim_allowed": (
            False
        ),

        "retrieval_is_verified_uav_pose": (
            False
        ),

        "nominal_level_status": (
            "NOT_VERIFIED_AS_AGL_OR_MSL"
        ),
    },

    "outputs": {
        "metrics_csv": str(
            METRICS_CSV
        ),

        "per_query_rankings": str(
            PER_QUERY_CSV
        ),

        "top5_rankings": str(
            TOP5_CSV
        ),

        "per_site_metrics": str(
            SITE_CSV
        ),

        "per_nominal_level_metrics": str(
            LEVEL_CSV
        ),
    },
}

METRICS_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 14. Final display
# ------------------------------------------------------------

print("\n" + "=" * 98)
print("✅ NOTEBOOK 04 CELL 9 COMPLETE")
print("=" * 98)

display_metrics = (
    metrics_df[
        [
            "map_size",
            "R@1",
            "R@5",
            "R@10",
            "mAP",
            "mean_positive_rank",
            "top1_failure_count",
            "search_ms_per_query",
        ]
    ]
    .copy()
)

for column in [
    "R@1",
    "R@5",
    "R@10",
    "mAP",
]:

    display_metrics[
        column
    ] = (
        display_metrics[
            column
        ]
        * 100.0
    )

print("\n12-SCALE RESULTS")
print(
    display_metrics.to_string(
        index=False,
        formatters={
            "R@1": (
                lambda x:
                f"{x:.2f}%"
            ),

            "R@5": (
                lambda x:
                f"{x:.2f}%"
            ),

            "R@10": (
                lambda x:
                f"{x:.2f}%"
            ),

            "mAP": (
                lambda x:
                f"{x:.2f}%"
            ),

            "mean_positive_rank": (
                lambda x:
                f"{x:.3f}"
            ),

            "search_ms_per_query": (
                lambda x:
                f"{x:.4f}"
            ),
        },
    )
)


print("\n" + "-" * 98)
print("MACRO MEAN ACROSS 12 SCALE-CONTROLLED BENCHMARKS")
print("-" * 98)

print(
    f"Macro R@1  : "
    f"{macro_summary['macro_R@1'] * 100:.2f}%"
)

print(
    f"Macro R@5  : "
    f"{macro_summary['macro_R@5'] * 100:.2f}%"
)

print(
    f"Macro R@10 : "
    f"{macro_summary['macro_R@10'] * 100:.2f}%"
)

print(
    f"Macro mAP  : "
    f"{macro_summary['macro_mAP'] * 100:.2f}%"
)

print(
    f"Macro mean positive rank: "
    f"{macro_summary['macro_mean_positive_rank']:.3f}"
)

print(
    "\nTotal scale-trial top-1 failures:"
)

print(
    f"{macro_summary['total_top1_failures_across_scale_trials']:,}"
)

print(
    "\n12-scale search time:"
)

print(
    f"{macro_summary['total_search_seconds_12_scales']:.4f} sec"
)

print(
    "\nAverage search time per query per scale:"
)

print(
    f"{macro_summary['mean_search_ms_per_query_per_scale']:.4f} ms"
)

print("\nSearch timing scope:")
print(
    "SIMILARITY MATRIX + FULL SORT; "
    "DESCRIPTOR EXTRACTION EXCLUDED"
)

print("\nModel provenance:")
print(
    "LOCAL_REIMPLEMENTATION"
)

print("\nCRS:")
print(
    "UNVERIFIED — WGS84 NOT CLAIMED"
)

print("\nMeter-level geodetic error:")
print(
    "NOT REPORTED — CRS PROVENANCE UNVERIFIED"
)

print("\nRetrieval interpretation:")
print(
    "RETRIEVAL RESULT IS NOT VERIFIED UAV POSE"
)

print("\nMetrics CSV:")
print(
    METRICS_CSV
)

print("\nMetrics JSON:")
print(
    METRICS_JSON
)

print("\nPer-query rankings:")
print(
    PER_QUERY_CSV
)

print("\nTop-5 rankings:")
print(
    TOP5_CSV
)

print("\n" + "=" * 98)
print(
    "✅ LOCAL VPS 12-SCALE RETRIEVAL BENCHMARK COMPLETE"
)
print("=" * 98)

NOTEBOOK 04 — VPS 12-SCALE RETRIEVAL BENCHMARK

Query embeddings:
(2331, 2048)

Gallery embeddings:
(27972, 2048)

Descriptor normalization valid:
True

Map sizes:
[700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800]

--------------------------------------------------------------------------------------------------
EVALUATING MAP SIZE 700
--------------------------------------------------------------------------------------------------
R@1  : 0.09%
R@5  : 0.47%
R@10 : 0.73%
mAP  : 0.60%
Mean positive rank: 963.309
Top-1 failures: 2,329
Search time: 0.6155 sec (0.2640 ms/query)

--------------------------------------------------------------------------------------------------
EVALUATING MAP SIZE 800
--------------------------------------------------------------------------------------------------
R@1  : 0.09%
R@5  : 0.26%
R@10 : 0.39%
mAP  : 0.46%
Mean positive rank: 999.523
Top-1 failures: 2,329
Search time: 0.5099 sec (0.2188 ms/query)

-------------------------------

In [36]:
# ============================================================
# NOTEBOOK 04 — CELL 10
# Chance-level + positive-similarity diagnostic
#
# Purpose:
# - Compare measured retrieval against exact random chance
# - Check whether correct query/gallery pairs have stronger
#   similarity than randomly sampled negatives
# - Detect descriptor collapse / indexing problems
#
# NO image inference
# NO extraction
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

EMBED_ROOT = (
    PHASE4_ROOT
    / "embeddings"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

REPORT_ROOT = (
    PHASE4_ROOT
    / "reports"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

QUERY_NPZ = (
    EMBED_ROOT
    / "vps_local_query_embeddings.npz"
)

GALLERY_NPZ = (
    EMBED_ROOT
    / "vps_local_gallery_embeddings.npz"
)

QUERY_CSV = (
    MANIFEST_ROOT
    / "vps_cached_query_manifest.csv"
)

GALLERY_CSV = (
    MANIFEST_ROOT
    / "vps_cached_gallery_manifest.csv"
)

METRICS_CSV = (
    REPORT_ROOT
    / "vps_local_12scale_retrieval_metrics.csv"
)

for path in [
    QUERY_NPZ,
    GALLERY_NPZ,
    QUERY_CSV,
    GALLERY_CSV,
    METRICS_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

print("=" * 98)
print("NOTEBOOK 04 — CHANCE-LEVEL / POSITIVE-SIGNAL DIAGNOSTIC")
print("=" * 98)

q_npz = np.load(
    QUERY_NPZ,
    allow_pickle=True,
)

g_npz = np.load(
    GALLERY_NPZ,
    allow_pickle=True,
)

q_emb = (
    q_npz[
        "embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

g_emb = (
    g_npz[
        "embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

query_df = pd.read_csv(
    QUERY_CSV
)

gallery_df = pd.read_csv(
    GALLERY_CSV
)

metrics_df = pd.read_csv(
    METRICS_CSV
)

N = len(
    query_df
)

print("\nQueries per scale:")
print(
    f"{N:,}"
)


# ------------------------------------------------------------
# 3. Exact random-retrieval expectations
# ------------------------------------------------------------

random_r1 = (
    1.0 / N
)

random_r5 = (
    min(
        5,
        N,
    )
    / N
)

random_r10 = (
    min(
        10,
        N,
    )
    / N
)

# Expected reciprocal rank under a uniform random ranking:
#
# E[1/rank] = H_N / N
#
harmonic_number = sum(
    1.0 / rank
    for rank in range(
        1,
        N + 1,
    )
)

random_mrr = (
    harmonic_number
    / N
)

random_mean_rank = (
    N + 1
) / 2.0

print("\n" + "=" * 98)
print("EXACT RANDOM-CHANCE BASELINE")
print("=" * 98)

print(
    f"Random R@1  : "
    f"{random_r1 * 100:.4f}%"
)

print(
    f"Random R@5  : "
    f"{random_r5 * 100:.4f}%"
)

print(
    f"Random R@10 : "
    f"{random_r10 * 100:.4f}%"
)

print(
    f"Random expected mAP/MRR: "
    f"{random_mrr * 100:.4f}%"
)

print(
    f"Random expected mean rank: "
    f"{random_mean_rank:.3f}"
)


# ------------------------------------------------------------
# 4. Descriptor diversity diagnostics
# ------------------------------------------------------------

q_std = float(
    q_emb.std(
        axis=0
    ).mean()
)

g_std = float(
    g_emb.std(
        axis=0
    ).mean()
)

q_unique_rounded = int(
    np.unique(
        np.round(
            q_emb,
            decimals=5,
        ),
        axis=0,
    ).shape[0]
)

g_unique_rounded = int(
    np.unique(
        np.round(
            g_emb,
            decimals=5,
        ),
        axis=0,
    ).shape[0]
)

descriptor_collapse_detected = bool(
    q_unique_rounded
    < int(
        0.99 * len(
            q_emb
        )
    )
    or
    g_unique_rounded
    < int(
        0.99 * len(
            g_emb
        )
    )
)

print("\n" + "=" * 98)
print("DESCRIPTOR DIVERSITY")
print("=" * 98)

print(
    "Query unique rounded descriptors:"
)

print(
    f"{q_unique_rounded:,} / "
    f"{len(q_emb):,}"
)

print(
    "\nGallery unique rounded descriptors:"
)

print(
    f"{g_unique_rounded:,} / "
    f"{len(g_emb):,}"
)

print(
    "\nMean query dimension std:"
)

print(
    f"{q_std:.8f}"
)

print(
    "\nMean gallery dimension std:"
)

print(
    f"{g_std:.8f}"
)

print(
    "\nDescriptor collapse detected:"
)

print(
    descriptor_collapse_detected
)


# ------------------------------------------------------------
# 5. Positive-vs-random-negative diagnostic
# ------------------------------------------------------------

RNG_SEED = 42
NEGATIVES_PER_QUERY = 64

rng = np.random.default_rng(
    RNG_SEED
)

query_sample_keys = (
    query_df[
        "sample_key"
    ]
    .astype(str)
    .to_numpy()
)

map_sizes = sorted(
    gallery_df[
        "map_size"
    ]
    .astype(int)
    .unique()
    .tolist()
)

diagnostic_records = []

for map_size in map_sizes:

    mask = (
        gallery_df[
            "map_size"
        ].astype(int)
        .to_numpy()
        == map_size
    )

    scale_indices = np.flatnonzero(
        mask
    )

    scale_df = (
        gallery_df.iloc[
            scale_indices
        ]
        .reset_index(
            drop=True
        )
    )

    scale_emb = (
        g_emb[
            scale_indices
        ]
    )

    gallery_keys = (
        scale_df[
            "sample_key"
        ]
        .astype(str)
        .to_numpy()
    )

    key_to_index = {
        key: index
        for index, key
        in enumerate(
            gallery_keys
        )
    }

    positive_indices = np.array(
        [
            key_to_index[
                key
            ]
            for key in (
                query_sample_keys
            )
        ],
        dtype=np.int32,
    )

    # --------------------------------------------------------
    # Positive similarity
    # --------------------------------------------------------

    positive_similarity = np.sum(
        q_emb
        * scale_emb[
            positive_indices
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # Deterministic random negatives
    # --------------------------------------------------------

    negative_indices = rng.integers(
        low=0,
        high=N,
        size=(
            N,
            NEGATIVES_PER_QUERY,
        ),
    )

    # Guarantee sampled item is never the positive.
    collision_mask = (
        negative_indices
        == positive_indices[
            :,
            None,
        ]
    )

    while collision_mask.any():

        negative_indices[
            collision_mask
        ] = rng.integers(
            low=0,
            high=N,
            size=int(
                collision_mask.sum()
            ),
        )

        collision_mask = (
            negative_indices
            == positive_indices[
                :,
                None,
            ]
        )

    negative_embeddings = (
        scale_emb[
            negative_indices
        ]
    )

    negative_similarity = np.einsum(
        "qd,qkd->qk",
        q_emb,
        negative_embeddings,
    )

    # --------------------------------------------------------
    # Positive-vs-negative AUC-like probability
    # --------------------------------------------------------

    positive_beats_negative = (
        positive_similarity[
            :,
            None,
        ]
        >
        negative_similarity
    )

    pairwise_win_rate = float(
        positive_beats_negative.mean()
    )

    positive_mean = float(
        positive_similarity.mean()
    )

    positive_median = float(
        np.median(
            positive_similarity
        )
    )

    negative_mean = float(
        negative_similarity.mean()
    )

    negative_std = float(
        negative_similarity.std()
    )

    separation = (
        positive_mean
        - negative_mean
    )

    pooled_effect = (
        separation
        / negative_std
        if negative_std > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Compare observed retrieval metrics to chance
    # --------------------------------------------------------

    metric_row = (
        metrics_df[
            metrics_df[
                "map_size"
            ].astype(int)
            == map_size
        ]
        .iloc[0]
    )

    observed_r1 = float(
        metric_row[
            "R@1"
        ]
    )

    observed_r5 = float(
        metric_row[
            "R@5"
        ]
    )

    observed_r10 = float(
        metric_row[
            "R@10"
        ]
    )

    observed_map = float(
        metric_row[
            "mAP"
        ]
    )

    observed_mean_rank = float(
        metric_row[
            "mean_positive_rank"
        ]
    )

    normalized_rank_gain = (
        (
            random_mean_rank
            - observed_mean_rank
        )
        /
        (
            random_mean_rank
            - 1.0
        )
    )

    diagnostic_records.append({
        "map_size": int(
            map_size
        ),

        "observed_R@1": (
            observed_r1
        ),

        "chance_R@1": (
            random_r1
        ),

        "observed_R@5": (
            observed_r5
        ),

        "chance_R@5": (
            random_r5
        ),

        "observed_R@10": (
            observed_r10
        ),

        "chance_R@10": (
            random_r10
        ),

        "observed_mAP": (
            observed_map
        ),

        "chance_expected_mAP": (
            random_mrr
        ),

        "observed_mean_rank": (
            observed_mean_rank
        ),

        "chance_mean_rank": (
            random_mean_rank
        ),

        "normalized_rank_gain": (
            normalized_rank_gain
        ),

        "positive_similarity_mean": (
            positive_mean
        ),

        "positive_similarity_median": (
            positive_median
        ),

        "random_negative_similarity_mean": (
            negative_mean
        ),

        "random_negative_similarity_std": (
            negative_std
        ),

        "positive_minus_negative_mean": (
            separation
        ),

        "effect_size_vs_random_negative": (
            pooled_effect
        ),

        "positive_beats_random_negative_probability": (
            pairwise_win_rate
        ),
    })


diagnostic_df = pd.DataFrame(
    diagnostic_records
)


# ------------------------------------------------------------
# 6. Macro diagnostic
# ------------------------------------------------------------

macro_observed_r1 = float(
    metrics_df[
        "R@1"
    ].mean()
)

macro_observed_r5 = float(
    metrics_df[
        "R@5"
    ].mean()
)

macro_observed_r10 = float(
    metrics_df[
        "R@10"
    ].mean()
)

macro_observed_map = float(
    metrics_df[
        "mAP"
    ].mean()
)

macro_mean_rank = float(
    metrics_df[
        "mean_positive_rank"
    ].mean()
)

macro_pairwise_win = float(
    diagnostic_df[
        "positive_beats_random_negative_probability"
    ].mean()
)

macro_effect_size = float(
    diagnostic_df[
        "effect_size_vs_random_negative"
    ].mean()
)

macro_normalized_rank_gain = float(
    diagnostic_df[
        "normalized_rank_gain"
    ].mean()
)


# ------------------------------------------------------------
# 7. Interpretation
# ------------------------------------------------------------

# This is deliberately conservative.
#
# Pairwise random baseline = 0.5.
# Retrieval recalls are also compared directly
# to exact random-search expectations.

recall_near_chance = bool(
    abs(
        macro_observed_r5
        - random_r5
    ) < 0.002
    and
    abs(
        macro_observed_r10
        - random_r10
    ) < 0.002
)

pairwise_weak = bool(
    macro_pairwise_win
    < 0.60
)

if (
    not descriptor_collapse_detected
    and
    recall_near_chance
    and
    pairwise_weak
):

    interpretation = (
        "DESCRIPTORS_ARE_NONCOLLAPSED_BUT_"
        "ZERO_SHOT_CROSS_DOMAIN_RETRIEVAL_SIGNAL_"
        "IS_NEAR_CHANCE"
    )

elif descriptor_collapse_detected:

    interpretation = (
        "POSSIBLE_DESCRIPTOR_COLLAPSE_"
        "REQUIRES_MODEL_PIPELINE_REVIEW"
    )

else:

    interpretation = (
        "WEAK_BUT_MEASURABLE_CROSS_DOMAIN_SIGNAL"
    )


# ------------------------------------------------------------
# 8. Save diagnostics
# ------------------------------------------------------------

DIAGNOSTIC_CSV = (
    REPORT_ROOT
    / "vps_local_chance_level_diagnostic.csv"
)

DIAGNOSTIC_JSON = (
    AUDIT_ROOT
    / "vps_local_chance_level_diagnostic.json"
)

diagnostic_df.to_csv(
    DIAGNOSTIC_CSV,
    index=False,
)

report = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "query_count_per_scale": (
        N
    ),

    "random_chance": {
        "R@1": (
            random_r1
        ),

        "R@5": (
            random_r5
        ),

        "R@10": (
            random_r10
        ),

        "expected_mAP_MRR": (
            random_mrr
        ),

        "expected_mean_rank": (
            random_mean_rank
        ),

        "pairwise_positive_vs_random_negative": (
            0.5
        ),
    },

    "observed_macro": {
        "R@1": (
            macro_observed_r1
        ),

        "R@5": (
            macro_observed_r5
        ),

        "R@10": (
            macro_observed_r10
        ),

        "mAP": (
            macro_observed_map
        ),

        "mean_positive_rank": (
            macro_mean_rank
        ),

        "positive_beats_random_negative_probability": (
            macro_pairwise_win
        ),

        "effect_size_vs_random_negative": (
            macro_effect_size
        ),

        "normalized_rank_gain": (
            macro_normalized_rank_gain
        ),
    },

    "descriptor_diversity": {
        "query_unique_rounded_5dp": (
            q_unique_rounded
        ),

        "gallery_unique_rounded_5dp": (
            g_unique_rounded
        ),

        "query_mean_dimension_std": (
            q_std
        ),

        "gallery_mean_dimension_std": (
            g_std
        ),

        "collapse_detected": (
            descriptor_collapse_detected
        ),
    },

    "interpretation": (
        interpretation
    ),

    "important_reporting_note": (
        "Near-chance VPS performance is a measured "
        "zero-shot cross-dataset result for the local "
        "reimplementation and must not be represented "
        "as an official MobileGeo benchmark result."
    ),

    "crs_status": (
        "UNVERIFIED"
    ),

    "meter_level_error_claim_allowed": (
        False
    ),

    "retrieval_is_verified_uav_pose": (
        False
    ),
}

DIAGNOSTIC_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 9. Final display
# ------------------------------------------------------------

print("\n" + "=" * 98)
print("PER-SCALE POSITIVE-SIGNAL DIAGNOSTIC")
print("=" * 98)

display_df = diagnostic_df[
    [
        "map_size",
        "positive_similarity_mean",
        "random_negative_similarity_mean",
        "positive_minus_negative_mean",
        "positive_beats_random_negative_probability",
        "normalized_rank_gain",
    ]
].copy()

print(
    display_df.to_string(
        index=False,
        formatters={
            "positive_similarity_mean":
            lambda x: f"{x:.5f}",

            "random_negative_similarity_mean":
            lambda x: f"{x:.5f}",

            "positive_minus_negative_mean":
            lambda x: f"{x:.5f}",

            "positive_beats_random_negative_probability":
            lambda x: f"{x * 100:.2f}%",

            "normalized_rank_gain":
            lambda x: f"{x * 100:.2f}%",
        },
    )
)


print("\n" + "=" * 98)
print("✅ NOTEBOOK 04 CELL 10 COMPLETE")
print("=" * 98)

print("\nObserved macro R@1:")
print(
    f"{macro_observed_r1 * 100:.4f}%"
)

print("Random R@1:")
print(
    f"{random_r1 * 100:.4f}%"
)

print("\nObserved macro R@5:")
print(
    f"{macro_observed_r5 * 100:.4f}%"
)

print("Random R@5:")
print(
    f"{random_r5 * 100:.4f}%"
)

print("\nObserved macro R@10:")
print(
    f"{macro_observed_r10 * 100:.4f}%"
)

print("Random R@10:")
print(
    f"{random_r10 * 100:.4f}%"
)

print("\nObserved macro mAP:")
print(
    f"{macro_observed_map * 100:.4f}%"
)

print("Random expected mAP:")
print(
    f"{random_mrr * 100:.4f}%"
)

print("\nObserved macro mean rank:")
print(
    f"{macro_mean_rank:.3f}"
)

print("Random expected mean rank:")
print(
    f"{random_mean_rank:.3f}"
)

print(
    "\nPositive beats random negative probability:"
)

print(
    f"{macro_pairwise_win * 100:.2f}%"
)

print(
    "Random pairwise baseline:"
)

print(
    "50.00%"
)

print("\nDescriptor collapse detected:")
print(
    descriptor_collapse_detected
)

print("\nINTERPRETATION:")
print(
    interpretation
)

print("\nDiagnostic CSV:")
print(
    DIAGNOSTIC_CSV
)

print("\nDiagnostic JSON:")
print(
    DIAGNOSTIC_JSON
)

print("\nCRS:")
print(
    "UNVERIFIED — METER-LEVEL ERROR STILL NOT CLAIMED"
)

print("\n" + "=" * 98)

if (
    interpretation
    ==
    "DESCRIPTORS_ARE_NONCOLLAPSED_BUT_"
    "ZERO_SHOT_CROSS_DOMAIN_RETRIEVAL_SIGNAL_"
    "IS_NEAR_CHANCE"
):

    print(
        "✅ EVALUATION PIPELINE CONSISTENCY CHECK PASSED"
    )

    print(
        "\nResult: the local SUES-200-oriented model "
        "does not meaningfully transfer zero-shot to "
        "this VPS retrieval domain."
    )

else:

    print(
        "⚠ REVIEW DIAGNOSTIC BEFORE FINALIZING PHASE 4"
    )

print("=" * 98)

NOTEBOOK 04 — CHANCE-LEVEL / POSITIVE-SIGNAL DIAGNOSTIC

Queries per scale:
2,331

EXACT RANDOM-CHANCE BASELINE
Random R@1  : 0.0429%
Random R@5  : 0.2145%
Random R@10 : 0.4290%
Random expected mAP/MRR: 0.3574%
Random expected mean rank: 1166.000

DESCRIPTOR DIVERSITY
Query unique rounded descriptors:
2,331 / 2,331

Gallery unique rounded descriptors:
27,972 / 27,972

Mean query dimension std:
0.01747053

Mean gallery dimension std:
0.01764631

Descriptor collapse detected:
False

PER-SCALE POSITIVE-SIGNAL DIAGNOSTIC
 map_size positive_similarity_mean random_negative_similarity_mean positive_minus_negative_mean positive_beats_random_negative_probability normalized_rank_gain
      700                  0.19196                         0.13062                      0.06134                                     58.70%               17.40%
      800                  0.17658                         0.12767                      0.04891                                     57.10%               14.2

In [37]:
# ============================================================
# NOTEBOOK 04 — CELL 11
# FINAL VALIDATION + REPORT + EVIDENCE PACKAGE
#
# Notebook:
# 04_georeferenced_project_benchmark
#
# IMPORTANT REPORTING LOCKS
# - Model is LOCAL_REIMPLEMENTATION
# - NOT an official MobileGeo VPS result
# - VPS protocol is project-defined
# - CRS remains UNVERIFIED
# - NO meter-level geodetic error claim
# - Retrieval != verified UAV pose
# - Colab timing != Jetson Orin Nano latency
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import zipfile

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE4_ROOT = (
    PROJECT_ROOT
    / "results"
    / "georeferenced_benchmark"
)

AUDIT_ROOT = (
    PHASE4_ROOT
    / "audit"
)

MANIFEST_ROOT = (
    PHASE4_ROOT
    / "manifests"
)

PROTOCOL_ROOT = (
    PHASE4_ROOT
    / "protocol"
)

REPORT_ROOT = (
    PHASE4_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE4_ROOT
    / "rankings"
)

EMBED_ROOT = (
    PHASE4_ROOT
    / "embeddings"
)

FINAL_ROOT = (
    PHASE4_ROOT
    / "final"
)

FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Required artifacts
# ------------------------------------------------------------

files = {
    "schema_audit":
        AUDIT_ROOT
        / "vps_archive_full_georeference_audit.json",

    "path_resolution_audit":
        AUDIT_ROOT
        / "vps_partition_safe_resolution_audit.json",

    "duplicate_audit":
        AUDIT_ROOT
        / "vps_cross_partition_duplicate_audit.csv",

    "protocol":
        PROTOCOL_ROOT
        / "vps_leakage_safe_protocol.json",

    "cache_audit":
        AUDIT_ROOT
        / "vps_local_cache_audit.json",

    "descriptor_report":
        AUDIT_ROOT
        / "vps_local_descriptor_extraction_report.json",

    "retrieval_metrics_csv":
        REPORT_ROOT
        / "vps_local_12scale_retrieval_metrics.csv",

    "retrieval_metrics_json":
        REPORT_ROOT
        / "vps_local_12scale_retrieval_metrics.json",

    "chance_diagnostic_csv":
        REPORT_ROOT
        / "vps_local_chance_level_diagnostic.csv",

    "chance_diagnostic_json":
        AUDIT_ROOT
        / "vps_local_chance_level_diagnostic.json",

    "query_manifest":
        MANIFEST_ROOT
        / "vps_core_eval_queries.csv",

    "gallery_manifest":
        MANIFEST_ROOT
        / "vps_core_eval_gallery_all_scales.csv",

    "cached_query_manifest":
        MANIFEST_ROOT
        / "vps_cached_query_manifest.csv",

    "cached_gallery_manifest":
        MANIFEST_ROOT
        / "vps_cached_gallery_manifest.csv",

    "per_query_rankings":
        RANKING_ROOT
        / "vps_local_per_query_rankings.csv",

    "top5_rankings":
        RANKING_ROOT
        / "vps_local_top5_rankings.csv",

    "per_site_metrics":
        REPORT_ROOT
        / "vps_local_per_site_metrics.csv",

    "per_nominal_level_metrics":
        REPORT_ROOT
        / "vps_local_per_nominal_level_metrics.csv",

    "query_embeddings":
        EMBED_ROOT
        / "vps_local_query_embeddings.npz",

    "gallery_embeddings":
        EMBED_ROOT
        / "vps_local_gallery_embeddings.npz",
}


missing = [
    str(path)
    for path in files.values()
    if not path.is_file()
]

if missing:
    print("Missing artifacts:")
    for path in missing:
        print(" -", path)

    raise FileNotFoundError(
        "Notebook 04 required artifacts are incomplete."
    )


# ------------------------------------------------------------
# 3. Load key audits/results
# ------------------------------------------------------------

def load_json(path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


schema_audit = load_json(
    files[
        "schema_audit"
    ]
)

resolution_audit = load_json(
    files[
        "path_resolution_audit"
    ]
)

protocol = load_json(
    files[
        "protocol"
    ]
)

cache_audit = load_json(
    files[
        "cache_audit"
    ]
)

descriptor_report = load_json(
    files[
        "descriptor_report"
    ]
)

retrieval_report = load_json(
    files[
        "retrieval_metrics_json"
    ]
)

chance_report = load_json(
    files[
        "chance_diagnostic_json"
    ]
)

metrics_df = pd.read_csv(
    files[
        "retrieval_metrics_csv"
    ]
)

duplicate_df = pd.read_csv(
    files[
        "duplicate_audit"
    ]
)

query_df = pd.read_csv(
    files[
        "query_manifest"
    ]
)

gallery_df = pd.read_csv(
    files[
        "gallery_manifest"
    ]
)


# ------------------------------------------------------------
# 4. Hard validation
# ------------------------------------------------------------

expected_model = (
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
)

expected_provenance = (
    "LOCAL_REIMPLEMENTATION"
)

expected_protocol = (
    "PROJECT_DEFINED_VPS_"
    "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL"
)

expected_interpretation = (
    "DESCRIPTORS_ARE_NONCOLLAPSED_BUT_"
    "ZERO_SHOT_CROSS_DOMAIN_RETRIEVAL_SIGNAL_"
    "IS_NEAR_CHANCE"
)


checks = {}


# Query/gallery counts
checks[
    "query_count_2331"
] = bool(
    len(query_df)
    == 2331
)

checks[
    "gallery_count_27972"
] = bool(
    len(gallery_df)
    == 27972
)


# Scale protocol
expected_scales = list(
    range(
        700,
        1801,
        100,
    )
)

actual_scales = sorted(
    gallery_df[
        "map_size"
    ]
    .astype(int)
    .unique()
    .tolist()
)

checks[
    "12_expected_scales"
] = bool(
    actual_scales
    == expected_scales
)


scale_counts = (
    gallery_df
    .groupby(
        "map_size"
    )
    .size()
)

checks[
    "2331_gallery_per_scale"
] = bool(
    len(scale_counts)
    == 12
    and
    (
        scale_counts
        == 2331
    ).all()
)


# Leakage audit
checks[
    "349_cross_partition_duplicates"
] = bool(
    len(
        duplicate_df
    )
    == 349
)

checks[
    "all_cross_partition_duplicates_exact"
] = bool(
    duplicate_df[
        "exact_duplicate"
    ]
    .astype(bool)
    .all()
)


# Protocol
checks[
    "protocol_ready"
] = bool(
    protocol.get(
        "protocol_ready",
        False,
    )
)

checks[
    "protocol_name_correct"
] = bool(
    protocol.get(
        "protocol_name"
    )
    == expected_protocol
)


# Local cache
checks[
    "full_cache_ready"
] = bool(
    cache_audit.get(
        "full_cache_ready",
        False,
    )
)


# Model extraction
checks[
    "model_name_correct"
] = bool(
    descriptor_report.get(
        "model_name"
    )
    == expected_model
)

checks[
    "model_provenance_correct"
] = bool(
    descriptor_report.get(
        "provenance"
    )
    == expected_provenance
)

checks[
    "checkpoint_verified"
] = bool(
    descriptor_report.get(
        "checkpoint_sha256_verified",
        False,
    )
)

checks[
    "descriptor_validation"
] = bool(
    descriptor_report.get(
        "descriptor_validation",
        False,
    )
)

checks[
    "query_embedding_shape"
] = bool(
    descriptor_report.get(
        "query_shape"
    )
    == [
        2331,
        2048,
    ]
)

checks[
    "gallery_embedding_shape"
] = bool(
    descriptor_report.get(
        "gallery_shape"
    )
    == [
        27972,
        2048,
    ]
)


# Chance diagnostic
checks[
    "descriptor_not_collapsed"
] = bool(
    not chance_report[
        "descriptor_diversity"
    ][
        "collapse_detected"
    ]
)

checks[
    "near_chance_interpretation"
] = bool(
    chance_report.get(
        "interpretation"
    )
    == expected_interpretation
)


# CRS/reporting restrictions
checks[
    "crs_unverified"
] = bool(
    chance_report.get(
        "crs_status"
    )
    == "UNVERIFIED"
)

checks[
    "meter_error_claim_blocked"
] = bool(
    chance_report.get(
        "meter_level_error_claim_allowed"
    )
    is False
)

checks[
    "retrieval_not_verified_pose"
] = bool(
    retrieval_report[
        "reporting_constraints"
    ][
        "retrieval_is_verified_uav_pose"
    ]
    is False
)


all_checks_pass = bool(
    all(
        checks.values()
    )
)


# ------------------------------------------------------------
# 5. Metrics
# ------------------------------------------------------------

macro = (
    retrieval_report[
        "macro_summary"
    ]
)

chance = (
    chance_report[
        "random_chance"
    ]
)

observed = (
    chance_report[
        "observed_macro"
    ]
)


macro_r1 = float(
    macro[
        "macro_R@1"
    ]
)

macro_r5 = float(
    macro[
        "macro_R@5"
    ]
)

macro_r10 = float(
    macro[
        "macro_R@10"
    ]
)

macro_map = float(
    macro[
        "macro_mAP"
    ]
)

macro_mean_rank = float(
    macro[
        "macro_mean_positive_rank"
    ]
)


# ------------------------------------------------------------
# 6. Best/worst scale diagnostics
# ------------------------------------------------------------

best_r1_row = (
    metrics_df
    .sort_values(
        [
            "R@1",
            "mAP",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_map_row = (
    metrics_df
    .sort_values(
        "mAP",
        ascending=False,
    )
    .iloc[0]
)

best_mean_rank_row = (
    metrics_df
    .sort_values(
        "mean_positive_rank",
        ascending=True,
    )
    .iloc[0]
)

worst_mean_rank_row = (
    metrics_df
    .sort_values(
        "mean_positive_rank",
        ascending=False,
    )
    .iloc[0]
)


# ------------------------------------------------------------
# 7. Final scientific interpretation
# ------------------------------------------------------------

final_interpretation = {
    "primary_result": (
        "The local SUES-200-oriented reimplementation "
        "shows near-chance zero-shot retrieval performance "
        "on the project-defined VPS benchmark."
    ),

    "pipeline_diagnostic": (
        "Descriptor collapse was not detected. "
        "All query and gallery descriptors were unique "
        "at five-decimal rounding, and positive pairs "
        "showed only weak average separation from "
        "random negatives."
    ),

    "transfer_interpretation": (
        "The measured result is consistent with weak "
        "cross-domain transfer rather than a successful "
        "VPS geolocalization result."
    ),

    "official_result_status": (
        "NOT_AN_OFFICIAL_MOBILEGEO_VPS_RESULT"
    ),

    "pose_status": (
        "RETRIEVAL_OUTPUT_IS_NOT_VERIFIED_UAV_POSE"
    ),

    "crs_status": (
        "CRS_UNVERIFIED"
    ),

    "meter_level_error_status": (
        "NOT_REPORTED_BECAUSE_CRS_PROVENANCE_IS_UNVERIFIED"
    ),

    "timing_status": (
        "GOOGLE_COLAB_RUNTIME_NOT_JETSON_ORIN_NANO"
    ),
}


# ------------------------------------------------------------
# 8. SHA256 helper
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# 9. Artifact hashes
# ------------------------------------------------------------

artifact_hashes = {}

# Hash important reproducibility artifacts.
# Embeddings are included too.
for name, path in files.items():

    artifact_hashes[
        name
    ] = {
        "path": str(
            path
        ),

        "bytes": int(
            path.stat().st_size
        ),

        "sha256": sha256_file(
            path
        ),
    }


# ------------------------------------------------------------
# 10. Completion manifest
# ------------------------------------------------------------

COMPLETION_JSON = (
    FINAL_ROOT
    / "notebook04_completion_manifest.json"
)

completion = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "notebook": (
        "04_georeferenced_project_benchmark"
    ),

    "status": (
        "COMPLETE"
        if all_checks_pass
        else "VALIDATION_FAILED"
    ),

    "validation_checks": (
        checks
    ),

    "all_validation_checks_pass": (
        all_checks_pass
    ),

    "benchmark": {
        "protocol": (
            expected_protocol
        ),

        "protocol_origin": (
            "PROJECT_DEFINED"
        ),

        "core_partition": (
            "merge_test_700-1800_cr0.95_stride100"
        ),

        "excluded_partition": (
            "val"
        ),

        "excluded_partition_reason": (
            "349_EXACT_DUPLICATE_SAMPLES_OF_CORE_PARTITION"
        ),

        "query_count": (
            2331
        ),

        "gallery_count_per_scale": (
            2331
        ),

        "scale_count": (
            12
        ),

        "map_sizes": (
            expected_scales
        ),

        "total_scale_controlled_gallery_records": (
            27972
        ),

        "positive_definition": (
            "EXACT_FULL_SAMPLE_KEY_MATCH"
        ),

        "positives_per_query_per_scale": (
            1
        ),
    },

    "model": {
        "name": (
            expected_model
        ),

        "provenance": (
            expected_provenance
        ),

        "official_mobilegeo_model": (
            False
        ),

        "descriptor_dim": (
            2048
        ),

        "checkpoint_sha256": (
            descriptor_report[
                "checkpoint_sha256"
            ]
        ),

        "checkpoint_sha256_verified": (
            True
        ),
    },

    "results": {
        "macro_R@1": (
            macro_r1
        ),

        "macro_R@5": (
            macro_r5
        ),

        "macro_R@10": (
            macro_r10
        ),

        "macro_mAP": (
            macro_map
        ),

        "macro_mean_positive_rank": (
            macro_mean_rank
        ),

        "random_R@1": float(
            chance[
                "R@1"
            ]
        ),

        "random_R@5": float(
            chance[
                "R@5"
            ]
        ),

        "random_R@10": float(
            chance[
                "R@10"
            ]
        ),

        "random_expected_mAP": float(
            chance[
                "expected_mAP_MRR"
            ]
        ),

        "random_expected_mean_rank": float(
            chance[
                "expected_mean_rank"
            ]
        ),

        "positive_beats_random_negative_probability": float(
            observed[
                "positive_beats_random_negative_probability"
            ]
        ),

        "descriptor_collapse_detected": (
            False
        ),

        "interpretation": (
            expected_interpretation
        ),
    },

    "scale_diagnostics": {
        "best_R@1_map_size": int(
            best_r1_row[
                "map_size"
            ]
        ),

        "best_mAP_map_size": int(
            best_map_row[
                "map_size"
            ]
        ),

        "best_mean_rank_map_size": int(
            best_mean_rank_row[
                "map_size"
            ]
        ),

        "worst_mean_rank_map_size": int(
            worst_mean_rank_row[
                "map_size"
            ]
        ),
    },

    "reporting_constraints": {
        "crs_status": (
            "UNVERIFIED"
        ),

        "wgs84_claim_allowed": (
            False
        ),

        "meter_level_geodetic_error_claim_allowed": (
            False
        ),

        "retrieval_is_verified_uav_pose": (
            False
        ),

        "nominal_level_status": (
            "DATASET_FOLDER_TOKEN_ONLY_"
            "NOT_VERIFIED_AS_AGL_OR_MSL"
        ),

        "runtime_timing_status": (
            "GOOGLE_COLAB_RUNTIME_NOT_JETSON_ORIN_NANO"
        ),

        "search_timing_scope": (
            "SIMILARITY_MATRIX_PLUS_FULL_SORT_"
            "DESCRIPTOR_EXTRACTION_EXCLUDED"
        ),
    },

    "scientific_interpretation": (
        final_interpretation
    ),

    "artifact_hashes": (
        artifact_hashes
    ),
}

COMPLETION_JSON.write_text(
    json.dumps(
        completion,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 11. Human-readable report
# ------------------------------------------------------------

REPORT_MD = (
    FINAL_ROOT
    / "notebook04_final_report.md"
)

report_lines = [
    "# Notebook 04 — Georeferenced Project Benchmark",
    "",
    "## Status",
    "",
    (
        "**COMPLETE**"
        if all_checks_pass
        else "**VALIDATION FAILED**"
    ),
    "",
    "## Benchmark protocol",
    "",
    (
        "`PROJECT_DEFINED_VPS_"
        "LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL`"
    ),
    "",
    "- Core UAV queries: **2,331**",
    "- Gallery per map scale: **2,331**",
    "- Map scales: **700–1800 in steps of 100**",
    "- Scale-controlled trials: **12**",
    "- Total gallery records across trials: **27,972**",
    (
        "- 349 `val` samples were verified as exact "
        "duplicates of samples in the core partition "
        "and were excluded as an independent set."
    ),
    "",
    "## Model",
    "",
    (
        "**AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION**"
    ),
    "",
    "- Provenance: `LOCAL_REIMPLEMENTATION`",
    "- Descriptor dimension: 2048",
    "- This is **not** an official MobileGeo VPS result.",
    "",
    "## Macro retrieval results",
    "",
    (
        f"- R@1: **{macro_r1 * 100:.4f}%**"
    ),
    (
        f"- R@5: **{macro_r5 * 100:.4f}%**"
    ),
    (
        f"- R@10: **{macro_r10 * 100:.4f}%**"
    ),
    (
        f"- mAP: **{macro_map * 100:.4f}%**"
    ),
    (
        "- Mean positive rank: "
        f"**{macro_mean_rank:.3f}**"
    ),
    "",
    "## Random retrieval reference",
    "",
    (
        f"- Random R@1: "
        f"{float(chance['R@1']) * 100:.4f}%"
    ),
    (
        f"- Random R@5: "
        f"{float(chance['R@5']) * 100:.4f}%"
    ),
    (
        f"- Random R@10: "
        f"{float(chance['R@10']) * 100:.4f}%"
    ),
    (
        "- Random expected mAP/MRR: "
        f"{float(chance['expected_mAP_MRR']) * 100:.4f}%"
    ),
    (
        "- Random expected mean rank: "
        f"{float(chance['expected_mean_rank']):.3f}"
    ),
    "",
    "## Diagnostic conclusion",
    "",
    (
        "- Descriptor collapse detected: **False**"
    ),
    (
        "- Positive beats random negative probability: "
        f"**{float(observed['positive_beats_random_negative_probability']) * 100:.2f}%**"
    ),
    "",
    (
        "The descriptors are non-collapsed, but the "
        "zero-shot cross-domain retrieval signal is "
        "near chance. The local SUES-200-oriented "
        "reimplementation therefore does not meaningfully "
        "transfer zero-shot to this VPS retrieval domain."
    ),
    "",
    "## Reporting restrictions",
    "",
    "- CRS: **UNVERIFIED**",
    "- WGS84 is **not claimed**.",
    "- Meter-level geodetic error is **not reported**.",
    "- Retrieval output is **not verified UAV pose**.",
    (
        "- Dataset level token is not claimed as "
        "verified AGL or MSL."
    ),
    (
        "- Runtime timings are Google Colab measurements, "
        "not Jetson Orin Nano deployment latency."
    ),
    (
        "- Search timing measures similarity matrix + "
        "full sort only; descriptor extraction is excluded."
    ),
]

REPORT_MD.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 12. Compact evidence ZIP
# ------------------------------------------------------------
# Deliberately excludes:
# - 4.05 GB local image cache
# - embedding NPZs
#
# Hashes of embeddings are still recorded in completion JSON.
# ------------------------------------------------------------

EVIDENCE_ZIP = (
    FINAL_ROOT
    / "notebook04_georeferenced_benchmark_evidence.zip"
)

evidence_files = [
    files[
        "schema_audit"
    ],
    files[
        "path_resolution_audit"
    ],
    files[
        "duplicate_audit"
    ],
    files[
        "protocol"
    ],
    files[
        "cache_audit"
    ],
    files[
        "descriptor_report"
    ],
    files[
        "retrieval_metrics_csv"
    ],
    files[
        "retrieval_metrics_json"
    ],
    files[
        "chance_diagnostic_csv"
    ],
    files[
        "chance_diagnostic_json"
    ],
    files[
        "query_manifest"
    ],
    files[
        "gallery_manifest"
    ],
    files[
        "per_query_rankings"
    ],
    files[
        "top5_rankings"
    ],
    files[
        "per_site_metrics"
    ],
    files[
        "per_nominal_level_metrics"
    ],
    COMPLETION_JSON,
    REPORT_MD,
]

with zipfile.ZipFile(
    EVIDENCE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for path in evidence_files:

        relative_name = (
            path.relative_to(
                PHASE4_ROOT
            )
        )

        zf.write(
            path,
            arcname=str(
                relative_name
            ),
        )


evidence_sha256 = sha256_file(
    EVIDENCE_ZIP
)


# ------------------------------------------------------------
# 13. Final output
# ------------------------------------------------------------

print("=" * 100)
print("✅ NOTEBOOK 04 CELL 11 COMPLETE")
print("=" * 100)

print("\nValidation checks:")

for key, value in checks.items():

    marker = (
        "✅"
        if value
        else "❌"
    )

    print(
        f"{marker} {key}: {value}"
    )


print("\nAll validation checks pass:")
print(
    all_checks_pass
)

print("\nFinal protocol:")
print(
    expected_protocol
)

print("\nCore queries:")
print(
    "2,331"
)

print("\nGallery per scale:")
print(
    "2,331"
)

print("\nScales:")
print(
    "12"
)

print("\nMacro R@1:")
print(
    f"{macro_r1 * 100:.4f}%"
)

print("\nMacro R@5:")
print(
    f"{macro_r5 * 100:.4f}%"
)

print("\nMacro R@10:")
print(
    f"{macro_r10 * 100:.4f}%"
)

print("\nMacro mAP:")
print(
    f"{macro_map * 100:.4f}%"
)

print("\nMacro mean positive rank:")
print(
    f"{macro_mean_rank:.3f}"
)

print("\nDescriptor collapse:")
print(
    False
)

print("\nFinal interpretation:")
print(
    expected_interpretation
)

print("\nModel provenance:")
print(
    "LOCAL_REIMPLEMENTATION"
)

print("\nOfficial MobileGeo VPS result:")
print(
    False
)

print("\nCRS:")
print(
    "UNVERIFIED"
)

print("\nWGS84 claimed:")
print(
    False
)

print("\nMeter-level geodetic error reported:")
print(
    False
)

print("\nRetrieval is verified UAV pose:")
print(
    False
)

print("\nCompletion manifest:")
print(
    COMPLETION_JSON
)

print("\nFinal report:")
print(
    REPORT_MD
)

print("\nEvidence ZIP:")
print(
    EVIDENCE_ZIP
)

print("\nEvidence ZIP SHA256:")
print(
    evidence_sha256
)

print("\n" + "=" * 100)

if all_checks_pass:

    print(
        "✅ NOTEBOOK 04 — GEOREFERENCED PROJECT BENCHMARK COMPLETE"
    )

    print(
        "\nNext notebook: "
        "05_robustness_and_rejection.ipynb"
    )

else:

    print(
        "❌ NOTEBOOK 04 FINAL VALIDATION FAILED"
    )

print("=" * 100)

✅ NOTEBOOK 04 CELL 11 COMPLETE

Validation checks:
✅ query_count_2331: True
✅ gallery_count_27972: True
✅ 12_expected_scales: True
✅ 2331_gallery_per_scale: True
✅ 349_cross_partition_duplicates: True
✅ all_cross_partition_duplicates_exact: True
✅ protocol_ready: True
✅ protocol_name_correct: True
✅ full_cache_ready: True
✅ model_name_correct: True
✅ model_provenance_correct: True
✅ checkpoint_verified: True
✅ descriptor_validation: True
✅ query_embedding_shape: True
✅ gallery_embedding_shape: True
✅ descriptor_not_collapsed: True
✅ near_chance_interpretation: True
✅ crs_unverified: True
✅ meter_error_claim_blocked: True
✅ retrieval_not_verified_pose: True

All validation checks pass:
True

Final protocol:
PROJECT_DEFINED_VPS_LEAKAGE_SAFE_SCALE_CONTROLLED_RETRIEVAL

Core queries:
2,331

Gallery per scale:
2,331

Scales:
12

Macro R@1:
0.0501%

Macro R@5:
0.2109%

Macro R@10:
0.4183%

Macro mAP:
0.3938%

Macro mean positive rank:
1088.051

Descriptor collapse:
False

Final interpretatio

## Work cells

Add the phase-specific implementation below this cell.
